# Baseline Models — Naïve / HistMean / LinReg / LightGBM

**Thesis section.** 4.1 — non-neural baselines in the ceiling study

**Inputs.** `data/features/<TICKER>_1hour_features.csv`, `data/splits/<TICKER>_1hour/`

**Outputs.** rows 1–4 of `results/ch2_standardised_results.json`

**Expected runtime.** < 10 min. **Expected GPU.** not required.

> All paths in the CONFIG cell below resolve relative to the repo root. On
> Google Colab, uncomment the Drive fallback line.


## What this notebook produces

This notebook trains the non-neural baselines of the Chapter 4.1 ceiling study (Naïve / Historical Mean / LinReg / LightGBM) plus the PyTorch sequence baselines (LSTM, Attention-LSTM, Transformer-LSTM, regression + classification heads) and TimesFM 2.5 zero-shot + fine-tuned, writing per-model prediction CSVs and then rendering the **eight shipped thesis figures** (`fig1_da_bar.png`, `fig2_pred_lag.png`, `fig3_da_heatmap.png`, `fig4_naive.png`, `fig4b_illusion_grid.png`, `fig5_confusion.png`, `fig6_clf_accuracy.png`, `fig7_da_by_type.png`) into `results/figures/`.

Together with `01_nine_method_ceiling.ipynb` (which handles the sequence-model grid sweep) and `02_missing_metrics.ipynb` (which fills gaps where the first pass crashed), this is the third notebook behind the central finding that **direct next-bar price-direction prediction plateaus at ~50 % directional accuracy across nine mainstream methods** under strict no-leakage evaluation. See the repo README's "Central finding" box and `results/ch2_standardised_results.json` for the aggregated table.


In [ ]:
# === CONFIG (edit paths here) ===
from pathlib import Path

CONFIG = {
    "data_dir":        Path("../../data"),         # processed + features + splits
    "results_dir":     Path("../../results"),
    "checkpoints_dir": Path("../../checkpoints"),
    "seed":            42,
    "device":          "cuda",                      # or "cpu"
    # Colab fallback — uncomment if running on Colab with the dataset mounted:
    # "data_dir": Path("/content/drive/MyDrive/thesis_data"),
}
for key, path in CONFIG.items():
    if isinstance(path, Path):
        path.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
os.chdir(CONFIG['data_dir'])

In [ ]:
"""
regression_baselines.py
=======================
三种回归基线模型对比：NaivePredictor / LinearRegression / LightGBM
目标：预测下一根K线的close价格，评估方向准确率(DA)、MSE、R²等指标
"""

import os
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import lightgbm as lgb

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG — 所有超参数 / 路径集中于此
# ============================================================
CONFIG: Dict = {
    "SPLITS_DIR": "splits",
    "FEATURES_DIR": "features",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "FREQ": "1hour",
    "OUTPUT_DIR": "results",
    # 如果splits不含技术指标，按行数比例切分
    "TRAIN_RATIO": 0.70,
    "VAL_RATIO": 0.15,
    # TEST_RATIO = 1 - TRAIN - VAL = 0.15
    # 不参与特征的列（小写匹配）
    "DROP_COLS": ["timestamp", "open", "high", "low", "close", "volume",
                  "date", "datetime", "time", "ts_event"],
    # LightGBM 超参数
    "LGB_PARAMS": {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "verbose": -1,
        "n_jobs": -1,
        "random_state": 42,
    },
    "LGB_EARLY_STOPPING": 20,
}


# ============================================================
# 数据加载
# ============================================================
def _find_feature_cols(df: pd.DataFrame) -> List[str]:
    """返回所有可用作特征的列名（排除 DROP_COLS 及目标列）"""
    drop_set = {c.lower() for c in CONFIG["DROP_COLS"]}
    return [c for c in df.columns if c.lower() not in drop_set]


def _load_splits_csv(ticker: str) -> Optional[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    """尝试从 splits/{ticker}_1hour/ 目录读取 train/val/test"""
    base = os.path.join(CONFIG["SPLITS_DIR"], f"{ticker}_{CONFIG['FREQ']}")
    paths = {k: os.path.join(base, f"{k}.csv") for k in ("train", "val", "test")}
    if not all(os.path.isfile(p) for p in paths.values()):
        return None
    try:
        train = pd.read_csv(paths["train"])
        val = pd.read_csv(paths["val"])
        test = pd.read_csv(paths["test"])
        return train, val, test
    except Exception as e:
        print(f"[WARN] 读取splits失败 {ticker}: {e}")
        return None


def _splits_have_features(df: pd.DataFrame) -> bool:
    """判断splits csv是否已包含技术指标列（多于OHLCV+timestamp）"""
    feat_cols = _find_feature_cols(df)
    return len(feat_cols) >= 5


def _load_features_csv(ticker: str) -> Optional[pd.DataFrame]:
    """从 features/{ticker}_1hour_features.csv 读取完整特征文件"""
    path = os.path.join(CONFIG["FEATURES_DIR"], f"{ticker}_{CONFIG['FREQ']}_features.csv")
    if not os.path.isfile(path):
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"[WARN] 读取features失败 {ticker}: {e}")
        return None


def _split_by_ratio(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """按 70/15/15 行数比例切分，保持时间顺序"""
    n = len(df)
    n_train = int(n * CONFIG["TRAIN_RATIO"])
    n_val = int(n * CONFIG["VAL_RATIO"])
    train = df.iloc[:n_train].copy()
    val = df.iloc[n_train:n_train + n_val].copy()
    test = df.iloc[n_train + n_val:].copy()
    return train, val, test


def load_ticker_data(ticker: str) -> Optional[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    """
    加载单只股票的 train/val/test 数据。
    优先读splits目录；若splits不含技术指标则合并features再按比例切。
    """
    splits_result = _load_splits_csv(ticker)
    if splits_result is not None:
        train, val, test = splits_result
        if _splits_have_features(train):
            return train, val, test
        # splits存在但缺少技术指标 → 用features文件补充
        feat_df = _load_features_csv(ticker)
        if feat_df is not None:
            return _split_by_ratio(feat_df)
        # 无features文件，仍用原始splits（仅OHLCV做特征）
        return train, val, test
    # splits目录不存在 → 直接用features按比例切
    feat_df = _load_features_csv(ticker)
    if feat_df is not None:
        return _split_by_ratio(feat_df)
    print(f"[ERROR] {ticker}: 无可用数据文件")
    return None


# ============================================================
# 特征 / 标签构建
# ============================================================
def _ensure_close_column(df: pd.DataFrame) -> str:
    """找到close列的实际列名（兼容大小写）"""
    for c in df.columns:
        if c.lower() == "close":
            return c
    raise KeyError("DataFrame 中找不到 close 列")


def _prepare_split(
    df: pd.DataFrame, close_col: str, feat_cols: List[str]
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    单个split的特征/标签构建。
    y = 下一行close（shift(-1)），丢弃最后一行NaN。
    返回 (X_raw, y, current_close)
    """
    df = df.copy()
    df["_target"] = df[close_col].shift(-1)
    df = df.dropna(subset=["_target"]).reset_index(drop=True)

    available_feats = [c for c in feat_cols if c in df.columns]
    X = df[available_feats].copy()
    X = X.fillna(X.median())
    X = X.fillna(0.0)

    y = df["_target"].values.astype(np.float64)
    current_close = df[close_col].values.astype(np.float64)
    return X.values.astype(np.float64), y, current_close


def build_xy(
    train: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame
) -> Dict[str, np.ndarray]:
    """
    构建特征矩阵和目标向量。
    StandardScaler fit on train，transform val/test。
    """
    close_col = _ensure_close_column(train)
    feat_cols = _find_feature_cols(train)
    if len(feat_cols) == 0:
        raise ValueError("无可用特征列")

    X_train_raw, y_train, close_train = _prepare_split(train, close_col, feat_cols)
    X_val_raw, y_val, close_val = _prepare_split(val, close_col, feat_cols)
    X_test_raw, y_test, close_test = _prepare_split(test, close_col, feat_cols)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_val = scaler.transform(X_val_raw)
    X_test = scaler.transform(X_test_raw)

    return {
        "X_train": X_train, "y_train": y_train, "close_train": close_train,
        "X_val": X_val, "y_val": y_val, "close_val": close_val,
        "X_test": X_test, "y_test": y_test, "close_test": close_test,
        "scaler": scaler,
        "feature_names": [c for c in feat_cols if c in train.columns],
    }


# ============================================================
# 评估函数
# ============================================================
def _directional_accuracy(
    y_pred: np.ndarray, y_true: np.ndarray, y_current: np.ndarray
) -> float:
    """方向准确率：预测涨跌方向是否与实际一致"""
    pred_dir = np.sign(y_pred - y_current)
    true_dir = np.sign(y_true - y_current)
    # 排除实际变化为0的样本
    mask = true_dir != 0
    if mask.sum() == 0:
        return 0.5
    return float(np.mean(pred_dir[mask] == true_dir[mask]))


def _prediction_lag(y_pred: np.ndarray, y_current: np.ndarray) -> float:
    """
    预测滞后度：pred与current_close的相关系数。
    接近1.0说明模型退化为复制上一个价格。
    """
    if len(y_pred) < 3:
        return np.nan
    corr = np.corrcoef(y_pred, y_current)[0, 1]
    return float(corr)


def _da_z_score_and_p(da: float, n: int) -> Tuple[float, float]:
    """DA的z-score和p-value（单侧，H0: DA=0.5）"""
    if n == 0:
        return 0.0, 1.0
    se = np.sqrt(0.25 / n)
    z = (da - 0.5) / se
    p = 1.0 - stats.norm.cdf(z)
    return float(z), float(p)


def eval_regression(
    y_pred: np.ndarray, y_true: np.ndarray, y_current: np.ndarray
) -> Dict[str, float]:
    """
    回归模型综合评估。
    返回: da, mse, rmse, mae, r2, z_score, p_value, pred_lag
    """
    da = _directional_accuracy(y_pred, y_true, y_current)
    mse = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(mse))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    z, p = _da_z_score_and_p(da, len(y_true))
    lag = _prediction_lag(y_pred, y_current)
    return {
        "da": da,
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "z_score": z,
        "p_value": p,
        "pred_lag": lag,
    }


# ============================================================
# 模型定义
# ============================================================
class NaivePredictor:
    """Naive基线：预测值 = 当前close"""

    def __init__(self) -> None:
        self.name = "Naive"

    def fit(
        self, X_train: np.ndarray, y_train: np.ndarray,
        X_val: np.ndarray = None, y_val: np.ndarray = None,
    ) -> None:
        pass  # 无需训练

    def predict(self, X: np.ndarray, current_close: np.ndarray) -> np.ndarray:
        return current_close.copy()


class LinearRegressionModel:
    """线性回归基线"""

    def __init__(self) -> None:
        self.name = "LinearRegression"
        self.model = LinearRegression()

    def fit(
        self, X_train: np.ndarray, y_train: np.ndarray,
        X_val: np.ndarray = None, y_val: np.ndarray = None,
    ) -> None:
        self.model.fit(X_train, y_train)

    def predict(self, X: np.ndarray, current_close: np.ndarray) -> np.ndarray:
        return self.model.predict(X)


class LGBMModel:
    """LightGBM 回归，支持 early_stopping on val"""

    def __init__(self) -> None:
        self.name = "LightGBM"
        params = CONFIG["LGB_PARAMS"].copy()
        self.model = lgb.LGBMRegressor(**params)

    def fit(
        self, X_train: np.ndarray, y_train: np.ndarray,
        X_val: np.ndarray = None, y_val: np.ndarray = None,
    ) -> None:
        callbacks = [
            lgb.early_stopping(CONFIG["LGB_EARLY_STOPPING"], verbose=False),
            lgb.log_evaluation(period=-1),
        ]
        if X_val is not None and y_val is not None:
            eval_set = [(X_val, y_val)]
            self.model.fit(
                X_train, y_train,
                eval_set=eval_set,
                callbacks=callbacks,
            )
        else:
            self.model.fit(X_train, y_train)

    def predict(self, X: np.ndarray, current_close: np.ndarray) -> np.ndarray:
        return self.model.predict(X)


# ============================================================
# 单只股票全流程
# ============================================================
def run_single_ticker(
    ticker: str, models: list
) -> Optional[List[Dict]]:
    """对单只股票运行所有模型，返回评估结果列表"""
    data = load_ticker_data(ticker)
    if data is None:
        return None

    train, val, test = data
    try:
        arrays = build_xy(train, val, test)
    except (KeyError, ValueError) as e:
        print(f"[ERROR] {ticker} 特征构建失败: {e}")
        return None

    n_feats = arrays["X_train"].shape[1]
    print(f"  {ticker}: train={arrays['X_train'].shape[0]}, "
          f"val={arrays['X_val'].shape[0]}, test={arrays['X_test'].shape[0]}, "
          f"features={n_feats}")

    results: List[Dict] = []
    for model_cls in models:
        model_obj = model_cls()
        model_obj.fit(
            arrays["X_train"], arrays["y_train"],
            arrays["X_val"], arrays["y_val"],
        )
        y_pred = model_obj.predict(arrays["X_test"], arrays["close_test"])
        metrics = eval_regression(y_pred, arrays["y_test"], arrays["close_test"])
        metrics["ticker"] = ticker
        metrics["model"] = model_obj.name
        metrics["n_test"] = len(arrays["y_test"])
        results.append(metrics)

        _save_predictions(
            ticker, model_obj.name,
            arrays["close_test"], y_pred, arrays["y_test"],
        )

    return results


# ============================================================
# 保存预测结果
# ============================================================
def _save_predictions(
    ticker: str, model_name: str,
    current: np.ndarray, predicted: np.ndarray, actual: np.ndarray,
) -> None:
    """保存单模型单股票的逐条预测到CSV（追加模式）"""
    out_dir = CONFIG["OUTPUT_DIR"]
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{model_name}_reg_predictions.csv")
    df = pd.DataFrame({
        "ticker": ticker,
        "current": current,
        "predicted": predicted,
        "actual": actual,
    })
    header = not os.path.isfile(path)
    try:
        df.to_csv(path, mode="a", header=header, index=False)
    except Exception as e:
        print(f"[WARN] 保存预测失败 {path}: {e}")


# ============================================================
# 汇总表打印
# ============================================================
def _print_per_ticker_table(all_results: List[Dict]) -> None:
    """打印逐ticker Markdown 表"""
    df = pd.DataFrame(all_results)
    cols = ["model", "ticker", "n_test", "da", "mse", "rmse",
            "mae", "r2", "z_score", "p_value", "pred_lag"]
    df = df[[c for c in cols if c in df.columns]]

    fmt_map = {
        "da": "{:.4f}", "mse": "{:.4f}", "rmse": "{:.4f}",
        "mae": "{:.4f}", "r2": "{:.4f}", "z_score": "{:.2f}",
        "p_value": "{:.6f}", "pred_lag": "{:.4f}",
    }
    for col, f in fmt_map.items():
        if col in df.columns:
            df[col] = df[col].apply(lambda x: f.format(x) if pd.notna(x) else "N/A")

    print("\n## Regression Baselines — Per-Ticker Results\n")
    print(df.to_markdown(index=False))


def _print_summary_table(all_results: List[Dict]) -> None:
    """打印按模型汇总的 Markdown 表"""
    df = pd.DataFrame(all_results)
    rows = []
    for model_name in df["model"].unique():
        sub = df[df["model"] == model_name]
        rows.append({
            "model": model_name,
            "mean_da": f"{sub['da'].mean():.4f}",
            "std_da": f"{sub['da'].std():.4f}",
            "mean_rmse": f"{sub['rmse'].mean():.4f}",
            "mean_r2": f"{sub['r2'].mean():.4f}",
            "mean_pred_lag": f"{sub['pred_lag'].mean():.4f}",
            "n_tickers": len(sub),
        })
    print("\n## Regression Baselines — Model Summary\n")
    print(pd.DataFrame(rows).to_markdown(index=False))


# ============================================================
# 清理旧预测文件
# ============================================================
def _cleanup_old_predictions(model_names: List[str]) -> None:
    """删除上一次运行留下的追加式CSV，避免重复数据"""
    out_dir = CONFIG["OUTPUT_DIR"]
    for name in model_names:
        path = os.path.join(out_dir, f"{name}_reg_predictions.csv")
        if os.path.isfile(path):
            os.remove(path)


# ============================================================
# main
# ============================================================
def main() -> None:
    print("=" * 60)
    print("Regression Baselines: Naive / LinearRegression / LightGBM")
    print("=" * 60)

    os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

    model_classes = [NaivePredictor, LinearRegressionModel, LGBMModel]
    model_names = [cls().name for cls in model_classes]
    _cleanup_old_predictions(model_names)

    all_results: List[Dict] = []

    for ticker in CONFIG["TICKERS"]:
        print(f"\n[Processing] {ticker}")
        ticker_results = run_single_ticker(ticker, model_classes)
        if ticker_results is not None:
            all_results.extend(ticker_results)

    if len(all_results) == 0:
        print("[ERROR] 所有股票均无有效数据，请检查路径配置。")
        return

    # 保存汇总CSV
    summary_path = os.path.join(CONFIG["OUTPUT_DIR"], "regression_baselines_summary.csv")
    try:
        pd.DataFrame(all_results).to_csv(summary_path, index=False)
        print(f"\n[SAVED] 汇总结果 → {summary_path}")
    except Exception as e:
        print(f"[WARN] 保存汇总失败: {e}")

    _print_per_ticker_table(all_results)
    _print_summary_table(all_results)


if __name__ == "__main__":
    main()

Regression Baselines: Naive / LinearRegression / LightGBM

[Processing] AAPL
  AAPL: train=8237, val=1175, test=2354, features=40

[Processing] MSFT
  MSFT: train=8234, val=1175, test=2353, features=40

[Processing] GOOGL
  GOOGL: train=7983, val=1139, test=2281, features=40

[Processing] GOOG
  GOOG: train=7995, val=1141, test=2285, features=40

[Processing] NVDA
  NVDA: train=8224, val=1174, test=2350, features=40

[Processing] TSLA
  TSLA: train=8236, val=1175, test=2354, features=40

[Processing] SPY
  SPY: train=8234, val=1175, test=2353, features=40

[Processing] QQQ
  QQQ: train=8234, val=1175, test=2353, features=40

[SAVED] 汇总结果 → results/regression_baselines_summary.csv

## Regression Baselines — Per-Ticker Results

| model            | ticker   |   n_test |     da |     mse |   rmse |    mae |     r2 |   z_score |   p_value |   pred_lag |
|:-----------------|:---------|---------:|-------:|--------:|-------:|-------:|-------:|----------:|----------:|-----------:|
| Naive     

In [ ]:
"""
classification_baselines.py
============================
三分类基线模型：LogisticRegression / LGBMClassifier
标签定义：ret = (next_close - close) / close
  ret >  THRESHOLD → 2 (Up)
  ret < -THRESHOLD → 0 (Down)
  else             → 1 (Flat)
"""

import os
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)
import lightgbm as lgb

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG — 所有超参数 / 路径集中于此
# ============================================================
CONFIG: Dict = {
    "SPLITS_DIR": "splits",
    "FEATURES_DIR": "features",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "FREQ": "1hour",
    "OUTPUT_DIR": "results",
    "TRAIN_RATIO": 0.70,
    "VAL_RATIO": 0.15,
    "THRESHOLD": 0.001,
    "DROP_COLS": ["timestamp", "open", "high", "low", "close", "volume",
                  "date", "datetime", "time", "ts_event"],
    "LGB_PARAMS": {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "is_unbalance": True,
        "verbose": -1,
        "n_jobs": -1,
        "random_state": 42,
    },
    "LGB_EARLY_STOPPING": 20,
    "CLASS_NAMES": {0: "Down", 1: "Flat", 2: "Up"},
}


# ============================================================
# 数据加载（与 regression_baselines.py 相同逻辑）
# ============================================================
def _find_feature_cols(df: pd.DataFrame) -> List[str]:
    """返回所有可用作特征的列名"""
    drop_set = {c.lower() for c in CONFIG["DROP_COLS"]}
    return [c for c in df.columns if c.lower() not in drop_set]


def _load_splits_csv(ticker: str) -> Optional[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    """尝试从 splits/{ticker}_1hour/ 读取 train/val/test"""
    base = os.path.join(CONFIG["SPLITS_DIR"], f"{ticker}_{CONFIG['FREQ']}")
    paths = {k: os.path.join(base, f"{k}.csv") for k in ("train", "val", "test")}
    if not all(os.path.isfile(p) for p in paths.values()):
        return None
    try:
        return (pd.read_csv(paths["train"]),
                pd.read_csv(paths["val"]),
                pd.read_csv(paths["test"]))
    except Exception as e:
        print(f"[WARN] 读取splits失败 {ticker}: {e}")
        return None


def _splits_have_features(df: pd.DataFrame) -> bool:
    """判断splits是否已含技术指标"""
    return len(_find_feature_cols(df)) >= 5


def _load_features_csv(ticker: str) -> Optional[pd.DataFrame]:
    """从 features/ 读取完整特征文件"""
    path = os.path.join(CONFIG["FEATURES_DIR"], f"{ticker}_{CONFIG['FREQ']}_features.csv")
    if not os.path.isfile(path):
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"[WARN] 读取features失败 {ticker}: {e}")
        return None


def _split_by_ratio(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """按 70/15/15 切分"""
    n = len(df)
    n_train = int(n * CONFIG["TRAIN_RATIO"])
    n_val = int(n * CONFIG["VAL_RATIO"])
    return (df.iloc[:n_train].copy(),
            df.iloc[n_train:n_train + n_val].copy(),
            df.iloc[n_train + n_val:].copy())


def load_ticker_data(ticker: str) -> Optional[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    """加载单只股票 train/val/test"""
    splits_result = _load_splits_csv(ticker)
    if splits_result is not None:
        train, val, test = splits_result
        if _splits_have_features(train):
            return train, val, test
        feat_df = _load_features_csv(ticker)
        if feat_df is not None:
            return _split_by_ratio(feat_df)
        return train, val, test
    feat_df = _load_features_csv(ticker)
    if feat_df is not None:
        return _split_by_ratio(feat_df)
    print(f"[ERROR] {ticker}: 无可用数据文件")
    return None


# ============================================================
# 特征 / 标签构建
# ============================================================
def _ensure_close_column(df: pd.DataFrame) -> str:
    """找到close列实际列名"""
    for c in df.columns:
        if c.lower() == "close":
            return c
    raise KeyError("DataFrame 中找不到 close 列")


def _make_label(close_now: np.ndarray, close_next: np.ndarray) -> np.ndarray:
    """
    三分类标签：
      ret >  THRESHOLD → 2 (Up)
      ret < -THRESHOLD → 0 (Down)
      else             → 1 (Flat)
    """
    ret = (close_next - close_now) / close_now
    labels = np.ones(len(ret), dtype=np.int64)  # 默认Flat
    labels[ret > CONFIG["THRESHOLD"]] = 2
    labels[ret < -CONFIG["THRESHOLD"]] = 0
    return labels


def _prepare_split(
    df: pd.DataFrame, close_col: str, feat_cols: List[str]
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    单个split：构建X, y(三分类), current_close
    y基于shift(-1)的close计算return，丢弃最后一行
    """
    df = df.copy()
    df["_next_close"] = df[close_col].shift(-1)
    df = df.dropna(subset=["_next_close"]).reset_index(drop=True)

    available_feats = [c for c in feat_cols if c in df.columns]
    X = df[available_feats].copy()
    X = X.fillna(X.median())
    X = X.fillna(0.0)

    close_now = df[close_col].values.astype(np.float64)
    close_next = df["_next_close"].values.astype(np.float64)
    y = _make_label(close_now, close_next)
    return X.values.astype(np.float64), y, close_now


def build_xy(
    train: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame
) -> Dict[str, np.ndarray]:
    """构建所有split的X/y，StandardScaler fit on train"""
    close_col = _ensure_close_column(train)
    feat_cols = _find_feature_cols(train)
    if len(feat_cols) == 0:
        raise ValueError("无可用特征列")

    X_tr_raw, y_tr, cl_tr = _prepare_split(train, close_col, feat_cols)
    X_va_raw, y_va, cl_va = _prepare_split(val, close_col, feat_cols)
    X_te_raw, y_te, cl_te = _prepare_split(test, close_col, feat_cols)

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr_raw)
    X_va = scaler.transform(X_va_raw)
    X_te = scaler.transform(X_te_raw)

    return {
        "X_train": X_tr, "y_train": y_tr, "close_train": cl_tr,
        "X_val": X_va, "y_val": y_va, "close_val": cl_va,
        "X_test": X_te, "y_test": y_te, "close_test": cl_te,
        "scaler": scaler,
        "feature_names": [c for c in feat_cols if c in train.columns],
    }


# ============================================================
# 评估函数
# ============================================================
def _class_distribution(y: np.ndarray) -> Dict[str, str]:
    """统计各类占比"""
    total = len(y)
    if total == 0:
        return {}
    dist = {}
    for cls_id, cls_name in CONFIG["CLASS_NAMES"].items():
        cnt = int(np.sum(y == cls_id))
        dist[f"{cls_name}_count"] = cnt
        dist[f"{cls_name}_pct"] = cnt / total
    return dist


def _majority_baseline(y: np.ndarray) -> float:
    """多数类基线准确率"""
    if len(y) == 0:
        return 0.0
    counts = np.bincount(y, minlength=3)
    return float(counts.max() / len(y))


def _da_z_score_and_p(acc: float, n: int, baseline: float) -> Tuple[float, float]:
    """
    准确率的z-score（相对于majority baseline）
    H0: acc = baseline
    """
    if n == 0 or baseline <= 0.0 or baseline >= 1.0:
        return 0.0, 1.0
    se = np.sqrt(baseline * (1 - baseline) / n)
    z = (acc - baseline) / se
    p = 1.0 - stats.norm.cdf(z)
    return float(z), float(p)


def eval_classification(y_pred: np.ndarray, y_true: np.ndarray) -> Dict:
    """
    分类模型综合评估。
    返回: accuracy, per_class precision/recall/f1, confusion_matrix,
          class_dist, majority_baseline, z_score, p_value
    """
    acc = float(accuracy_score(y_true, y_pred))
    prec, rec, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], zero_division=0.0
    )
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    dist = _class_distribution(y_true)
    maj = _majority_baseline(y_true)
    z, p = _da_z_score_and_p(acc, len(y_true), maj)

    result: Dict = {
        "accuracy": acc,
        "majority_baseline": maj,
        "z_score": z,
        "p_value": p,
        "n_test": len(y_true),
    }
    # per-class 指标
    for i, cls_name in CONFIG["CLASS_NAMES"].items():
        result[f"{cls_name}_precision"] = float(prec[i])
        result[f"{cls_name}_recall"] = float(rec[i])
        result[f"{cls_name}_f1"] = float(f1[i])
        result[f"{cls_name}_support"] = int(sup[i])
    # 类别分布
    result.update(dist)
    # 混淆矩阵（展平存储）
    result["confusion_matrix"] = cm.tolist()
    return result


# ============================================================
# 模型定义
# ============================================================
class LogRegModel:
    """LogisticRegression with class_weight='balanced'"""

    def __init__(self) -> None:
        self.name = "LogisticRegression"
        self.model = LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42,
            n_jobs=-1,
        )

    def fit(
        self, X_train: np.ndarray, y_train: np.ndarray,
        X_val: np.ndarray = None, y_val: np.ndarray = None,
    ) -> None:
        self.model.fit(X_train, y_train)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.model.predict(X)


class LGBMClfModel:
    """LightGBM Classifier with is_unbalance=True"""

    def __init__(self) -> None:
        self.name = "LightGBM_Clf"
        params = CONFIG["LGB_PARAMS"].copy()
        self.model = lgb.LGBMClassifier(**params)

    def fit(
        self, X_train: np.ndarray, y_train: np.ndarray,
        X_val: np.ndarray = None, y_val: np.ndarray = None,
    ) -> None:
        callbacks = [
            lgb.early_stopping(CONFIG["LGB_EARLY_STOPPING"], verbose=False),
            lgb.log_evaluation(period=-1),
        ]
        if X_val is not None and y_val is not None:
            self.model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                callbacks=callbacks,
            )
        else:
            self.model.fit(X_train, y_train)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.model.predict(X)


# ============================================================
# 单只股票全流程
# ============================================================
def run_single_ticker(
    ticker: str, models: list
) -> Optional[List[Dict]]:
    """对单只股票运行所有分类模型"""
    data = load_ticker_data(ticker)
    if data is None:
        return None

    train, val, test = data
    try:
        arrays = build_xy(train, val, test)
    except (KeyError, ValueError) as e:
        print(f"[ERROR] {ticker} 特征构建失败: {e}")
        return None

    n_feats = arrays["X_train"].shape[1]
    train_dist = _class_distribution(arrays["y_train"])
    print(f"  {ticker}: train={arrays['X_train'].shape[0]}, "
          f"val={arrays['X_val'].shape[0]}, test={arrays['X_test'].shape[0]}, "
          f"features={n_feats}")
    print(f"    train label dist: "
          f"Down={train_dist.get('Down_pct', 0):.1%}  "
          f"Flat={train_dist.get('Flat_pct', 0):.1%}  "
          f"Up={train_dist.get('Up_pct', 0):.1%}")

    results: List[Dict] = []
    for model_cls in models:
        model_obj = model_cls()
        model_obj.fit(
            arrays["X_train"], arrays["y_train"],
            arrays["X_val"], arrays["y_val"],
        )
        y_pred = model_obj.predict(arrays["X_test"])
        metrics = eval_classification(y_pred, arrays["y_test"])
        metrics["ticker"] = ticker
        metrics["model"] = model_obj.name
        results.append(metrics)

        _save_predictions(
            ticker, model_obj.name,
            arrays["y_test"], y_pred, arrays["close_test"],
        )

    return results


# ============================================================
# 保存预测结果
# ============================================================
def _save_predictions(
    ticker: str, model_name: str,
    y_true: np.ndarray, y_pred: np.ndarray, close: np.ndarray,
) -> None:
    """保存逐条预测到CSV（追加模式）"""
    out_dir = CONFIG["OUTPUT_DIR"]
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{model_name}_clf_predictions.csv")
    df = pd.DataFrame({
        "ticker": ticker,
        "close": close,
        "y_true": y_true,
        "y_pred": y_pred,
    })
    header = not os.path.isfile(path)
    try:
        df.to_csv(path, mode="a", header=header, index=False)
    except Exception as e:
        print(f"[WARN] 保存预测失败 {path}: {e}")


# ============================================================
# 汇总表打印
# ============================================================
def _print_per_ticker_table(all_results: List[Dict]) -> None:
    """打印逐ticker结果表"""
    rows = []
    for r in all_results:
        rows.append({
            "model": r["model"],
            "ticker": r["ticker"],
            "n_test": r["n_test"],
            "accuracy": f"{r['accuracy']:.4f}",
            "majority_bl": f"{r['majority_baseline']:.4f}",
            "z_score": f"{r['z_score']:.2f}",
            "p_value": f"{r['p_value']:.6f}",
            "Down_f1": f"{r['Down_f1']:.3f}",
            "Flat_f1": f"{r['Flat_f1']:.3f}",
            "Up_f1": f"{r['Up_f1']:.3f}",
        })
    print("\n## Classification Baselines — Per-Ticker Results\n")
    print(pd.DataFrame(rows).to_markdown(index=False))


def _print_class_dist_table(all_results: List[Dict]) -> None:
    """打印类别分布表"""
    rows = []
    for r in all_results:
        if r["model"] == all_results[0]["model"]:  # 只打印一次
            rows.append({
                "ticker": r["ticker"],
                "Down": f"{r.get('Down_count', 0)} ({r.get('Down_pct', 0):.1%})",
                "Flat": f"{r.get('Flat_count', 0)} ({r.get('Flat_pct', 0):.1%})",
                "Up": f"{r.get('Up_count', 0)} ({r.get('Up_pct', 0):.1%})",
                "majority_bl": f"{r['majority_baseline']:.4f}",
            })
    print(f"\n## Test Set Class Distribution (threshold={CONFIG['THRESHOLD']})\n")
    print(pd.DataFrame(rows).to_markdown(index=False))


def _print_confusion_matrices(all_results: List[Dict]) -> None:
    """打印各模型×股票的混淆矩阵"""
    print("\n## Confusion Matrices\n")
    for r in all_results:
        cm = np.array(r["confusion_matrix"])
        print(f"### {r['model']} — {r['ticker']}")
        cm_df = pd.DataFrame(
            cm,
            index=["true_Down", "true_Flat", "true_Up"],
            columns=["pred_Down", "pred_Flat", "pred_Up"],
        )
        print(cm_df.to_markdown())
        print()


def _print_summary_table(all_results: List[Dict]) -> None:
    """按模型汇总"""
    df = pd.DataFrame(all_results)
    rows = []
    for model_name in df["model"].unique():
        sub = df[df["model"] == model_name]
        rows.append({
            "model": model_name,
            "mean_acc": f"{sub['accuracy'].mean():.4f}",
            "std_acc": f"{sub['accuracy'].std():.4f}",
            "mean_majority_bl": f"{sub['majority_baseline'].mean():.4f}",
            "mean_Down_f1": f"{sub['Down_f1'].mean():.3f}",
            "mean_Flat_f1": f"{sub['Flat_f1'].mean():.3f}",
            "mean_Up_f1": f"{sub['Up_f1'].mean():.3f}",
            "n_tickers": len(sub),
        })
    print("\n## Classification Baselines — Model Summary\n")
    print(pd.DataFrame(rows).to_markdown(index=False))


# ============================================================
# 清理旧预测文件
# ============================================================
def _cleanup_old_predictions(model_names: List[str]) -> None:
    """删除旧追加式CSV"""
    out_dir = CONFIG["OUTPUT_DIR"]
    for name in model_names:
        path = os.path.join(out_dir, f"{name}_clf_predictions.csv")
        if os.path.isfile(path):
            os.remove(path)


# ============================================================
# main
# ============================================================
def main() -> None:
    print("=" * 60)
    print("Classification Baselines: LogisticRegression / LightGBM")
    print(f"Threshold = {CONFIG['THRESHOLD']}")
    print("=" * 60)

    os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

    model_classes = [LogRegModel, LGBMClfModel]
    model_names = [cls().name for cls in model_classes]
    _cleanup_old_predictions(model_names)

    all_results: List[Dict] = []

    for ticker in CONFIG["TICKERS"]:
        print(f"\n[Processing] {ticker}")
        ticker_results = run_single_ticker(ticker, model_classes)
        if ticker_results is not None:
            all_results.extend(ticker_results)

    if len(all_results) == 0:
        print("[ERROR] 所有股票均无有效数据，请检查路径配置。")
        return

    # 保存汇总CSV
    summary_path = os.path.join(CONFIG["OUTPUT_DIR"], "classification_baselines_summary.csv")
    try:
        # confusion_matrix列是list，转字符串存储
        df_save = pd.DataFrame(all_results).copy()
        df_save["confusion_matrix"] = df_save["confusion_matrix"].apply(str)
        df_save.to_csv(summary_path, index=False)
        print(f"\n[SAVED] 汇总结果 → {summary_path}")
    except Exception as e:
        print(f"[WARN] 保存汇总失败: {e}")

    _print_class_dist_table(all_results)
    _print_per_ticker_table(all_results)
    _print_summary_table(all_results)
    _print_confusion_matrices(all_results)


if __name__ == "__main__":
    main()

Classification Baselines: LogisticRegression / LightGBM
Threshold = 0.001

[Processing] AAPL
  AAPL: train=8237, val=1175, test=2354, features=40
    train label dist: Down=32.2%  Flat=32.2%  Up=35.6%

[Processing] MSFT
  MSFT: train=8234, val=1175, test=2353, features=40
    train label dist: Down=31.0%  Flat=34.8%  Up=34.2%

[Processing] GOOGL
  GOOGL: train=7983, val=1139, test=2281, features=40
    train label dist: Down=32.3%  Flat=32.5%  Up=35.2%

[Processing] GOOG
  GOOG: train=7995, val=1141, test=2285, features=40
    train label dist: Down=32.2%  Flat=32.2%  Up=35.6%

[Processing] NVDA
  NVDA: train=8224, val=1174, test=2350, features=40
    train label dist: Down=35.3%  Flat=24.2%  Up=40.5%

[Processing] TSLA
  TSLA: train=8236, val=1175, test=2354, features=40
    train label dist: Down=40.5%  Flat=18.6%  Up=40.9%

[Processing] SPY
  SPY: train=8234, val=1175, test=2353, features=40
    train label dist: Down=24.2%  Flat=48.0%  Up=27.8%

[Processing] QQQ
  QQQ: train=8234, 

In [ ]:
"""
lstm_regression.py
==================
Vanilla LSTM 回归：预测下一根K线close价格
滑动窗口输入 → LSTM → Linear → 单值输出
Colab A100 可运行，支持 CUDA / MPS / CPU 自动检测
"""

import os
import time
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG — 所有超参数 / 路径集中于此
# ============================================================
CONFIG: Dict = {
    "SPLITS_DIR": "splits",
    "FEATURES_DIR": "features",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "FREQ": "1hour",
    "SEQ_LEN": 60,
    "HIDDEN": 128,
    "NUM_LAYERS": 2,
    "DROPOUT": 0.2,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-5,
    "BATCH": 64,
    "EPOCHS": 100,
    "PATIENCE": 10,
    "OUTPUT_DIR": "results",
    "CKPT_DIR": "checkpoints",
    "TRAIN_RATIO": 0.70,
    "VAL_RATIO": 0.15,
    "DROP_COLS": ["timestamp", "open", "high", "low", "close", "volume",
                  "date", "datetime", "time", "ts_event"],
}


# ============================================================
# 设备检测
# ============================================================
def get_device() -> torch.device:
    """自动检测最佳可用设备"""
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        print(f"[DEVICE] CUDA — {torch.cuda.get_device_name(0)}")
        return dev
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("[DEVICE] Apple MPS")
        return torch.device("mps")
    print("[DEVICE] CPU")
    return torch.device("cpu")


# ============================================================
# 数据加载（与 regression_baselines.py 相同逻辑）
# ============================================================
def _find_feature_cols(df: pd.DataFrame) -> List[str]:
    """返回特征列名"""
    drop_set = {c.lower() for c in CONFIG["DROP_COLS"]}
    return [c for c in df.columns if c.lower() not in drop_set]


def _load_splits_csv(ticker: str) -> Optional[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    """从 splits/ 读取"""
    base = os.path.join(CONFIG["SPLITS_DIR"], f"{ticker}_{CONFIG['FREQ']}")
    paths = {k: os.path.join(base, f"{k}.csv") for k in ("train", "val", "test")}
    if not all(os.path.isfile(p) for p in paths.values()):
        return None
    try:
        return (pd.read_csv(paths["train"]),
                pd.read_csv(paths["val"]),
                pd.read_csv(paths["test"]))
    except Exception as e:
        print(f"[WARN] 读取splits失败 {ticker}: {e}")
        return None


def _splits_have_features(df: pd.DataFrame) -> bool:
    """splits是否含技术指标"""
    return len(_find_feature_cols(df)) >= 5


def _load_features_csv(ticker: str) -> Optional[pd.DataFrame]:
    """从 features/ 读取"""
    path = os.path.join(CONFIG["FEATURES_DIR"], f"{ticker}_{CONFIG['FREQ']}_features.csv")
    if not os.path.isfile(path):
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"[WARN] 读取features失败 {ticker}: {e}")
        return None


def _split_by_ratio(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """70/15/15 切分"""
    n = len(df)
    n_train = int(n * CONFIG["TRAIN_RATIO"])
    n_val = int(n * CONFIG["VAL_RATIO"])
    return (df.iloc[:n_train].copy(),
            df.iloc[n_train:n_train + n_val].copy(),
            df.iloc[n_train + n_val:].copy())


def load_ticker_data(ticker: str) -> Optional[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    """加载 train/val/test"""
    splits_result = _load_splits_csv(ticker)
    if splits_result is not None:
        train, val, test = splits_result
        if _splits_have_features(train):
            return train, val, test
        feat_df = _load_features_csv(ticker)
        if feat_df is not None:
            return _split_by_ratio(feat_df)
        return train, val, test
    feat_df = _load_features_csv(ticker)
    if feat_df is not None:
        return _split_by_ratio(feat_df)
    print(f"[ERROR] {ticker}: 无可用数据文件")
    return None


# ============================================================
# 特征 / 标签 / Scaler
# ============================================================
def _ensure_close_column(df: pd.DataFrame) -> str:
    """找到close列名"""
    for c in df.columns:
        if c.lower() == "close":
            return c
    raise KeyError("找不到 close 列")


def prepare_arrays(
    train: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame
) -> Dict[str, np.ndarray]:
    """
    构建连续数组：features(scaled) + close + target(next_close)
    StandardScaler fit on train
    """
    close_col = _ensure_close_column(train)
    feat_cols = _find_feature_cols(train)
    if len(feat_cols) == 0:
        raise ValueError("无可用特征列")

    scaler = StandardScaler()
    result: Dict = {"feature_names": feat_cols, "close_col": close_col}

    for split_name, df in [("train", train), ("val", val), ("test", test)]:
        df = df.copy().reset_index(drop=True)
        available = [c for c in feat_cols if c in df.columns]
        X = df[available].copy()
        X = X.fillna(X.median()).fillna(0.0)

        if split_name == "train":
            X_scaled = scaler.fit_transform(X.values)
        else:
            X_scaled = scaler.transform(X.values)

        close_arr = df[close_col].values.astype(np.float64)
        # target = 下一行close
        target = np.empty(len(close_arr), dtype=np.float64)
        target[:-1] = close_arr[1:]
        target[-1] = np.nan

        result[f"X_{split_name}"] = X_scaled.astype(np.float32)
        result[f"close_{split_name}"] = close_arr
        result[f"target_{split_name}"] = target

    result["scaler"] = scaler
    result["n_features"] = result["X_train"].shape[1]
    return result


# ============================================================
# PyTorch Dataset
# ============================================================
class StockDataset(Dataset):
    """滑动窗口数据集：输入(SEQ_LEN, n_features), 目标=窗口后一根close"""

    def __init__(
        self,
        X: np.ndarray,
        close: np.ndarray,
        target: np.ndarray,
        seq_len: int,
    ) -> None:
        self.X = X
        self.close = close
        self.target = target
        self.seq_len = seq_len
        # 有效索引：窗口起始i，使得 i+seq_len-1 处target有值
        self.valid_indices = self._build_valid_indices()

    def _build_valid_indices(self) -> np.ndarray:
        """筛选有效窗口：最后一个窗口元素的target不能是NaN"""
        max_start = len(self.X) - self.seq_len
        indices = []
        for i in range(max_start):
            end_idx = i + self.seq_len - 1
            if not np.isnan(self.target[end_idx]):
                indices.append(i)
        return np.array(indices, dtype=np.int64)

    def __len__(self) -> int:
        return len(self.valid_indices)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        start = self.valid_indices[idx]
        end = start + self.seq_len
        # 特征窗口: (seq_len, n_features)
        x_seq = torch.from_numpy(self.X[start:end])
        # 目标: 窗口最后一个时间步的next_close
        y = torch.tensor(self.target[end - 1], dtype=torch.float32)
        # 当前close（用于DA计算）
        cur = torch.tensor(self.close[end - 1], dtype=torch.float32)
        return x_seq, y, cur


def build_dataloaders(arrays: Dict[str, np.ndarray]) -> Dict[str, DataLoader]:
    """构建 train/val/test DataLoader"""
    seq_len = CONFIG["SEQ_LEN"]
    batch = CONFIG["BATCH"]
    loaders: Dict[str, DataLoader] = {}

    for split in ("train", "val", "test"):
        ds = StockDataset(
            X=arrays[f"X_{split}"],
            close=arrays[f"close_{split}"],
            target=arrays[f"target_{split}"],
            seq_len=seq_len,
        )
        loaders[split] = DataLoader(
            ds,
            batch_size=batch,
            shuffle=False,  # 时序数据不shuffle
            num_workers=0,
            pin_memory=True,
            drop_last=False,
        )

    return loaders


# ============================================================
# 模型
# ============================================================
class VanillaLSTM(nn.Module):
    """标准 LSTM 回归：预测下一根close"""

    def __init__(self, input_size: int) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=CONFIG["HIDDEN"],
            num_layers=CONFIG["NUM_LAYERS"],
            dropout=CONFIG["DROPOUT"] if CONFIG["NUM_LAYERS"] > 1 else 0.0,
            batch_first=True,
        )
        self.fc = nn.Linear(CONFIG["HIDDEN"], 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, n_features)
        out, _ = self.lstm(x)
        # 取最后一个时间步
        last = out[:, -1, :]  # (batch, hidden)
        return self.fc(last).squeeze(-1)  # (batch,)


# ============================================================
# 训练循环
# ============================================================
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float:
    """单epoch训练，返回平均loss"""
    model.train()
    total_loss = 0.0
    n_batches = 0
    for x_seq, y, _ in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad()
        pred = model(x_seq)
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def eval_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    """单epoch验证，返回平均loss"""
    model.eval()
    total_loss = 0.0
    n_batches = 0
    for x_seq, y, _ in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        pred = model(x_seq)
        loss = criterion(pred, y)
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)


def train_model(
    model: nn.Module,
    loaders: Dict[str, DataLoader],
    device: torch.device,
    ticker: str,
) -> nn.Module:
    """完整训练流程：early stopping + best checkpoint"""
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CONFIG["LR"],
        weight_decay=CONFIG["WEIGHT_DECAY"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5
    )

    ckpt_dir = CONFIG["CKPT_DIR"]
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, f"vanilla_lstm_{ticker}.pt")

    best_val_loss = float("inf")
    patience_counter = 0
    best_epoch = 0

    for epoch in range(1, CONFIG["EPOCHS"] + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, loaders["train"], criterion, optimizer, device)
        val_loss = eval_one_epoch(model, loaders["val"], criterion, device)
        scheduler.step(val_loss)
        elapsed = time.time() - t0

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_epoch = epoch
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1

        if epoch <= 5 or epoch % 10 == 0 or patience_counter >= CONFIG["PATIENCE"]:
            lr_now = optimizer.param_groups[0]["lr"]
            print(f"    Epoch {epoch:3d} | "
                  f"train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | "
                  f"lr={lr_now:.1e} | {elapsed:.1f}s | "
                  f"patience={patience_counter}/{CONFIG['PATIENCE']}")

        if patience_counter >= CONFIG["PATIENCE"]:
            print(f"    Early stopping at epoch {epoch}, best={best_epoch}")
            break

    # 加载最优权重
    if os.path.isfile(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    return model


# ============================================================
# 推理 & 评估
# ============================================================
@torch.no_grad()
def predict_all(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """对loader全量推理，返回 (y_pred, y_true, current_close)"""
    model.eval()
    preds, trues, closes = [], [], []
    for x_seq, y, cur in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        pred = model(x_seq)
        preds.append(pred.cpu().numpy())
        trues.append(y.numpy())
        closes.append(cur.numpy())
    return (np.concatenate(preds),
            np.concatenate(trues),
            np.concatenate(closes))


def _directional_accuracy(
    y_pred: np.ndarray, y_true: np.ndarray, y_current: np.ndarray
) -> float:
    """方向准确率"""
    pred_dir = np.sign(y_pred - y_current)
    true_dir = np.sign(y_true - y_current)
    mask = true_dir != 0
    if mask.sum() == 0:
        return 0.5
    return float(np.mean(pred_dir[mask] == true_dir[mask]))


def _prediction_lag(y_pred: np.ndarray, y_current: np.ndarray) -> float:
    """预测滞后度"""
    if len(y_pred) < 3:
        return np.nan
    return float(np.corrcoef(y_pred, y_current)[0, 1])


def _da_z_score_and_p(da: float, n: int) -> Tuple[float, float]:
    """DA z-score (H0: DA=0.5)"""
    if n == 0:
        return 0.0, 1.0
    se = np.sqrt(0.25 / n)
    z = (da - 0.5) / se
    p = 1.0 - stats.norm.cdf(z)
    return float(z), float(p)


def eval_regression(
    y_pred: np.ndarray, y_true: np.ndarray, y_current: np.ndarray
) -> Dict[str, float]:
    """综合评估"""
    da = _directional_accuracy(y_pred, y_true, y_current)
    mse = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(mse))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    z, p = _da_z_score_and_p(da, len(y_true))
    lag = _prediction_lag(y_pred, y_current)
    return {
        "da": da, "mse": mse, "rmse": rmse, "mae": mae,
        "r2": r2, "z_score": z, "p_value": p, "pred_lag": lag,
    }


# ============================================================
# 保存预测
# ============================================================
def _save_predictions(
    ticker: str,
    current: np.ndarray,
    predicted: np.ndarray,
    actual: np.ndarray,
    path: str,
) -> None:
    """追加保存预测CSV"""
    df = pd.DataFrame({
        "ticker": ticker,
        "current": current,
        "predicted": predicted,
        "actual": actual,
    })
    header = not os.path.isfile(path)
    try:
        df.to_csv(path, mode="a", header=header, index=False)
    except Exception as e:
        print(f"[WARN] 保存预测失败 {path}: {e}")


# ============================================================
# 单只股票全流程
# ============================================================
def run_single_ticker(
    ticker: str, device: torch.device
) -> Optional[Dict]:
    """对单只股票训练 + 评估"""
    data = load_ticker_data(ticker)
    if data is None:
        return None

    train, val, test = data
    try:
        arrays = prepare_arrays(train, val, test)
    except (KeyError, ValueError) as e:
        print(f"[ERROR] {ticker} 特征构建失败: {e}")
        return None

    n_features = arrays["n_features"]
    print(f"  {ticker}: features={n_features}, "
          f"train={len(arrays['X_train'])}, "
          f"val={len(arrays['X_val'])}, "
          f"test={len(arrays['X_test'])}")

    loaders = build_dataloaders(arrays)
    print(f"  DataLoader samples: "
          f"train={len(loaders['train'].dataset)}, "
          f"val={len(loaders['val'].dataset)}, "
          f"test={len(loaders['test'].dataset)}")

    # 构建 & 训练模型
    model = VanillaLSTM(input_size=n_features).to(device)
    model = train_model(model, loaders, device, ticker)

    # 测试集推理
    y_pred, y_true, y_current = predict_all(model, loaders["test"], device)
    metrics = eval_regression(y_pred, y_true, y_current)
    metrics["ticker"] = ticker
    metrics["n_test"] = len(y_true)
    metrics["model"] = "VanillaLSTM"

    # 保存逐条预测
    pred_path = os.path.join(CONFIG["OUTPUT_DIR"], "vanilla_lstm_reg_predictions.csv")
    _save_predictions(ticker, y_current, y_pred, y_true, pred_path)

    print(f"  → DA={metrics['da']:.4f} | RMSE={metrics['rmse']:.4f} | "
          f"R²={metrics['r2']:.4f} | pred_lag={metrics['pred_lag']:.4f} | "
          f"z={metrics['z_score']:.2f}")

    # 释放显存
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics


# ============================================================
# 汇总表打印
# ============================================================
def _print_per_ticker_table(all_results: List[Dict]) -> None:
    """逐ticker Markdown表"""
    df = pd.DataFrame(all_results)
    cols = ["model", "ticker", "n_test", "da", "mse", "rmse",
            "mae", "r2", "z_score", "p_value", "pred_lag"]
    df = df[[c for c in cols if c in df.columns]]

    fmt_map = {
        "da": "{:.4f}", "mse": "{:.4f}", "rmse": "{:.4f}",
        "mae": "{:.4f}", "r2": "{:.4f}", "z_score": "{:.2f}",
        "p_value": "{:.6f}", "pred_lag": "{:.4f}",
    }
    for col, f in fmt_map.items():
        if col in df.columns:
            df[col] = df[col].apply(lambda x: f.format(x) if pd.notna(x) else "N/A")

    print("\n## Vanilla LSTM Regression — Per-Ticker Results\n")
    print(df.to_markdown(index=False))


def _print_summary(all_results: List[Dict]) -> None:
    """模型汇总"""
    df = pd.DataFrame(all_results)
    print("\n## Vanilla LSTM Regression — Summary\n")
    summary = {
        "mean_da": f"{df['da'].mean():.4f}",
        "std_da": f"{df['da'].std():.4f}",
        "mean_rmse": f"{df['rmse'].mean():.4f}",
        "mean_r2": f"{df['r2'].mean():.4f}",
        "mean_pred_lag": f"{df['pred_lag'].mean():.4f}",
        "n_tickers": len(df),
    }
    print(pd.DataFrame([summary]).to_markdown(index=False))


# ============================================================
# main
# ============================================================
def main() -> None:
    print("=" * 60)
    print("Vanilla LSTM Regression — Next Close Prediction")
    print(f"SEQ_LEN={CONFIG['SEQ_LEN']} | HIDDEN={CONFIG['HIDDEN']} | "
          f"LAYERS={CONFIG['NUM_LAYERS']} | EPOCHS≤{CONFIG['EPOCHS']}")
    print("=" * 60)

    os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
    os.makedirs(CONFIG["CKPT_DIR"], exist_ok=True)

    # 清理旧预测文件
    pred_path = os.path.join(CONFIG["OUTPUT_DIR"], "vanilla_lstm_reg_predictions.csv")
    if os.path.isfile(pred_path):
        os.remove(pred_path)

    device = get_device()
    all_results: List[Dict] = []

    for ticker in CONFIG["TICKERS"]:
        print(f"\n{'='*40}")
        print(f"[Processing] {ticker}")
        print(f"{'='*40}")
        result = run_single_ticker(ticker, device)
        if result is not None:
            all_results.append(result)

    if len(all_results) == 0:
        print("[ERROR] 所有股票均无有效数据。")
        return

    # 保存汇总
    summary_path = os.path.join(CONFIG["OUTPUT_DIR"], "vanilla_lstm_reg_summary.csv")
    try:
        pd.DataFrame(all_results).to_csv(summary_path, index=False)
        print(f"\n[SAVED] 汇总 → {summary_path}")
    except Exception as e:
        print(f"[WARN] 保存汇总失败: {e}")

    _print_per_ticker_table(all_results)
    _print_summary(all_results)


if __name__ == "__main__":
    main()

Vanilla LSTM Regression — Next Close Prediction
SEQ_LEN=60 | HIDDEN=128 | LAYERS=2 | EPOCHS≤100
[DEVICE] CUDA — NVIDIA RTX PRO 6000 Blackwell Server Edition

[Processing] AAPL
  AAPL: features=40, train=8238, val=1176, test=2355
  DataLoader samples: train=8178, val=1116, test=2295
    Epoch   1 | train_loss=12466.215826 | val_loss=19319.473524 | lr=1.0e-03 | 1.1s | patience=0/10
    Epoch   2 | train_loss=8923.536473 | val_loss=14897.266168 | lr=1.0e-03 | 0.3s | patience=0/10
    Epoch   3 | train_loss=6151.656594 | val_loss=11106.725505 | lr=1.0e-03 | 0.3s | patience=0/10
    Epoch   4 | train_loss=3961.777190 | val_loss=7892.237250 | lr=1.0e-03 | 0.2s | patience=0/10
    Epoch   5 | train_loss=2591.953722 | val_loss=5694.630547 | lr=1.0e-03 | 0.3s | patience=0/10
    Epoch  10 | train_loss=881.668288 | val_loss=1430.514097 | lr=1.0e-03 | 0.2s | patience=0/10
    Epoch  20 | train_loss=973.538506 | val_loss=1372.028039 | lr=1.0e-03 | 0.2s | patience=2/10
    Epoch  30 | train_loss=20

In [ ]:
"""
lstm_classification.py
======================
Vanilla LSTM 三分类：预测下一根K线方向
标签：ret = (next_close - close) / close
  ret >  THRESHOLD → 2 (Up)
  ret < -THRESHOLD → 0 (Down)
  else             → 1 (Flat)
Colab A100 / RTX PRO 6000 可运行
"""

import os
import time
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG — 所有超参数 / 路径集中于此
# ============================================================
CONFIG: Dict = {
    "SPLITS_DIR": "splits",
    "FEATURES_DIR": "features",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "FREQ": "1hour",
    "SEQ_LEN": 60,
    "HIDDEN": 128,
    "NUM_LAYERS": 2,
    "DROPOUT": 0.2,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-5,
    "BATCH": 64,
    "EPOCHS": 100,
    "PATIENCE": 10,
    "THRESHOLD": 0.001,
    "OUTPUT_DIR": "results",
    "CKPT_DIR": "checkpoints",
    "TRAIN_RATIO": 0.70,
    "VAL_RATIO": 0.15,
    "DROP_COLS": ["timestamp", "open", "high", "low", "close", "volume",
                  "date", "datetime", "time", "ts_event"],
    "NUM_CLASSES": 3,
    "CLASS_NAMES": {0: "Down", 1: "Flat", 2: "Up"},
}


# ============================================================
# 设备检测
# ============================================================
def get_device() -> torch.device:
    """自动检测最佳可用设备"""
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        print(f"[DEVICE] CUDA — {torch.cuda.get_device_name(0)}")
        return dev
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("[DEVICE] Apple MPS")
        return torch.device("mps")
    print("[DEVICE] CPU")
    return torch.device("cpu")


# ============================================================
# 数据加载
# ============================================================
def _find_feature_cols(df: pd.DataFrame) -> List[str]:
    """返回特征列名"""
    drop_set = {c.lower() for c in CONFIG["DROP_COLS"]}
    return [c for c in df.columns if c.lower() not in drop_set]


def _load_splits_csv(ticker: str) -> Optional[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    """从 splits/ 读取"""
    base = os.path.join(CONFIG["SPLITS_DIR"], f"{ticker}_{CONFIG['FREQ']}")
    paths = {k: os.path.join(base, f"{k}.csv") for k in ("train", "val", "test")}
    if not all(os.path.isfile(p) for p in paths.values()):
        return None
    try:
        return (pd.read_csv(paths["train"]),
                pd.read_csv(paths["val"]),
                pd.read_csv(paths["test"]))
    except Exception as e:
        print(f"[WARN] 读取splits失败 {ticker}: {e}")
        return None


def _splits_have_features(df: pd.DataFrame) -> bool:
    """splits是否含技术指标"""
    return len(_find_feature_cols(df)) >= 5


def _load_features_csv(ticker: str) -> Optional[pd.DataFrame]:
    """从 features/ 读取"""
    path = os.path.join(CONFIG["FEATURES_DIR"], f"{ticker}_{CONFIG['FREQ']}_features.csv")
    if not os.path.isfile(path):
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"[WARN] 读取features失败 {ticker}: {e}")
        return None


def _split_by_ratio(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """70/15/15 切分"""
    n = len(df)
    n_train = int(n * CONFIG["TRAIN_RATIO"])
    n_val = int(n * CONFIG["VAL_RATIO"])
    return (df.iloc[:n_train].copy(),
            df.iloc[n_train:n_train + n_val].copy(),
            df.iloc[n_train + n_val:].copy())


def load_ticker_data(ticker: str) -> Optional[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    """加载 train/val/test"""
    splits_result = _load_splits_csv(ticker)
    if splits_result is not None:
        train, val, test = splits_result
        if _splits_have_features(train):
            return train, val, test
        feat_df = _load_features_csv(ticker)
        if feat_df is not None:
            return _split_by_ratio(feat_df)
        return train, val, test
    feat_df = _load_features_csv(ticker)
    if feat_df is not None:
        return _split_by_ratio(feat_df)
    print(f"[ERROR] {ticker}: 无可用数据文件")
    return None


# ============================================================
# 标签构建
# ============================================================
def _ensure_close_column(df: pd.DataFrame) -> str:
    """找到close列名"""
    for c in df.columns:
        if c.lower() == "close":
            return c
    raise KeyError("找不到 close 列")


def _make_label(close_now: np.ndarray, close_next: np.ndarray) -> np.ndarray:
    """
    三分类标签：
      ret >  THRESHOLD → 2 (Up)
      ret < -THRESHOLD → 0 (Down)
      else             → 1 (Flat)
    """
    ret = (close_next - close_now) / close_now
    labels = np.ones(len(ret), dtype=np.int64)
    labels[ret > CONFIG["THRESHOLD"]] = 2
    labels[ret < -CONFIG["THRESHOLD"]] = 0
    return labels


# ============================================================
# 特征 / Scaler
# ============================================================
def prepare_arrays(
    train: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame
) -> Dict[str, np.ndarray]:
    """
    构建连续数组：features(scaled) + close + label
    StandardScaler fit on train
    """
    close_col = _ensure_close_column(train)
    feat_cols = _find_feature_cols(train)
    if len(feat_cols) == 0:
        raise ValueError("无可用特征列")

    scaler = StandardScaler()
    result: Dict = {"feature_names": feat_cols, "close_col": close_col}

    for split_name, df in [("train", train), ("val", val), ("test", test)]:
        df = df.copy().reset_index(drop=True)
        available = [c for c in feat_cols if c in df.columns]
        X = df[available].copy()
        X = X.fillna(X.median()).fillna(0.0)

        if split_name == "train":
            X_scaled = scaler.fit_transform(X.values)
        else:
            X_scaled = scaler.transform(X.values)

        close_arr = df[close_col].values.astype(np.float64)
        # next_close 用于生成标签
        next_close = np.empty(len(close_arr), dtype=np.float64)
        next_close[:-1] = close_arr[1:]
        next_close[-1] = np.nan
        # 生成标签（最后一行为NaN的位置用-1标记，后续dataset排除）
        labels = np.full(len(close_arr), -1, dtype=np.int64)
        valid_mask = ~np.isnan(next_close)
        labels[valid_mask] = _make_label(close_arr[valid_mask], next_close[valid_mask])

        result[f"X_{split_name}"] = X_scaled.astype(np.float32)
        result[f"close_{split_name}"] = close_arr
        result[f"label_{split_name}"] = labels

    result["scaler"] = scaler
    result["n_features"] = result["X_train"].shape[1]
    return result


def _compute_class_weights(labels: np.ndarray, device: torch.device) -> torch.Tensor:
    """
    计算类别权重：weight_i = 1/count_i，归一化使和=num_classes
    """
    valid = labels[labels >= 0]
    counts = np.bincount(valid, minlength=CONFIG["NUM_CLASSES"]).astype(np.float64)
    # 避免除零
    counts = np.maximum(counts, 1.0)
    weights = 1.0 / counts
    weights = weights / weights.sum() * CONFIG["NUM_CLASSES"]
    return torch.tensor(weights, dtype=torch.float32).to(device)


# ============================================================
# PyTorch Dataset
# ============================================================
class StockDataset(Dataset):
    """滑动窗口数据集：输入(SEQ_LEN, n_features), 目标=分类标签"""

    def __init__(
        self,
        X: np.ndarray,
        close: np.ndarray,
        labels: np.ndarray,
        seq_len: int,
    ) -> None:
        self.X = X
        self.close = close
        self.labels = labels
        self.seq_len = seq_len
        self.valid_indices = self._build_valid_indices()

    def _build_valid_indices(self) -> np.ndarray:
        """筛选有效窗口：最后一个窗口元素的label不为-1"""
        max_start = len(self.X) - self.seq_len
        indices = []
        for i in range(max_start):
            end_idx = i + self.seq_len - 1
            if self.labels[end_idx] >= 0:
                indices.append(i)
        return np.array(indices, dtype=np.int64)

    def __len__(self) -> int:
        return len(self.valid_indices)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        start = self.valid_indices[idx]
        end = start + self.seq_len
        x_seq = torch.from_numpy(self.X[start:end])
        label = torch.tensor(self.labels[end - 1], dtype=torch.long)
        return x_seq, label


def build_dataloaders(arrays: Dict[str, np.ndarray]) -> Dict[str, DataLoader]:
    """构建 train/val/test DataLoader"""
    seq_len = CONFIG["SEQ_LEN"]
    batch = CONFIG["BATCH"]
    loaders: Dict[str, DataLoader] = {}

    for split in ("train", "val", "test"):
        ds = StockDataset(
            X=arrays[f"X_{split}"],
            close=arrays[f"close_{split}"],
            labels=arrays[f"label_{split}"],
            seq_len=seq_len,
        )
        loaders[split] = DataLoader(
            ds,
            batch_size=batch,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            drop_last=False,
        )

    return loaders


# ============================================================
# 模型
# ============================================================
class VanillaLSTMClf(nn.Module):
    """LSTM 三分类：输出logits（不加softmax，由CrossEntropyLoss内部处理）"""

    def __init__(self, input_size: int) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=CONFIG["HIDDEN"],
            num_layers=CONFIG["NUM_LAYERS"],
            dropout=CONFIG["DROPOUT"] if CONFIG["NUM_LAYERS"] > 1 else 0.0,
            batch_first=True,
        )
        self.fc = nn.Linear(CONFIG["HIDDEN"], CONFIG["NUM_CLASSES"])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, n_features)
        out, _ = self.lstm(x)
        last = out[:, -1, :]  # (batch, hidden)
        return self.fc(last)  # (batch, num_classes)


# ============================================================
# 训练循环
# ============================================================
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> Tuple[float, float]:
    """单epoch训练，返回 (avg_loss, accuracy)"""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for x_seq, label in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(x_seq)
        loss = criterion(logits, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * label.size(0)
        correct += (logits.argmax(dim=1) == label).sum().item()
        total += label.size(0)
    avg_loss = total_loss / max(total, 1)
    acc = correct / max(total, 1)
    return avg_loss, acc


@torch.no_grad()
def eval_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> Tuple[float, float]:
    """单epoch验证，返回 (avg_loss, accuracy)"""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    for x_seq, label in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)
        logits = model(x_seq)
        loss = criterion(logits, label)
        total_loss += loss.item() * label.size(0)
        correct += (logits.argmax(dim=1) == label).sum().item()
        total += label.size(0)
    avg_loss = total_loss / max(total, 1)
    acc = correct / max(total, 1)
    return avg_loss, acc


def train_model(
    model: nn.Module,
    loaders: Dict[str, DataLoader],
    criterion: nn.Module,
    device: torch.device,
    ticker: str,
) -> nn.Module:
    """完整训练：early stopping on val_loss + best checkpoint"""
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CONFIG["LR"],
        weight_decay=CONFIG["WEIGHT_DECAY"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5
    )

    ckpt_dir = CONFIG["CKPT_DIR"]
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, f"vanilla_lstm_clf_{ticker}.pt")

    best_val_loss = float("inf")
    patience_counter = 0
    best_epoch = 0

    for epoch in range(1, CONFIG["EPOCHS"] + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, loaders["train"], criterion, optimizer, device)
        val_loss, val_acc = eval_one_epoch(model, loaders["val"], criterion, device)
        scheduler.step(val_loss)
        elapsed = time.time() - t0

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_epoch = epoch
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1

        if epoch <= 5 or epoch % 10 == 0 or patience_counter >= CONFIG["PATIENCE"]:
            lr_now = optimizer.param_groups[0]["lr"]
            print(f"    Epoch {epoch:3d} | "
                  f"t_loss={train_loss:.4f} t_acc={train_acc:.4f} | "
                  f"v_loss={val_loss:.4f} v_acc={val_acc:.4f} | "
                  f"lr={lr_now:.1e} | {elapsed:.1f}s | "
                  f"pat={patience_counter}/{CONFIG['PATIENCE']}")

        if patience_counter >= CONFIG["PATIENCE"]:
            print(f"    Early stopping at epoch {epoch}, best={best_epoch}")
            break

    if os.path.isfile(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    return model


# ============================================================
# 推理 & 评估
# ============================================================
@torch.no_grad()
def predict_all(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray]:
    """全量推理，返回 (y_pred, y_true)"""
    model.eval()
    preds, trues = [], []
    for x_seq, label in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        logits = model(x_seq)
        preds.append(logits.argmax(dim=1).cpu().numpy())
        trues.append(label.numpy())
    return np.concatenate(preds), np.concatenate(trues)


def _class_distribution(y: np.ndarray) -> Dict[str, str]:
    """类别分布字符串"""
    total = len(y)
    if total == 0:
        return {}
    dist = {}
    for cls_id, cls_name in CONFIG["CLASS_NAMES"].items():
        cnt = int(np.sum(y == cls_id))
        dist[f"{cls_name}_count"] = cnt
        dist[f"{cls_name}_pct"] = cnt / total
    return dist


def _majority_baseline(y: np.ndarray) -> float:
    """多数类基线"""
    if len(y) == 0:
        return 0.0
    counts = np.bincount(y, minlength=CONFIG["NUM_CLASSES"])
    return float(counts.max() / len(y))


def _acc_z_score_and_p(acc: float, n: int, baseline: float) -> Tuple[float, float]:
    """准确率 z-score（相对majority baseline）"""
    if n == 0 or baseline <= 0.0 or baseline >= 1.0:
        return 0.0, 1.0
    se = np.sqrt(baseline * (1 - baseline) / n)
    z = (acc - baseline) / se
    p = 1.0 - stats.norm.cdf(z)
    return float(z), float(p)


def eval_classification(y_pred: np.ndarray, y_true: np.ndarray) -> Dict:
    """分类综合评估"""
    acc = float(accuracy_score(y_true, y_pred))
    prec, rec, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], zero_division=0.0
    )
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    dist = _class_distribution(y_true)
    maj = _majority_baseline(y_true)
    z, p = _acc_z_score_and_p(acc, len(y_true), maj)

    result: Dict = {
        "accuracy": acc,
        "majority_baseline": maj,
        "z_score": z,
        "p_value": p,
        "n_test": len(y_true),
    }
    for i, cls_name in CONFIG["CLASS_NAMES"].items():
        result[f"{cls_name}_precision"] = float(prec[i])
        result[f"{cls_name}_recall"] = float(rec[i])
        result[f"{cls_name}_f1"] = float(f1[i])
        result[f"{cls_name}_support"] = int(sup[i])
    result.update(dist)
    result["confusion_matrix"] = cm.tolist()
    return result


# ============================================================
# 单只股票全流程
# ============================================================
def run_single_ticker(
    ticker: str, device: torch.device
) -> Optional[Dict]:
    """训练 + 评估"""
    data = load_ticker_data(ticker)
    if data is None:
        return None

    train, val, test = data
    try:
        arrays = prepare_arrays(train, val, test)
    except (KeyError, ValueError) as e:
        print(f"[ERROR] {ticker} 特征构建失败: {e}")
        return None

    n_features = arrays["n_features"]

    # 打印训练集类别分布
    train_labels = arrays["label_train"]
    train_valid = train_labels[train_labels >= 0]
    train_dist = _class_distribution(train_valid)
    print(f"  {ticker}: features={n_features}, "
          f"train={len(arrays['X_train'])}, "
          f"val={len(arrays['X_val'])}, "
          f"test={len(arrays['X_test'])}")
    print(f"    train dist: "
          f"Down={train_dist.get('Down_pct', 0):.1%}  "
          f"Flat={train_dist.get('Flat_pct', 0):.1%}  "
          f"Up={train_dist.get('Up_pct', 0):.1%}")

    loaders = build_dataloaders(arrays)
    print(f"  DataLoader samples: "
          f"train={len(loaders['train'].dataset)}, "
          f"val={len(loaders['val'].dataset)}, "
          f"test={len(loaders['test'].dataset)}")

    # 类别权重
    class_weights = _compute_class_weights(train_valid, device)
    print(f"    class_weights: {class_weights.cpu().numpy().round(3)}")
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # 构建 & 训练
    model = VanillaLSTMClf(input_size=n_features).to(device)
    model = train_model(model, loaders, criterion, device, ticker)

    # 测试集推理
    y_pred, y_true = predict_all(model, loaders["test"], device)
    metrics = eval_classification(y_pred, y_true)
    metrics["ticker"] = ticker
    metrics["model"] = "VanillaLSTM_Clf"

    # 保存逐条预测
    _save_predictions(ticker, y_true, y_pred)

    print(f"  → acc={metrics['accuracy']:.4f} | "
          f"majority_bl={metrics['majority_baseline']:.4f} | "
          f"z={metrics['z_score']:.2f} | "
          f"Down_f1={metrics['Down_f1']:.3f} | "
          f"Flat_f1={metrics['Flat_f1']:.3f} | "
          f"Up_f1={metrics['Up_f1']:.3f}")

    # 释放显存
    del model, criterion
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics


# ============================================================
# 保存预测
# ============================================================
def _save_predictions(
    ticker: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> None:
    """追加保存预测CSV"""
    out_dir = CONFIG["OUTPUT_DIR"]
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "vanilla_lstm_clf_predictions.csv")
    df = pd.DataFrame({
        "ticker": ticker,
        "y_true": y_true,
        "y_pred": y_pred,
    })
    header = not os.path.isfile(path)
    try:
        df.to_csv(path, mode="a", header=header, index=False)
    except Exception as e:
        print(f"[WARN] 保存预测失败 {path}: {e}")


# ============================================================
# 汇总表打印
# ============================================================
def _print_per_ticker_table(all_results: List[Dict]) -> None:
    """逐ticker结果表"""
    rows = []
    for r in all_results:
        rows.append({
            "model": r["model"],
            "ticker": r["ticker"],
            "n_test": r["n_test"],
            "accuracy": f"{r['accuracy']:.4f}",
            "majority_bl": f"{r['majority_baseline']:.4f}",
            "z_score": f"{r['z_score']:.2f}",
            "p_value": f"{r['p_value']:.6f}",
            "Down_f1": f"{r['Down_f1']:.3f}",
            "Flat_f1": f"{r['Flat_f1']:.3f}",
            "Up_f1": f"{r['Up_f1']:.3f}",
        })
    print("\n## Vanilla LSTM Classification — Per-Ticker Results\n")
    print(pd.DataFrame(rows).to_markdown(index=False))


def _print_class_dist_table(all_results: List[Dict]) -> None:
    """类别分布表"""
    rows = []
    for r in all_results:
        rows.append({
            "ticker": r["ticker"],
            "Down": f"{r.get('Down_count', 0)} ({r.get('Down_pct', 0):.1%})",
            "Flat": f"{r.get('Flat_count', 0)} ({r.get('Flat_pct', 0):.1%})",
            "Up": f"{r.get('Up_count', 0)} ({r.get('Up_pct', 0):.1%})",
            "majority_bl": f"{r['majority_baseline']:.4f}",
        })
    print(f"\n## Test Set Class Distribution (threshold={CONFIG['THRESHOLD']})\n")
    print(pd.DataFrame(rows).to_markdown(index=False))


def _print_confusion_matrices(all_results: List[Dict]) -> None:
    """打印混淆矩阵"""
    print("\n## Confusion Matrices\n")
    for r in all_results:
        cm = np.array(r["confusion_matrix"])
        print(f"### {r['ticker']}")
        cm_df = pd.DataFrame(
            cm,
            index=["true_Down", "true_Flat", "true_Up"],
            columns=["pred_Down", "pred_Flat", "pred_Up"],
        )
        print(cm_df.to_markdown())
        print()


def _print_summary(all_results: List[Dict]) -> None:
    """模型汇总"""
    df = pd.DataFrame(all_results)
    summary = {
        "mean_acc": f"{df['accuracy'].mean():.4f}",
        "std_acc": f"{df['accuracy'].std():.4f}",
        "mean_majority_bl": f"{df['majority_baseline'].mean():.4f}",
        "mean_Down_f1": f"{df['Down_f1'].mean():.3f}",
        "mean_Flat_f1": f"{df['Flat_f1'].mean():.3f}",
        "mean_Up_f1": f"{df['Up_f1'].mean():.3f}",
        "n_tickers": len(df),
    }
    print("\n## Vanilla LSTM Classification — Summary\n")
    print(pd.DataFrame([summary]).to_markdown(index=False))


# ============================================================
# main
# ============================================================
def main() -> None:
    print("=" * 60)
    print("Vanilla LSTM Classification — 3-Class Direction Prediction")
    print(f"SEQ_LEN={CONFIG['SEQ_LEN']} | HIDDEN={CONFIG['HIDDEN']} | "
          f"LAYERS={CONFIG['NUM_LAYERS']} | THRESHOLD={CONFIG['THRESHOLD']}")
    print("=" * 60)

    os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
    os.makedirs(CONFIG["CKPT_DIR"], exist_ok=True)

    # 清理旧预测文件
    pred_path = os.path.join(CONFIG["OUTPUT_DIR"], "vanilla_lstm_clf_predictions.csv")
    if os.path.isfile(pred_path):
        os.remove(pred_path)

    device = get_device()
    all_results: List[Dict] = []

    for ticker in CONFIG["TICKERS"]:
        print(f"\n{'='*40}")
        print(f"[Processing] {ticker}")
        print(f"{'='*40}")
        result = run_single_ticker(ticker, device)
        if result is not None:
            all_results.append(result)

    if len(all_results) == 0:
        print("[ERROR] 所有股票均无有效数据。")
        return

    # 保存汇总CSV
    summary_path = os.path.join(CONFIG["OUTPUT_DIR"], "vanilla_lstm_clf_summary.csv")
    try:
        df_save = pd.DataFrame(all_results).copy()
        df_save["confusion_matrix"] = df_save["confusion_matrix"].apply(str)
        df_save.to_csv(summary_path, index=False)
        print(f"\n[SAVED] 汇总 → {summary_path}")
    except Exception as e:
        print(f"[WARN] 保存汇总失败: {e}")

    _print_class_dist_table(all_results)
    _print_per_ticker_table(all_results)
    _print_summary(all_results)
    _print_confusion_matrices(all_results)


if __name__ == "__main__":
    main()

Vanilla LSTM Classification — 3-Class Direction Prediction
SEQ_LEN=60 | HIDDEN=128 | LAYERS=2 | THRESHOLD=0.001
[DEVICE] CUDA — NVIDIA RTX PRO 6000 Blackwell Server Edition

[Processing] AAPL
  AAPL: features=40, train=8238, val=1176, test=2355
    train dist: Down=32.2%  Flat=32.2%  Up=35.6%
  DataLoader samples: train=8178, val=1116, test=2295
    class_weights: [1.033 1.034 0.933]
    Epoch   1 | t_loss=1.0526 t_acc=0.4210 | v_loss=1.0501 v_acc=0.4077 | lr=1.0e-03 | 0.3s | pat=0/10
    Epoch   2 | t_loss=1.0263 t_acc=0.4340 | v_loss=1.0303 v_acc=0.4283 | lr=1.0e-03 | 0.2s | pat=0/10
    Epoch   3 | t_loss=1.0127 t_acc=0.4527 | v_loss=1.0236 v_acc=0.4373 | lr=1.0e-03 | 0.2s | pat=0/10
    Epoch   4 | t_loss=1.0037 t_acc=0.4674 | v_loss=1.0250 v_acc=0.4319 | lr=1.0e-03 | 0.2s | pat=1/10
    Epoch   5 | t_loss=0.9982 t_acc=0.4754 | v_loss=1.0238 v_acc=0.4391 | lr=1.0e-03 | 0.2s | pat=2/10
    Epoch  10 | t_loss=0.9649 t_acc=0.5088 | v_loss=1.0584 v_acc=0.4274 | lr=5.0e-04 | 0.2s | pat=

In [ ]:
"""
attention_lstm_regression.py
============================
Attention-LSTM 回归：在Vanilla LSTM基础上加 TemporalAttention
对全部时间步输出加权聚合，替代仅取最后时间步
Colab A100 / RTX PRO 6000 可运行
"""

import os
import time
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================
CONFIG: Dict = {
    "SPLITS_DIR": "splits",
    "FEATURES_DIR": "features",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "FREQ": "1hour",
    "SEQ_LEN": 60,
    "HIDDEN": 128,
    "NUM_LAYERS": 2,
    "DROPOUT": 0.2,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-5,
    "BATCH": 64,
    "EPOCHS": 100,
    "PATIENCE": 10,
    "OUTPUT_DIR": "results",
    "CKPT_DIR": "checkpoints",
    "TRAIN_RATIO": 0.70,
    "VAL_RATIO": 0.15,
    "DROP_COLS": ["timestamp", "open", "high", "low", "close", "volume",
                  "date", "datetime", "time", "ts_event"],
}


# ============================================================
# 设备检测
# ============================================================
def get_device() -> torch.device:
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        print(f"[DEVICE] CUDA — {torch.cuda.get_device_name(0)}")
        return dev
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("[DEVICE] Apple MPS")
        return torch.device("mps")
    print("[DEVICE] CPU")
    return torch.device("cpu")


# ============================================================
# 数据加载
# ============================================================
def _find_feature_cols(df: pd.DataFrame) -> List[str]:
    drop_set = {c.lower() for c in CONFIG["DROP_COLS"]}
    return [c for c in df.columns if c.lower() not in drop_set]


def _load_splits_csv(ticker: str) -> Optional[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    base = os.path.join(CONFIG["SPLITS_DIR"], f"{ticker}_{CONFIG['FREQ']}")
    paths = {k: os.path.join(base, f"{k}.csv") for k in ("train", "val", "test")}
    if not all(os.path.isfile(p) for p in paths.values()):
        return None
    try:
        return (pd.read_csv(paths["train"]),
                pd.read_csv(paths["val"]),
                pd.read_csv(paths["test"]))
    except Exception as e:
        print(f"[WARN] 读取splits失败 {ticker}: {e}")
        return None


def _splits_have_features(df: pd.DataFrame) -> bool:
    return len(_find_feature_cols(df)) >= 5


def _load_features_csv(ticker: str) -> Optional[pd.DataFrame]:
    path = os.path.join(CONFIG["FEATURES_DIR"], f"{ticker}_{CONFIG['FREQ']}_features.csv")
    if not os.path.isfile(path):
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"[WARN] 读取features失败 {ticker}: {e}")
        return None


def _split_by_ratio(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    n = len(df)
    n_train = int(n * CONFIG["TRAIN_RATIO"])
    n_val = int(n * CONFIG["VAL_RATIO"])
    return (df.iloc[:n_train].copy(),
            df.iloc[n_train:n_train + n_val].copy(),
            df.iloc[n_train + n_val:].copy())


def load_ticker_data(ticker: str) -> Optional[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    splits_result = _load_splits_csv(ticker)
    if splits_result is not None:
        train, val, test = splits_result
        if _splits_have_features(train):
            return train, val, test
        feat_df = _load_features_csv(ticker)
        if feat_df is not None:
            return _split_by_ratio(feat_df)
        return train, val, test
    feat_df = _load_features_csv(ticker)
    if feat_df is not None:
        return _split_by_ratio(feat_df)
    print(f"[ERROR] {ticker}: 无可用数据文件")
    return None


# ============================================================
# 特征 / Scaler
# ============================================================
def _ensure_close_column(df: pd.DataFrame) -> str:
    for c in df.columns:
        if c.lower() == "close":
            return c
    raise KeyError("找不到 close 列")


def prepare_arrays(
    train: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame
) -> Dict[str, np.ndarray]:
    close_col = _ensure_close_column(train)
    feat_cols = _find_feature_cols(train)
    if len(feat_cols) == 0:
        raise ValueError("无可用特征列")

    scaler = StandardScaler()
    result: Dict = {"feature_names": feat_cols, "close_col": close_col}

    for split_name, df in [("train", train), ("val", val), ("test", test)]:
        df = df.copy().reset_index(drop=True)
        available = [c for c in feat_cols if c in df.columns]
        X = df[available].copy()
        X = X.fillna(X.median()).fillna(0.0)

        if split_name == "train":
            X_scaled = scaler.fit_transform(X.values)
        else:
            X_scaled = scaler.transform(X.values)

        close_arr = df[close_col].values.astype(np.float64)
        target = np.empty(len(close_arr), dtype=np.float64)
        target[:-1] = close_arr[1:]
        target[-1] = np.nan

        result[f"X_{split_name}"] = X_scaled.astype(np.float32)
        result[f"close_{split_name}"] = close_arr
        result[f"target_{split_name}"] = target

    result["scaler"] = scaler
    result["n_features"] = result["X_train"].shape[1]
    return result


# ============================================================
# Dataset & DataLoader
# ============================================================
class StockDataset(Dataset):
    """滑动窗口：输入(SEQ_LEN, n_features), 目标=窗口后一根close"""

    def __init__(self, X: np.ndarray, close: np.ndarray,
                 target: np.ndarray, seq_len: int) -> None:
        self.X = X
        self.close = close
        self.target = target
        self.seq_len = seq_len
        self.valid_indices = self._build_valid_indices()

    def _build_valid_indices(self) -> np.ndarray:
        max_start = len(self.X) - self.seq_len
        indices = []
        for i in range(max_start):
            end_idx = i + self.seq_len - 1
            if not np.isnan(self.target[end_idx]):
                indices.append(i)
        return np.array(indices, dtype=np.int64)

    def __len__(self) -> int:
        return len(self.valid_indices)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        start = self.valid_indices[idx]
        end = start + self.seq_len
        x_seq = torch.from_numpy(self.X[start:end])
        y = torch.tensor(self.target[end - 1], dtype=torch.float32)
        cur = torch.tensor(self.close[end - 1], dtype=torch.float32)
        return x_seq, y, cur


def build_dataloaders(arrays: Dict[str, np.ndarray]) -> Dict[str, DataLoader]:
    seq_len = CONFIG["SEQ_LEN"]
    batch = CONFIG["BATCH"]
    loaders: Dict[str, DataLoader] = {}
    for split in ("train", "val", "test"):
        ds = StockDataset(
            X=arrays[f"X_{split}"],
            close=arrays[f"close_{split}"],
            target=arrays[f"target_{split}"],
            seq_len=seq_len,
        )
        loaders[split] = DataLoader(
            ds, batch_size=batch, shuffle=False,
            num_workers=0, pin_memory=True, drop_last=False,
        )
    return loaders


# ============================================================
# 模型：TemporalAttention + LSTM
# ============================================================
class TemporalAttention(nn.Module):
    """Bahdanau-style additive attention over LSTM time steps"""

    def __init__(self, hidden_size: int) -> None:
        super().__init__()
        self.W = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, lstm_out: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # lstm_out: (B, T, H)
        scores = self.v(torch.tanh(self.W(lstm_out)))  # (B, T, 1)
        weights = F.softmax(scores, dim=1)              # (B, T, 1)
        context = (weights * lstm_out).sum(dim=1)       # (B, H)
        return context, weights.squeeze(-1)             # (B, H), (B, T)


class AttentionLSTM(nn.Module):
    """LSTM + TemporalAttention → fc → 单值回归输出"""

    def __init__(self, input_size: int) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=CONFIG["HIDDEN"],
            num_layers=CONFIG["NUM_LAYERS"],
            dropout=CONFIG["DROPOUT"] if CONFIG["NUM_LAYERS"] > 1 else 0.0,
            batch_first=True,
        )
        self.attention = TemporalAttention(CONFIG["HIDDEN"])
        self.fc = nn.Linear(CONFIG["HIDDEN"], 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)
        lstm_out, _ = self.lstm(x)          # (B, T, H)
        context, _ = self.attention(lstm_out)  # (B, H)
        return self.fc(context).squeeze(-1)    # (B,)


# ============================================================
# 训练循环
# ============================================================
def train_one_epoch(
    model: nn.Module, loader: DataLoader,
    criterion: nn.Module, optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float:
    model.train()
    total_loss = 0.0
    n_batches = 0
    for x_seq, y, _ in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad()
        pred = model(x_seq)
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def eval_one_epoch(
    model: nn.Module, loader: DataLoader,
    criterion: nn.Module, device: torch.device,
) -> float:
    model.eval()
    total_loss = 0.0
    n_batches = 0
    for x_seq, y, _ in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        pred = model(x_seq)
        loss = criterion(pred, y)
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)


def train_model(
    model: nn.Module, loaders: Dict[str, DataLoader],
    device: torch.device, ticker: str,
) -> nn.Module:
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5
    )

    ckpt_dir = CONFIG["CKPT_DIR"]
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, f"attention_lstm_reg_{ticker}.pt")

    best_val_loss = float("inf")
    patience_counter = 0
    best_epoch = 0

    for epoch in range(1, CONFIG["EPOCHS"] + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, loaders["train"], criterion, optimizer, device)
        val_loss = eval_one_epoch(model, loaders["val"], criterion, device)
        scheduler.step(val_loss)
        elapsed = time.time() - t0

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_epoch = epoch
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1

        if epoch <= 5 or epoch % 10 == 0 or patience_counter >= CONFIG["PATIENCE"]:
            lr_now = optimizer.param_groups[0]["lr"]
            print(f"    Epoch {epoch:3d} | "
                  f"train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | "
                  f"lr={lr_now:.1e} | {elapsed:.1f}s | "
                  f"patience={patience_counter}/{CONFIG['PATIENCE']}")

        if patience_counter >= CONFIG["PATIENCE"]:
            print(f"    Early stopping at epoch {epoch}, best={best_epoch}")
            break

    if os.path.isfile(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    return model


# ============================================================
# 推理 & 评估
# ============================================================
@torch.no_grad()
def predict_all(
    model: nn.Module, loader: DataLoader, device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    preds, trues, closes = [], [], []
    for x_seq, y, cur in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        pred = model(x_seq)
        preds.append(pred.cpu().numpy())
        trues.append(y.numpy())
        closes.append(cur.numpy())
    return np.concatenate(preds), np.concatenate(trues), np.concatenate(closes)


def _directional_accuracy(y_pred: np.ndarray, y_true: np.ndarray, y_current: np.ndarray) -> float:
    pred_dir = np.sign(y_pred - y_current)
    true_dir = np.sign(y_true - y_current)
    mask = true_dir != 0
    if mask.sum() == 0:
        return 0.5
    return float(np.mean(pred_dir[mask] == true_dir[mask]))


def _prediction_lag(y_pred: np.ndarray, y_current: np.ndarray) -> float:
    if len(y_pred) < 3:
        return np.nan
    return float(np.corrcoef(y_pred, y_current)[0, 1])


def _da_z_score_and_p(da: float, n: int) -> Tuple[float, float]:
    if n == 0:
        return 0.0, 1.0
    se = np.sqrt(0.25 / n)
    z = (da - 0.5) / se
    p = 1.0 - stats.norm.cdf(z)
    return float(z), float(p)


def eval_regression(
    y_pred: np.ndarray, y_true: np.ndarray, y_current: np.ndarray
) -> Dict[str, float]:
    da = _directional_accuracy(y_pred, y_true, y_current)
    mse = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(mse))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    z, p = _da_z_score_and_p(da, len(y_true))
    lag = _prediction_lag(y_pred, y_current)
    return {
        "da": da, "mse": mse, "rmse": rmse, "mae": mae,
        "r2": r2, "z_score": z, "p_value": p, "pred_lag": lag,
    }


# ============================================================
# 保存预测
# ============================================================
def _save_predictions(
    ticker: str, current: np.ndarray,
    predicted: np.ndarray, actual: np.ndarray, path: str,
) -> None:
    df = pd.DataFrame({
        "ticker": ticker,
        "current": current,
        "predicted": predicted,
        "actual": actual,
    })
    header = not os.path.isfile(path)
    try:
        df.to_csv(path, mode="a", header=header, index=False)
    except Exception as e:
        print(f"[WARN] 保存预测失败 {path}: {e}")


# ============================================================
# 单只股票全流程
# ============================================================
def run_single_ticker(ticker: str, device: torch.device) -> Optional[Dict]:
    data = load_ticker_data(ticker)
    if data is None:
        return None

    train, val, test = data
    try:
        arrays = prepare_arrays(train, val, test)
    except (KeyError, ValueError) as e:
        print(f"[ERROR] {ticker} 特征构建失败: {e}")
        return None

    n_features = arrays["n_features"]
    print(f"  {ticker}: features={n_features}, "
          f"train={len(arrays['X_train'])}, "
          f"val={len(arrays['X_val'])}, "
          f"test={len(arrays['X_test'])}")

    loaders = build_dataloaders(arrays)
    print(f"  DataLoader samples: "
          f"train={len(loaders['train'].dataset)}, "
          f"val={len(loaders['val'].dataset)}, "
          f"test={len(loaders['test'].dataset)}")

    model = AttentionLSTM(input_size=n_features).to(device)
    param_count = sum(p.numel() for p in model.parameters())
    print(f"  Model params: {param_count:,}")

    model = train_model(model, loaders, device, ticker)

    y_pred, y_true, y_current = predict_all(model, loaders["test"], device)
    metrics = eval_regression(y_pred, y_true, y_current)
    metrics["ticker"] = ticker
    metrics["n_test"] = len(y_true)
    metrics["model"] = "AttentionLSTM"

    pred_path = os.path.join(CONFIG["OUTPUT_DIR"], "attention_lstm_reg_predictions.csv")
    _save_predictions(ticker, y_current, y_pred, y_true, pred_path)

    print(f"  → DA={metrics['da']:.4f} | RMSE={metrics['rmse']:.4f} | "
          f"R²={metrics['r2']:.4f} | pred_lag={metrics['pred_lag']:.4f} | "
          f"z={metrics['z_score']:.2f}")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics


# ============================================================
# 汇总表
# ============================================================
def _print_per_ticker_table(all_results: List[Dict]) -> None:
    df = pd.DataFrame(all_results)
    cols = ["model", "ticker", "n_test", "da", "mse", "rmse",
            "mae", "r2", "z_score", "p_value", "pred_lag"]
    df = df[[c for c in cols if c in df.columns]]
    fmt_map = {
        "da": "{:.4f}", "mse": "{:.4f}", "rmse": "{:.4f}",
        "mae": "{:.4f}", "r2": "{:.4f}", "z_score": "{:.2f}",
        "p_value": "{:.6f}", "pred_lag": "{:.4f}",
    }
    for col, f in fmt_map.items():
        if col in df.columns:
            df[col] = df[col].apply(lambda x: f.format(x) if pd.notna(x) else "N/A")
    print("\n## Attention LSTM Regression — Per-Ticker Results\n")
    print(df.to_markdown(index=False))


def _print_summary(all_results: List[Dict]) -> None:
    df = pd.DataFrame(all_results)
    summary = {
        "mean_da": f"{df['da'].mean():.4f}",
        "std_da": f"{df['da'].std():.4f}",
        "mean_rmse": f"{df['rmse'].mean():.4f}",
        "mean_r2": f"{df['r2'].mean():.4f}",
        "mean_pred_lag": f"{df['pred_lag'].mean():.4f}",
        "n_tickers": len(df),
    }
    print("\n## Attention LSTM Regression — Summary\n")
    print(pd.DataFrame([summary]).to_markdown(index=False))


def _print_comparison_note(all_results: List[Dict]) -> None:
    """与Vanilla LSTM对比提示"""
    df = pd.DataFrame(all_results)
    print("\n## Note: Compare with Vanilla LSTM\n")
    print(f"Attention LSTM mean DA = {df['da'].mean():.4f}, "
          f"mean pred_lag = {df['pred_lag'].mean():.4f}")
    print("If pred_lag ≈ 1.0 and DA ≈ 0.50, attention did not help — "
          "model still degenerates to naive prediction.")
    print("This supports the 50% ceiling hypothesis for OHLCV-based direction prediction.")


# ============================================================
# main
# ============================================================
def main() -> None:
    print("=" * 60)
    print("Attention LSTM Regression — Next Close Prediction")
    print(f"SEQ_LEN={CONFIG['SEQ_LEN']} | HIDDEN={CONFIG['HIDDEN']} | "
          f"LAYERS={CONFIG['NUM_LAYERS']} | EPOCHS≤{CONFIG['EPOCHS']}")
    print("=" * 60)

    os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
    os.makedirs(CONFIG["CKPT_DIR"], exist_ok=True)

    pred_path = os.path.join(CONFIG["OUTPUT_DIR"], "attention_lstm_reg_predictions.csv")
    if os.path.isfile(pred_path):
        os.remove(pred_path)

    device = get_device()
    all_results: List[Dict] = []

    for ticker in CONFIG["TICKERS"]:
        print(f"\n{'='*40}")
        print(f"[Processing] {ticker}")
        print(f"{'='*40}")
        result = run_single_ticker(ticker, device)
        if result is not None:
            all_results.append(result)

    if len(all_results) == 0:
        print("[ERROR] 所有股票均无有效数据。")
        return

    summary_path = os.path.join(CONFIG["OUTPUT_DIR"], "attention_lstm_reg_summary.csv")
    try:
        pd.DataFrame(all_results).to_csv(summary_path, index=False)
        print(f"\n[SAVED] 汇总 → {summary_path}")
    except Exception as e:
        print(f"[WARN] 保存汇总失败: {e}")

    _print_per_ticker_table(all_results)
    _print_summary(all_results)
    _print_comparison_note(all_results)


if __name__ == "__main__":
    main()

Attention LSTM Regression — Next Close Prediction
SEQ_LEN=60 | HIDDEN=128 | LAYERS=2 | EPOCHS≤100
[DEVICE] CUDA — NVIDIA RTX PRO 6000 Blackwell Server Edition

[Processing] AAPL
  AAPL: features=40, train=8238, val=1176, test=2355
  DataLoader samples: train=8178, val=1116, test=2295
  Model params: 235,905
    Epoch   1 | train_loss=12475.694857 | val_loss=19392.083659 | lr=1.0e-03 | 0.3s | patience=0/10
    Epoch   2 | train_loss=9004.493567 | val_loss=15034.877821 | lr=1.0e-03 | 0.3s | patience=0/10
    Epoch   3 | train_loss=6259.726502 | val_loss=11282.762397 | lr=1.0e-03 | 0.3s | patience=0/10
    Epoch   4 | train_loss=4077.305756 | val_loss=8087.631592 | lr=1.0e-03 | 0.3s | patience=0/10
    Epoch   5 | train_loss=2738.976346 | val_loss=5954.651042 | lr=1.0e-03 | 0.3s | patience=0/10
    Epoch  10 | train_loss=938.788731 | val_loss=1956.371409 | lr=1.0e-03 | 0.3s | patience=0/10
    Epoch  20 | train_loss=534.094886 | val_loss=212.203594 | lr=1.0e-03 | 0.3s | patience=0/10
    

In [ ]:
"""
attention_lstm_classification.py
================================
Attention-LSTM 三分类：在Vanilla LSTM基础上加 TemporalAttention
标签：ret = (next_close - close) / close
  ret >  THRESHOLD → 2 (Up)
  ret < -THRESHOLD → 0 (Down)
  else             → 1 (Flat)
Colab A100 / RTX PRO 6000 可运行
"""

import os
import time
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================
CONFIG: Dict = {
    "SPLITS_DIR": "splits",
    "FEATURES_DIR": "features",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "FREQ": "1hour",
    "SEQ_LEN": 60,
    "HIDDEN": 128,
    "NUM_LAYERS": 2,
    "DROPOUT": 0.2,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-5,
    "BATCH": 64,
    "EPOCHS": 100,
    "PATIENCE": 10,
    "THRESHOLD": 0.001,
    "OUTPUT_DIR": "results",
    "CKPT_DIR": "checkpoints",
    "TRAIN_RATIO": 0.70,
    "VAL_RATIO": 0.15,
    "DROP_COLS": ["timestamp", "open", "high", "low", "close", "volume",
                  "date", "datetime", "time", "ts_event"],
    "NUM_CLASSES": 3,
    "CLASS_NAMES": {0: "Down", 1: "Flat", 2: "Up"},
}


# ============================================================
# 设备检测
# ============================================================
def get_device() -> torch.device:
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        print(f"[DEVICE] CUDA — {torch.cuda.get_device_name(0)}")
        return dev
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("[DEVICE] Apple MPS")
        return torch.device("mps")
    print("[DEVICE] CPU")
    return torch.device("cpu")


# ============================================================
# 数据加载
# ============================================================
def _find_feature_cols(df: pd.DataFrame) -> List[str]:
    drop_set = {c.lower() for c in CONFIG["DROP_COLS"]}
    return [c for c in df.columns if c.lower() not in drop_set]


def _load_splits_csv(ticker: str) -> Optional[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    base = os.path.join(CONFIG["SPLITS_DIR"], f"{ticker}_{CONFIG['FREQ']}")
    paths = {k: os.path.join(base, f"{k}.csv") for k in ("train", "val", "test")}
    if not all(os.path.isfile(p) for p in paths.values()):
        return None
    try:
        return (pd.read_csv(paths["train"]),
                pd.read_csv(paths["val"]),
                pd.read_csv(paths["test"]))
    except Exception as e:
        print(f"[WARN] 读取splits失败 {ticker}: {e}")
        return None


def _splits_have_features(df: pd.DataFrame) -> bool:
    return len(_find_feature_cols(df)) >= 5


def _load_features_csv(ticker: str) -> Optional[pd.DataFrame]:
    path = os.path.join(CONFIG["FEATURES_DIR"], f"{ticker}_{CONFIG['FREQ']}_features.csv")
    if not os.path.isfile(path):
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"[WARN] 读取features失败 {ticker}: {e}")
        return None


def _split_by_ratio(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    n = len(df)
    n_train = int(n * CONFIG["TRAIN_RATIO"])
    n_val = int(n * CONFIG["VAL_RATIO"])
    return (df.iloc[:n_train].copy(),
            df.iloc[n_train:n_train + n_val].copy(),
            df.iloc[n_train + n_val:].copy())


def load_ticker_data(ticker: str) -> Optional[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    splits_result = _load_splits_csv(ticker)
    if splits_result is not None:
        train, val, test = splits_result
        if _splits_have_features(train):
            return train, val, test
        feat_df = _load_features_csv(ticker)
        if feat_df is not None:
            return _split_by_ratio(feat_df)
        return train, val, test
    feat_df = _load_features_csv(ticker)
    if feat_df is not None:
        return _split_by_ratio(feat_df)
    print(f"[ERROR] {ticker}: 无可用数据文件")
    return None


# ============================================================
# 标签构建
# ============================================================
def _ensure_close_column(df: pd.DataFrame) -> str:
    for c in df.columns:
        if c.lower() == "close":
            return c
    raise KeyError("找不到 close 列")


def _make_label(close_now: np.ndarray, close_next: np.ndarray) -> np.ndarray:
    ret = (close_next - close_now) / close_now
    labels = np.ones(len(ret), dtype=np.int64)
    labels[ret > CONFIG["THRESHOLD"]] = 2
    labels[ret < -CONFIG["THRESHOLD"]] = 0
    return labels


# ============================================================
# 特征 / Scaler
# ============================================================
def prepare_arrays(
    train: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame
) -> Dict[str, np.ndarray]:
    close_col = _ensure_close_column(train)
    feat_cols = _find_feature_cols(train)
    if len(feat_cols) == 0:
        raise ValueError("无可用特征列")

    scaler = StandardScaler()
    result: Dict = {"feature_names": feat_cols, "close_col": close_col}

    for split_name, df in [("train", train), ("val", val), ("test", test)]:
        df = df.copy().reset_index(drop=True)
        available = [c for c in feat_cols if c in df.columns]
        X = df[available].copy()
        X = X.fillna(X.median()).fillna(0.0)

        if split_name == "train":
            X_scaled = scaler.fit_transform(X.values)
        else:
            X_scaled = scaler.transform(X.values)

        close_arr = df[close_col].values.astype(np.float64)
        next_close = np.empty(len(close_arr), dtype=np.float64)
        next_close[:-1] = close_arr[1:]
        next_close[-1] = np.nan

        labels = np.full(len(close_arr), -1, dtype=np.int64)
        valid_mask = ~np.isnan(next_close)
        labels[valid_mask] = _make_label(close_arr[valid_mask], next_close[valid_mask])

        result[f"X_{split_name}"] = X_scaled.astype(np.float32)
        result[f"close_{split_name}"] = close_arr
        result[f"label_{split_name}"] = labels

    result["scaler"] = scaler
    result["n_features"] = result["X_train"].shape[1]
    return result


def _compute_class_weights(labels: np.ndarray, device: torch.device) -> torch.Tensor:
    valid = labels[labels >= 0]
    counts = np.bincount(valid, minlength=CONFIG["NUM_CLASSES"]).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = 1.0 / counts
    weights = weights / weights.sum() * CONFIG["NUM_CLASSES"]
    return torch.tensor(weights, dtype=torch.float32).to(device)


# ============================================================
# Dataset & DataLoader
# ============================================================
class StockDataset(Dataset):
    def __init__(self, X: np.ndarray, close: np.ndarray,
                 labels: np.ndarray, seq_len: int) -> None:
        self.X = X
        self.close = close
        self.labels = labels
        self.seq_len = seq_len
        self.valid_indices = self._build_valid_indices()

    def _build_valid_indices(self) -> np.ndarray:
        max_start = len(self.X) - self.seq_len
        indices = []
        for i in range(max_start):
            end_idx = i + self.seq_len - 1
            if self.labels[end_idx] >= 0:
                indices.append(i)
        return np.array(indices, dtype=np.int64)

    def __len__(self) -> int:
        return len(self.valid_indices)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        start = self.valid_indices[idx]
        end = start + self.seq_len
        x_seq = torch.from_numpy(self.X[start:end])
        label = torch.tensor(self.labels[end - 1], dtype=torch.long)
        return x_seq, label


def build_dataloaders(arrays: Dict[str, np.ndarray]) -> Dict[str, DataLoader]:
    seq_len = CONFIG["SEQ_LEN"]
    batch = CONFIG["BATCH"]
    loaders: Dict[str, DataLoader] = {}
    for split in ("train", "val", "test"):
        ds = StockDataset(
            X=arrays[f"X_{split}"],
            close=arrays[f"close_{split}"],
            labels=arrays[f"label_{split}"],
            seq_len=seq_len,
        )
        loaders[split] = DataLoader(
            ds, batch_size=batch, shuffle=False,
            num_workers=0, pin_memory=True, drop_last=False,
        )
    return loaders


# ============================================================
# 模型：TemporalAttention + LSTM → 3分类
# ============================================================
class TemporalAttention(nn.Module):
    """Bahdanau-style additive attention over LSTM time steps"""

    def __init__(self, hidden_size: int) -> None:
        super().__init__()
        self.W = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, lstm_out: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        scores = self.v(torch.tanh(self.W(lstm_out)))   # (B, T, 1)
        weights = F.softmax(scores, dim=1)               # (B, T, 1)
        context = (weights * lstm_out).sum(dim=1)        # (B, H)
        return context, weights.squeeze(-1)              # (B, H), (B, T)


class AttentionLSTMClf(nn.Module):
    """LSTM + TemporalAttention → fc → 3类logits"""

    def __init__(self, input_size: int) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=CONFIG["HIDDEN"],
            num_layers=CONFIG["NUM_LAYERS"],
            dropout=CONFIG["DROPOUT"] if CONFIG["NUM_LAYERS"] > 1 else 0.0,
            batch_first=True,
        )
        self.attention = TemporalAttention(CONFIG["HIDDEN"])
        self.fc = nn.Linear(CONFIG["HIDDEN"], CONFIG["NUM_CLASSES"])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lstm_out, _ = self.lstm(x)             # (B, T, H)
        context, _ = self.attention(lstm_out)  # (B, H)
        return self.fc(context)                # (B, 3)


# ============================================================
# 训练循环
# ============================================================
def train_one_epoch(
    model: nn.Module, loader: DataLoader,
    criterion: nn.Module, optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> Tuple[float, float]:
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for x_seq, label in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(x_seq)
        loss = criterion(logits, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * label.size(0)
        correct += (logits.argmax(dim=1) == label).sum().item()
        total += label.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def eval_one_epoch(
    model: nn.Module, loader: DataLoader,
    criterion: nn.Module, device: torch.device,
) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    for x_seq, label in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)
        logits = model(x_seq)
        loss = criterion(logits, label)
        total_loss += loss.item() * label.size(0)
        correct += (logits.argmax(dim=1) == label).sum().item()
        total += label.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


def train_model(
    model: nn.Module, loaders: Dict[str, DataLoader],
    criterion: nn.Module, device: torch.device, ticker: str,
) -> nn.Module:
    optimizer = torch.optim.Adam(
        model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5
    )

    ckpt_dir = CONFIG["CKPT_DIR"]
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, f"attention_lstm_clf_{ticker}.pt")

    best_val_loss = float("inf")
    patience_counter = 0
    best_epoch = 0

    for epoch in range(1, CONFIG["EPOCHS"] + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(
            model, loaders["train"], criterion, optimizer, device)
        val_loss, val_acc = eval_one_epoch(
            model, loaders["val"], criterion, device)
        scheduler.step(val_loss)
        elapsed = time.time() - t0

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_epoch = epoch
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1

        if epoch <= 5 or epoch % 10 == 0 or patience_counter >= CONFIG["PATIENCE"]:
            lr_now = optimizer.param_groups[0]["lr"]
            print(f"    Epoch {epoch:3d} | "
                  f"t_loss={train_loss:.4f} t_acc={train_acc:.4f} | "
                  f"v_loss={val_loss:.4f} v_acc={val_acc:.4f} | "
                  f"lr={lr_now:.1e} | {elapsed:.1f}s | "
                  f"pat={patience_counter}/{CONFIG['PATIENCE']}")

        if patience_counter >= CONFIG["PATIENCE"]:
            print(f"    Early stopping at epoch {epoch}, best={best_epoch}")
            break

    if os.path.isfile(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    return model


# ============================================================
# 推理 & 评估
# ============================================================
@torch.no_grad()
def predict_all(
    model: nn.Module, loader: DataLoader, device: torch.device,
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    preds, trues = [], []
    for x_seq, label in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        logits = model(x_seq)
        preds.append(logits.argmax(dim=1).cpu().numpy())
        trues.append(label.numpy())
    return np.concatenate(preds), np.concatenate(trues)


def _class_distribution(y: np.ndarray) -> Dict[str, str]:
    total = len(y)
    if total == 0:
        return {}
    dist = {}
    for cls_id, cls_name in CONFIG["CLASS_NAMES"].items():
        cnt = int(np.sum(y == cls_id))
        dist[f"{cls_name}_count"] = cnt
        dist[f"{cls_name}_pct"] = cnt / total
    return dist


def _majority_baseline(y: np.ndarray) -> float:
    if len(y) == 0:
        return 0.0
    counts = np.bincount(y, minlength=CONFIG["NUM_CLASSES"])
    return float(counts.max() / len(y))


def _acc_z_score_and_p(acc: float, n: int, baseline: float) -> Tuple[float, float]:
    if n == 0 or baseline <= 0.0 or baseline >= 1.0:
        return 0.0, 1.0
    se = np.sqrt(baseline * (1 - baseline) / n)
    z = (acc - baseline) / se
    p = 1.0 - stats.norm.cdf(z)
    return float(z), float(p)


def eval_classification(y_pred: np.ndarray, y_true: np.ndarray) -> Dict:
    acc = float(accuracy_score(y_true, y_pred))
    prec, rec, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], zero_division=0.0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    dist = _class_distribution(y_true)
    maj = _majority_baseline(y_true)
    z, p = _acc_z_score_and_p(acc, len(y_true), maj)

    result: Dict = {
        "accuracy": acc, "majority_baseline": maj,
        "z_score": z, "p_value": p, "n_test": len(y_true),
    }
    for i, cls_name in CONFIG["CLASS_NAMES"].items():
        result[f"{cls_name}_precision"] = float(prec[i])
        result[f"{cls_name}_recall"] = float(rec[i])
        result[f"{cls_name}_f1"] = float(f1[i])
        result[f"{cls_name}_support"] = int(sup[i])
    result.update(dist)
    result["confusion_matrix"] = cm.tolist()
    return result


# ============================================================
# 单只股票全流程
# ============================================================
def run_single_ticker(ticker: str, device: torch.device) -> Optional[Dict]:
    data = load_ticker_data(ticker)
    if data is None:
        return None

    train, val, test = data
    try:
        arrays = prepare_arrays(train, val, test)
    except (KeyError, ValueError) as e:
        print(f"[ERROR] {ticker} 特征构建失败: {e}")
        return None

    n_features = arrays["n_features"]
    train_labels = arrays["label_train"]
    train_valid = train_labels[train_labels >= 0]
    train_dist = _class_distribution(train_valid)
    print(f"  {ticker}: features={n_features}, "
          f"train={len(arrays['X_train'])}, "
          f"val={len(arrays['X_val'])}, "
          f"test={len(arrays['X_test'])}")
    print(f"    train dist: "
          f"Down={train_dist.get('Down_pct', 0):.1%}  "
          f"Flat={train_dist.get('Flat_pct', 0):.1%}  "
          f"Up={train_dist.get('Up_pct', 0):.1%}")

    loaders = build_dataloaders(arrays)
    print(f"  DataLoader samples: "
          f"train={len(loaders['train'].dataset)}, "
          f"val={len(loaders['val'].dataset)}, "
          f"test={len(loaders['test'].dataset)}")

    class_weights = _compute_class_weights(train_valid, device)
    print(f"    class_weights: {class_weights.cpu().numpy().round(3)}")
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    model = AttentionLSTMClf(input_size=n_features).to(device)
    param_count = sum(p.numel() for p in model.parameters())
    print(f"  Model params: {param_count:,}")

    model = train_model(model, loaders, criterion, device, ticker)

    y_pred, y_true = predict_all(model, loaders["test"], device)
    metrics = eval_classification(y_pred, y_true)
    metrics["ticker"] = ticker
    metrics["model"] = "AttentionLSTM_Clf"

    _save_predictions(ticker, y_true, y_pred)

    print(f"  → acc={metrics['accuracy']:.4f} | "
          f"majority_bl={metrics['majority_baseline']:.4f} | "
          f"z={metrics['z_score']:.2f} | "
          f"Down_f1={metrics['Down_f1']:.3f} | "
          f"Flat_f1={metrics['Flat_f1']:.3f} | "
          f"Up_f1={metrics['Up_f1']:.3f}")

    del model, criterion
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics


# ============================================================
# 保存预测
# ============================================================
def _save_predictions(ticker: str, y_true: np.ndarray, y_pred: np.ndarray) -> None:
    out_dir = CONFIG["OUTPUT_DIR"]
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "attention_lstm_clf_predictions.csv")
    df = pd.DataFrame({"ticker": ticker, "y_true": y_true, "y_pred": y_pred})
    header = not os.path.isfile(path)
    try:
        df.to_csv(path, mode="a", header=header, index=False)
    except Exception as e:
        print(f"[WARN] 保存预测失败 {path}: {e}")


# ============================================================
# 汇总表
# ============================================================
def _print_per_ticker_table(all_results: List[Dict]) -> None:
    rows = []
    for r in all_results:
        rows.append({
            "model": r["model"], "ticker": r["ticker"], "n_test": r["n_test"],
            "accuracy": f"{r['accuracy']:.4f}",
            "majority_bl": f"{r['majority_baseline']:.4f}",
            "z_score": f"{r['z_score']:.2f}",
            "p_value": f"{r['p_value']:.6f}",
            "Down_f1": f"{r['Down_f1']:.3f}",
            "Flat_f1": f"{r['Flat_f1']:.3f}",
            "Up_f1": f"{r['Up_f1']:.3f}",
        })
    print("\n## Attention LSTM Classification — Per-Ticker Results\n")
    print(pd.DataFrame(rows).to_markdown(index=False))


def _print_class_dist_table(all_results: List[Dict]) -> None:
    rows = []
    for r in all_results:
        rows.append({
            "ticker": r["ticker"],
            "Down": f"{r.get('Down_count', 0)} ({r.get('Down_pct', 0):.1%})",
            "Flat": f"{r.get('Flat_count', 0)} ({r.get('Flat_pct', 0):.1%})",
            "Up": f"{r.get('Up_count', 0)} ({r.get('Up_pct', 0):.1%})",
            "majority_bl": f"{r['majority_baseline']:.4f}",
        })
    print(f"\n## Test Set Class Distribution (threshold={CONFIG['THRESHOLD']})\n")
    print(pd.DataFrame(rows).to_markdown(index=False))


def _print_confusion_matrices(all_results: List[Dict]) -> None:
    print("\n## Confusion Matrices\n")
    for r in all_results:
        cm = np.array(r["confusion_matrix"])
        print(f"### {r['ticker']}")
        cm_df = pd.DataFrame(
            cm,
            index=["true_Down", "true_Flat", "true_Up"],
            columns=["pred_Down", "pred_Flat", "pred_Up"],
        )
        print(cm_df.to_markdown())
        print()


def _print_summary(all_results: List[Dict]) -> None:
    df = pd.DataFrame(all_results)
    summary = {
        "mean_acc": f"{df['accuracy'].mean():.4f}",
        "std_acc": f"{df['accuracy'].std():.4f}",
        "mean_majority_bl": f"{df['majority_baseline'].mean():.4f}",
        "mean_Down_f1": f"{df['Down_f1'].mean():.3f}",
        "mean_Flat_f1": f"{df['Flat_f1'].mean():.3f}",
        "mean_Up_f1": f"{df['Up_f1'].mean():.3f}",
        "n_tickers": len(df),
    }
    print("\n## Attention LSTM Classification — Summary\n")
    print(pd.DataFrame([summary]).to_markdown(index=False))


# ============================================================
# main
# ============================================================
def main() -> None:
    print("=" * 60)
    print("Attention LSTM Classification — 3-Class Direction")
    print(f"SEQ_LEN={CONFIG['SEQ_LEN']} | HIDDEN={CONFIG['HIDDEN']} | "
          f"LAYERS={CONFIG['NUM_LAYERS']} | THRESHOLD={CONFIG['THRESHOLD']}")
    print("=" * 60)

    os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
    os.makedirs(CONFIG["CKPT_DIR"], exist_ok=True)

    pred_path = os.path.join(CONFIG["OUTPUT_DIR"], "attention_lstm_clf_predictions.csv")
    if os.path.isfile(pred_path):
        os.remove(pred_path)

    device = get_device()
    all_results: List[Dict] = []

    for ticker in CONFIG["TICKERS"]:
        print(f"\n{'='*40}")
        print(f"[Processing] {ticker}")
        print(f"{'='*40}")
        result = run_single_ticker(ticker, device)
        if result is not None:
            all_results.append(result)

    if len(all_results) == 0:
        print("[ERROR] 所有股票均无有效数据。")
        return

    summary_path = os.path.join(CONFIG["OUTPUT_DIR"], "attention_lstm_clf_summary.csv")
    try:
        df_save = pd.DataFrame(all_results).copy()
        df_save["confusion_matrix"] = df_save["confusion_matrix"].apply(str)
        df_save.to_csv(summary_path, index=False)
        print(f"\n[SAVED] 汇总 → {summary_path}")
    except Exception as e:
        print(f"[WARN] 保存汇总失败: {e}")

    _print_class_dist_table(all_results)
    _print_per_ticker_table(all_results)
    _print_summary(all_results)
    _print_confusion_matrices(all_results)


if __name__ == "__main__":
    main()


Attention LSTM Classification — 3-Class Direction
SEQ_LEN=60 | HIDDEN=128 | LAYERS=2 | THRESHOLD=0.001
[DEVICE] CUDA — NVIDIA RTX PRO 6000 Blackwell Server Edition

[Processing] AAPL
  AAPL: features=40, train=8238, val=1176, test=2355
    train dist: Down=32.2%  Flat=32.2%  Up=35.6%
  DataLoader samples: train=8178, val=1116, test=2295
    class_weights: [1.033 1.034 0.933]
  Model params: 236,163
    Epoch   1 | t_loss=1.0844 t_acc=0.3895 | v_loss=1.0846 v_acc=0.3763 | lr=1.0e-03 | 0.3s | pat=0/10
    Epoch   2 | t_loss=1.0774 t_acc=0.3945 | v_loss=1.0808 v_acc=0.3772 | lr=1.0e-03 | 0.3s | pat=0/10
    Epoch   3 | t_loss=1.0741 t_acc=0.3951 | v_loss=1.0801 v_acc=0.3781 | lr=1.0e-03 | 0.3s | pat=0/10
    Epoch   4 | t_loss=1.0726 t_acc=0.3958 | v_loss=1.0795 v_acc=0.3799 | lr=1.0e-03 | 0.3s | pat=0/10
    Epoch   5 | t_loss=1.0710 t_acc=0.3955 | v_loss=1.0786 v_acc=0.3790 | lr=1.0e-03 | 0.3s | pat=0/10
    Epoch  10 | t_loss=1.0536 t_acc=0.4079 | v_loss=1.0760 v_acc=0.4077 | lr=1.0e-0

In [ ]:
"""
transformer_lstm_regression.py
==============================
Transformer Encoder + LSTM 混合架构回归：预测下一根K线close
Features → Linear proj → PositionalEncoding → TransformerEncoder → LSTM → MLP head
Colab A100 / RTX PRO 6000 可运行
"""

import os
import math
import time
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================
CONFIG: Dict = {
    "SPLITS_DIR": "splits",
    "FEATURES_DIR": "features",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "FREQ": "1hour",
    "SEQ_LEN": 60,
    # Transformer 超参
    "D_MODEL": 64,
    "NHEAD": 4,
    "TF_LAYERS": 2,
    "FF_DIM": 256,
    "TF_DROPOUT": 0.1,
    # LSTM 超参
    "HIDDEN": 128,
    "LSTM_LAYERS": 1,
    "DROPOUT": 0.2,
    # 训练超参
    "LR": 5e-4,
    "WEIGHT_DECAY": 1e-5,
    "BATCH": 64,
    "EPOCHS": 100,
    "PATIENCE": 10,
    "OUTPUT_DIR": "results",
    "CKPT_DIR": "checkpoints",
    "TRAIN_RATIO": 0.70,
    "VAL_RATIO": 0.15,
    "DROP_COLS": ["timestamp", "open", "high", "low", "close", "volume",
                  "date", "datetime", "time", "ts_event"],
}


# ============================================================
# 设备检测
# ============================================================
def get_device() -> torch.device:
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        print(f"[DEVICE] CUDA — {torch.cuda.get_device_name(0)}")
        return dev
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("[DEVICE] Apple MPS")
        return torch.device("mps")
    print("[DEVICE] CPU")
    return torch.device("cpu")


# ============================================================
# 数据加载
# ============================================================
def _find_feature_cols(df: pd.DataFrame) -> List[str]:
    drop_set = {c.lower() for c in CONFIG["DROP_COLS"]}
    return [c for c in df.columns if c.lower() not in drop_set]


def _load_splits_csv(ticker: str) -> Optional[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    base = os.path.join(CONFIG["SPLITS_DIR"], f"{ticker}_{CONFIG['FREQ']}")
    paths = {k: os.path.join(base, f"{k}.csv") for k in ("train", "val", "test")}
    if not all(os.path.isfile(p) for p in paths.values()):
        return None
    try:
        return (pd.read_csv(paths["train"]),
                pd.read_csv(paths["val"]),
                pd.read_csv(paths["test"]))
    except Exception as e:
        print(f"[WARN] 读取splits失败 {ticker}: {e}")
        return None


def _splits_have_features(df: pd.DataFrame) -> bool:
    return len(_find_feature_cols(df)) >= 5


def _load_features_csv(ticker: str) -> Optional[pd.DataFrame]:
    path = os.path.join(CONFIG["FEATURES_DIR"], f"{ticker}_{CONFIG['FREQ']}_features.csv")
    if not os.path.isfile(path):
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"[WARN] 读取features失败 {ticker}: {e}")
        return None


def _split_by_ratio(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    n = len(df)
    n_train = int(n * CONFIG["TRAIN_RATIO"])
    n_val = int(n * CONFIG["VAL_RATIO"])
    return (df.iloc[:n_train].copy(),
            df.iloc[n_train:n_train + n_val].copy(),
            df.iloc[n_train + n_val:].copy())


def load_ticker_data(ticker: str) -> Optional[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    splits_result = _load_splits_csv(ticker)
    if splits_result is not None:
        train, val, test = splits_result
        if _splits_have_features(train):
            return train, val, test
        feat_df = _load_features_csv(ticker)
        if feat_df is not None:
            return _split_by_ratio(feat_df)
        return train, val, test
    feat_df = _load_features_csv(ticker)
    if feat_df is not None:
        return _split_by_ratio(feat_df)
    print(f"[ERROR] {ticker}: 无可用数据文件")
    return None


# ============================================================
# 特征 / Scaler
# ============================================================
def _ensure_close_column(df: pd.DataFrame) -> str:
    for c in df.columns:
        if c.lower() == "close":
            return c
    raise KeyError("找不到 close 列")


def prepare_arrays(
    train: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame
) -> Dict[str, np.ndarray]:
    close_col = _ensure_close_column(train)
    feat_cols = _find_feature_cols(train)
    if len(feat_cols) == 0:
        raise ValueError("无可用特征列")

    scaler = StandardScaler()
    result: Dict = {"feature_names": feat_cols, "close_col": close_col}

    for split_name, df in [("train", train), ("val", val), ("test", test)]:
        df = df.copy().reset_index(drop=True)
        available = [c for c in feat_cols if c in df.columns]
        X = df[available].copy()
        X = X.fillna(X.median()).fillna(0.0)

        if split_name == "train":
            X_scaled = scaler.fit_transform(X.values)
        else:
            X_scaled = scaler.transform(X.values)

        close_arr = df[close_col].values.astype(np.float64)
        target = np.empty(len(close_arr), dtype=np.float64)
        target[:-1] = close_arr[1:]
        target[-1] = np.nan

        result[f"X_{split_name}"] = X_scaled.astype(np.float32)
        result[f"close_{split_name}"] = close_arr
        result[f"target_{split_name}"] = target

    result["scaler"] = scaler
    result["n_features"] = result["X_train"].shape[1]
    return result


# ============================================================
# Dataset & DataLoader
# ============================================================
class StockDataset(Dataset):
    def __init__(self, X: np.ndarray, close: np.ndarray,
                 target: np.ndarray, seq_len: int) -> None:
        self.X = X
        self.close = close
        self.target = target
        self.seq_len = seq_len
        self.valid_indices = self._build_valid_indices()

    def _build_valid_indices(self) -> np.ndarray:
        max_start = len(self.X) - self.seq_len
        indices = []
        for i in range(max_start):
            end_idx = i + self.seq_len - 1
            if not np.isnan(self.target[end_idx]):
                indices.append(i)
        return np.array(indices, dtype=np.int64)

    def __len__(self) -> int:
        return len(self.valid_indices)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        start = self.valid_indices[idx]
        end = start + self.seq_len
        x_seq = torch.from_numpy(self.X[start:end])
        y = torch.tensor(self.target[end - 1], dtype=torch.float32)
        cur = torch.tensor(self.close[end - 1], dtype=torch.float32)
        return x_seq, y, cur


def build_dataloaders(arrays: Dict[str, np.ndarray]) -> Dict[str, DataLoader]:
    seq_len = CONFIG["SEQ_LEN"]
    batch = CONFIG["BATCH"]
    loaders: Dict[str, DataLoader] = {}
    for split in ("train", "val", "test"):
        ds = StockDataset(
            X=arrays[f"X_{split}"],
            close=arrays[f"close_{split}"],
            target=arrays[f"target_{split}"],
            seq_len=seq_len,
        )
        loaders[split] = DataLoader(
            ds, batch_size=batch, shuffle=False,
            num_workers=0, pin_memory=True, drop_last=False,
        )
    return loaders


# ============================================================
# 模型：Transformer Encoder + LSTM
# ============================================================
class PositionalEncoding(nn.Module):
    """标准 sin/cos 位置编码"""

    def __init__(self, d_model: int, max_len: int = 500) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, d_model)
        return x + self.pe[:, :x.size(1), :]


class TransformerLSTM(nn.Module):
    """
    Features → Linear proj(N_FEATURES → D_MODEL) → PositionalEncoding
    → TransformerEncoder → LSTM → MLP head → 单值回归
    """

    def __init__(self, input_size: int) -> None:
        super().__init__()
        d_model = CONFIG["D_MODEL"]

        # 特征投影：N_FEATURES → D_MODEL
        self.proj = nn.Linear(input_size, d_model)

        # 位置编码
        self.pe = PositionalEncoding(d_model, max_len=CONFIG["SEQ_LEN"] + 50)

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=CONFIG["NHEAD"],
            dim_feedforward=CONFIG["FF_DIM"],
            dropout=CONFIG["TF_DROPOUT"],
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=CONFIG["TF_LAYERS"],
        )

        # LSTM：接收Transformer输出
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=CONFIG["HIDDEN"],
            num_layers=CONFIG["LSTM_LAYERS"],
            batch_first=True,
        )

        # MLP head
        self.head = nn.Sequential(
            nn.Linear(CONFIG["HIDDEN"], 64),
            nn.ReLU(),
            nn.Dropout(CONFIG["DROPOUT"]),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, N_FEATURES)
        x = self.proj(x)            # (B, T, D_MODEL)
        x = self.pe(x)              # (B, T, D_MODEL)
        x = self.encoder(x)         # (B, T, D_MODEL)
        lstm_out, _ = self.lstm(x)   # (B, T, HIDDEN)
        last = lstm_out[:, -1, :]    # (B, HIDDEN)
        return self.head(last).squeeze(-1)  # (B,)


# ============================================================
# 训练循环
# ============================================================
def train_one_epoch(
    model: nn.Module, loader: DataLoader,
    criterion: nn.Module, optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float:
    model.train()
    total_loss = 0.0
    n_batches = 0
    for x_seq, y, _ in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad()
        pred = model(x_seq)
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def eval_one_epoch(
    model: nn.Module, loader: DataLoader,
    criterion: nn.Module, device: torch.device,
) -> float:
    model.eval()
    total_loss = 0.0
    n_batches = 0
    for x_seq, y, _ in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        pred = model(x_seq)
        loss = criterion(pred, y)
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)


def train_model(
    model: nn.Module, loaders: Dict[str, DataLoader],
    device: torch.device, ticker: str,
) -> nn.Module:
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5
    )

    ckpt_dir = CONFIG["CKPT_DIR"]
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, f"transformer_lstm_reg_{ticker}.pt")

    best_val_loss = float("inf")
    patience_counter = 0
    best_epoch = 0

    for epoch in range(1, CONFIG["EPOCHS"] + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, loaders["train"], criterion, optimizer, device)
        val_loss = eval_one_epoch(model, loaders["val"], criterion, device)
        scheduler.step(val_loss)
        elapsed = time.time() - t0

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_epoch = epoch
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1

        if epoch <= 5 or epoch % 10 == 0 or patience_counter >= CONFIG["PATIENCE"]:
            lr_now = optimizer.param_groups[0]["lr"]
            print(f"    Epoch {epoch:3d} | "
                  f"train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | "
                  f"lr={lr_now:.1e} | {elapsed:.1f}s | "
                  f"patience={patience_counter}/{CONFIG['PATIENCE']}")

        if patience_counter >= CONFIG["PATIENCE"]:
            print(f"    Early stopping at epoch {epoch}, best={best_epoch}")
            break

    if os.path.isfile(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    return model


# ============================================================
# 推理 & 评估
# ============================================================
@torch.no_grad()
def predict_all(
    model: nn.Module, loader: DataLoader, device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    preds, trues, closes = [], [], []
    for x_seq, y, cur in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        pred = model(x_seq)
        preds.append(pred.cpu().numpy())
        trues.append(y.numpy())
        closes.append(cur.numpy())
    return np.concatenate(preds), np.concatenate(trues), np.concatenate(closes)


def _directional_accuracy(y_pred: np.ndarray, y_true: np.ndarray, y_current: np.ndarray) -> float:
    pred_dir = np.sign(y_pred - y_current)
    true_dir = np.sign(y_true - y_current)
    mask = true_dir != 0
    if mask.sum() == 0:
        return 0.5
    return float(np.mean(pred_dir[mask] == true_dir[mask]))


def _prediction_lag(y_pred: np.ndarray, y_current: np.ndarray) -> float:
    if len(y_pred) < 3:
        return np.nan
    return float(np.corrcoef(y_pred, y_current)[0, 1])


def _da_z_score_and_p(da: float, n: int) -> Tuple[float, float]:
    if n == 0:
        return 0.0, 1.0
    se = np.sqrt(0.25 / n)
    z = (da - 0.5) / se
    p = 1.0 - stats.norm.cdf(z)
    return float(z), float(p)


def eval_regression(
    y_pred: np.ndarray, y_true: np.ndarray, y_current: np.ndarray
) -> Dict[str, float]:
    da = _directional_accuracy(y_pred, y_true, y_current)
    mse = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(mse))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    z, p = _da_z_score_and_p(da, len(y_true))
    lag = _prediction_lag(y_pred, y_current)
    return {
        "da": da, "mse": mse, "rmse": rmse, "mae": mae,
        "r2": r2, "z_score": z, "p_value": p, "pred_lag": lag,
    }


# ============================================================
# 保存预测
# ============================================================
def _save_predictions(
    ticker: str, current: np.ndarray,
    predicted: np.ndarray, actual: np.ndarray, path: str,
) -> None:
    df = pd.DataFrame({
        "ticker": ticker, "current": current,
        "predicted": predicted, "actual": actual,
    })
    header = not os.path.isfile(path)
    try:
        df.to_csv(path, mode="a", header=header, index=False)
    except Exception as e:
        print(f"[WARN] 保存预测失败 {path}: {e}")


# ============================================================
# 单只股票全流程
# ============================================================
def run_single_ticker(ticker: str, device: torch.device) -> Optional[Dict]:
    data = load_ticker_data(ticker)
    if data is None:
        return None

    train, val, test = data
    try:
        arrays = prepare_arrays(train, val, test)
    except (KeyError, ValueError) as e:
        print(f"[ERROR] {ticker} 特征构建失败: {e}")
        return None

    n_features = arrays["n_features"]
    print(f"  {ticker}: features={n_features}, "
          f"train={len(arrays['X_train'])}, "
          f"val={len(arrays['X_val'])}, "
          f"test={len(arrays['X_test'])}")

    loaders = build_dataloaders(arrays)
    print(f"  DataLoader samples: "
          f"train={len(loaders['train'].dataset)}, "
          f"val={len(loaders['val'].dataset)}, "
          f"test={len(loaders['test'].dataset)}")

    model = TransformerLSTM(input_size=n_features).to(device)
    param_count = sum(p.numel() for p in model.parameters())
    print(f"  Model params: {param_count:,}")

    model = train_model(model, loaders, device, ticker)

    y_pred, y_true, y_current = predict_all(model, loaders["test"], device)
    metrics = eval_regression(y_pred, y_true, y_current)
    metrics["ticker"] = ticker
    metrics["n_test"] = len(y_true)
    metrics["model"] = "TransformerLSTM"

    pred_path = os.path.join(CONFIG["OUTPUT_DIR"], "transformer_lstm_reg_predictions.csv")
    _save_predictions(ticker, y_current, y_pred, y_true, pred_path)

    print(f"  → DA={metrics['da']:.4f} | RMSE={metrics['rmse']:.4f} | "
          f"R²={metrics['r2']:.4f} | pred_lag={metrics['pred_lag']:.4f} | "
          f"z={metrics['z_score']:.2f}")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics


# ============================================================
# 汇总表
# ============================================================
def _print_per_ticker_table(all_results: List[Dict]) -> None:
    df = pd.DataFrame(all_results)
    cols = ["model", "ticker", "n_test", "da", "mse", "rmse",
            "mae", "r2", "z_score", "p_value", "pred_lag"]
    df = df[[c for c in cols if c in df.columns]]
    fmt_map = {
        "da": "{:.4f}", "mse": "{:.4f}", "rmse": "{:.4f}",
        "mae": "{:.4f}", "r2": "{:.4f}", "z_score": "{:.2f}",
        "p_value": "{:.6f}", "pred_lag": "{:.4f}",
    }
    for col, f in fmt_map.items():
        if col in df.columns:
            df[col] = df[col].apply(lambda x: f.format(x) if pd.notna(x) else "N/A")
    print("\n## Transformer+LSTM Regression — Per-Ticker Results\n")
    print(df.to_markdown(index=False))


def _print_summary(all_results: List[Dict]) -> None:
    df = pd.DataFrame(all_results)
    summary = {
        "mean_da": f"{df['da'].mean():.4f}",
        "std_da": f"{df['da'].std():.4f}",
        "mean_rmse": f"{df['rmse'].mean():.4f}",
        "mean_r2": f"{df['r2'].mean():.4f}",
        "mean_pred_lag": f"{df['pred_lag'].mean():.4f}",
        "n_tickers": len(df),
    }
    print("\n## Transformer+LSTM Regression — Summary\n")
    print(pd.DataFrame([summary]).to_markdown(index=False))


# ============================================================
# main
# ============================================================
def main() -> None:
    print("=" * 60)
    print("Transformer+LSTM Regression — Next Close Prediction")
    print(f"D_MODEL={CONFIG['D_MODEL']} | NHEAD={CONFIG['NHEAD']} | "
          f"TF_LAYERS={CONFIG['TF_LAYERS']} | FF_DIM={CONFIG['FF_DIM']}")
    print(f"LSTM_HIDDEN={CONFIG['HIDDEN']} | SEQ_LEN={CONFIG['SEQ_LEN']} | "
          f"LR={CONFIG['LR']} | EPOCHS≤{CONFIG['EPOCHS']}")
    print("=" * 60)

    os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
    os.makedirs(CONFIG["CKPT_DIR"], exist_ok=True)

    pred_path = os.path.join(CONFIG["OUTPUT_DIR"], "transformer_lstm_reg_predictions.csv")
    if os.path.isfile(pred_path):
        os.remove(pred_path)

    device = get_device()
    all_results: List[Dict] = []

    for ticker in CONFIG["TICKERS"]:
        print(f"\n{'='*40}")
        print(f"[Processing] {ticker}")
        print(f"{'='*40}")
        result = run_single_ticker(ticker, device)
        if result is not None:
            all_results.append(result)

    if len(all_results) == 0:
        print("[ERROR] 所有股票均无有效数据。")
        return

    summary_path = os.path.join(CONFIG["OUTPUT_DIR"], "transformer_lstm_reg_summary.csv")
    try:
        pd.DataFrame(all_results).to_csv(summary_path, index=False)
        print(f"\n[SAVED] 汇总 → {summary_path}")
    except Exception as e:
        print(f"[WARN] 保存汇总失败: {e}")

    _print_per_ticker_table(all_results)
    _print_summary(all_results)


if __name__ == "__main__":
    main()

Transformer+LSTM Regression — Next Close Prediction
D_MODEL=64 | NHEAD=4 | TF_LAYERS=2 | FF_DIM=256
LSTM_HIDDEN=128 | SEQ_LEN=60 | LR=0.0005 | EPOCHS≤100
[DEVICE] CUDA — NVIDIA RTX PRO 6000 Blackwell Server Edition

[Processing] AAPL
  AAPL: features=40, train=8238, val=1176, test=2355
  DataLoader samples: train=8178, val=1116, test=2295
  Model params: 210,241
    Epoch   1 | train_loss=12852.904236 | val_loss=18693.396430 | lr=5.0e-04 | 0.6s | patience=0/10
    Epoch   2 | train_loss=6425.211667 | val_loss=8911.736789 | lr=5.0e-04 | 0.5s | patience=0/10
    Epoch   3 | train_loss=1409.174503 | val_loss=1694.976983 | lr=5.0e-04 | 0.4s | patience=0/10
    Epoch   4 | train_loss=800.608261 | val_loss=257.877172 | lr=5.0e-04 | 0.4s | patience=0/10
    Epoch   5 | train_loss=1643.199652 | val_loss=285.092553 | lr=5.0e-04 | 0.4s | patience=1/10
    Epoch  10 | train_loss=1602.142158 | val_loss=252.612636 | lr=5.0e-04 | 0.5s | patience=0/10
    Epoch  20 | train_loss=758.639070 | val_loss=

In [ ]:
"""
transformer_lstm_classification.py
===================================
Transformer Encoder + LSTM 混合架构三分类
Features → Linear proj → PE → TransformerEncoder → LSTM → MLP head → 3类logits
标签：ret >THRESHOLD→2(Up), <-THRESHOLD→0(Down), else→1(Flat)
Colab A100 / RTX PRO 6000 可运行
"""

import os
import math
import time
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================
CONFIG: Dict = {
    "SPLITS_DIR": "splits",
    "FEATURES_DIR": "features",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "FREQ": "1hour",
    "SEQ_LEN": 60,
    "D_MODEL": 64,
    "NHEAD": 4,
    "TF_LAYERS": 2,
    "FF_DIM": 256,
    "TF_DROPOUT": 0.1,
    "HIDDEN": 128,
    "LSTM_LAYERS": 1,
    "DROPOUT": 0.2,
    "LR": 5e-4,
    "WEIGHT_DECAY": 1e-5,
    "BATCH": 64,
    "EPOCHS": 100,
    "PATIENCE": 10,
    "THRESHOLD": 0.001,
    "OUTPUT_DIR": "results",
    "CKPT_DIR": "checkpoints",
    "TRAIN_RATIO": 0.70,
    "VAL_RATIO": 0.15,
    "DROP_COLS": ["timestamp", "open", "high", "low", "close", "volume",
                  "date", "datetime", "time", "ts_event"],
    "NUM_CLASSES": 3,
    "CLASS_NAMES": {0: "Down", 1: "Flat", 2: "Up"},
}


# ============================================================
# 设备检测
# ============================================================
def get_device() -> torch.device:
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        print(f"[DEVICE] CUDA — {torch.cuda.get_device_name(0)}")
        return dev
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("[DEVICE] Apple MPS")
        return torch.device("mps")
    print("[DEVICE] CPU")
    return torch.device("cpu")


# ============================================================
# 数据加载
# ============================================================
def _find_feature_cols(df: pd.DataFrame) -> List[str]:
    drop_set = {c.lower() for c in CONFIG["DROP_COLS"]}
    return [c for c in df.columns if c.lower() not in drop_set]


def _load_splits_csv(ticker: str) -> Optional[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    base = os.path.join(CONFIG["SPLITS_DIR"], f"{ticker}_{CONFIG['FREQ']}")
    paths = {k: os.path.join(base, f"{k}.csv") for k in ("train", "val", "test")}
    if not all(os.path.isfile(p) for p in paths.values()):
        return None
    try:
        return (pd.read_csv(paths["train"]),
                pd.read_csv(paths["val"]),
                pd.read_csv(paths["test"]))
    except Exception as e:
        print(f"[WARN] 读取splits失败 {ticker}: {e}")
        return None


def _splits_have_features(df: pd.DataFrame) -> bool:
    return len(_find_feature_cols(df)) >= 5


def _load_features_csv(ticker: str) -> Optional[pd.DataFrame]:
    path = os.path.join(CONFIG["FEATURES_DIR"], f"{ticker}_{CONFIG['FREQ']}_features.csv")
    if not os.path.isfile(path):
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"[WARN] 读取features失败 {ticker}: {e}")
        return None


def _split_by_ratio(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    n = len(df)
    n_train = int(n * CONFIG["TRAIN_RATIO"])
    n_val = int(n * CONFIG["VAL_RATIO"])
    return (df.iloc[:n_train].copy(),
            df.iloc[n_train:n_train + n_val].copy(),
            df.iloc[n_train + n_val:].copy())


def load_ticker_data(ticker: str) -> Optional[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    splits_result = _load_splits_csv(ticker)
    if splits_result is not None:
        train, val, test = splits_result
        if _splits_have_features(train):
            return train, val, test
        feat_df = _load_features_csv(ticker)
        if feat_df is not None:
            return _split_by_ratio(feat_df)
        return train, val, test
    feat_df = _load_features_csv(ticker)
    if feat_df is not None:
        return _split_by_ratio(feat_df)
    print(f"[ERROR] {ticker}: 无可用数据文件")
    return None


# ============================================================
# 标签构建
# ============================================================
def _ensure_close_column(df: pd.DataFrame) -> str:
    for c in df.columns:
        if c.lower() == "close":
            return c
    raise KeyError("找不到 close 列")


def _make_label(close_now: np.ndarray, close_next: np.ndarray) -> np.ndarray:
    ret = (close_next - close_now) / close_now
    labels = np.ones(len(ret), dtype=np.int64)
    labels[ret > CONFIG["THRESHOLD"]] = 2
    labels[ret < -CONFIG["THRESHOLD"]] = 0
    return labels


# ============================================================
# 特征 / Scaler
# ============================================================
def prepare_arrays(
    train: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame
) -> Dict[str, np.ndarray]:
    close_col = _ensure_close_column(train)
    feat_cols = _find_feature_cols(train)
    if len(feat_cols) == 0:
        raise ValueError("无可用特征列")

    scaler = StandardScaler()
    result: Dict = {"feature_names": feat_cols, "close_col": close_col}

    for split_name, df in [("train", train), ("val", val), ("test", test)]:
        df = df.copy().reset_index(drop=True)
        available = [c for c in feat_cols if c in df.columns]
        X = df[available].copy()
        X = X.fillna(X.median()).fillna(0.0)

        if split_name == "train":
            X_scaled = scaler.fit_transform(X.values)
        else:
            X_scaled = scaler.transform(X.values)

        close_arr = df[close_col].values.astype(np.float64)
        next_close = np.empty(len(close_arr), dtype=np.float64)
        next_close[:-1] = close_arr[1:]
        next_close[-1] = np.nan

        labels = np.full(len(close_arr), -1, dtype=np.int64)
        valid_mask = ~np.isnan(next_close)
        labels[valid_mask] = _make_label(close_arr[valid_mask], next_close[valid_mask])

        result[f"X_{split_name}"] = X_scaled.astype(np.float32)
        result[f"close_{split_name}"] = close_arr
        result[f"label_{split_name}"] = labels

    result["scaler"] = scaler
    result["n_features"] = result["X_train"].shape[1]
    return result


def _compute_class_weights(labels: np.ndarray, device: torch.device) -> torch.Tensor:
    valid = labels[labels >= 0]
    counts = np.bincount(valid, minlength=CONFIG["NUM_CLASSES"]).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = 1.0 / counts
    weights = weights / weights.sum() * CONFIG["NUM_CLASSES"]
    return torch.tensor(weights, dtype=torch.float32).to(device)


# ============================================================
# Dataset & DataLoader
# ============================================================
class StockDataset(Dataset):
    def __init__(self, X: np.ndarray, close: np.ndarray,
                 labels: np.ndarray, seq_len: int) -> None:
        self.X = X
        self.close = close
        self.labels = labels
        self.seq_len = seq_len
        self.valid_indices = self._build_valid_indices()

    def _build_valid_indices(self) -> np.ndarray:
        max_start = len(self.X) - self.seq_len
        indices = []
        for i in range(max_start):
            end_idx = i + self.seq_len - 1
            if self.labels[end_idx] >= 0:
                indices.append(i)
        return np.array(indices, dtype=np.int64)

    def __len__(self) -> int:
        return len(self.valid_indices)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        start = self.valid_indices[idx]
        end = start + self.seq_len
        x_seq = torch.from_numpy(self.X[start:end])
        label = torch.tensor(self.labels[end - 1], dtype=torch.long)
        return x_seq, label


def build_dataloaders(arrays: Dict[str, np.ndarray]) -> Dict[str, DataLoader]:
    seq_len = CONFIG["SEQ_LEN"]
    batch = CONFIG["BATCH"]
    loaders: Dict[str, DataLoader] = {}
    for split in ("train", "val", "test"):
        ds = StockDataset(
            X=arrays[f"X_{split}"],
            close=arrays[f"close_{split}"],
            labels=arrays[f"label_{split}"],
            seq_len=seq_len,
        )
        loaders[split] = DataLoader(
            ds, batch_size=batch, shuffle=False,
            num_workers=0, pin_memory=True, drop_last=False,
        )
    return loaders


# ============================================================
# 模型：Transformer Encoder + LSTM → 3分类
# ============================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 500) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1), :]


class TransformerLSTMClf(nn.Module):
    """
    Features → proj(N_F→64) → PE → TransformerEncoder → LSTM → MLP → 3类logits
    """

    def __init__(self, input_size: int) -> None:
        super().__init__()
        d_model = CONFIG["D_MODEL"]

        self.proj = nn.Linear(input_size, d_model)
        self.pe = PositionalEncoding(d_model, max_len=CONFIG["SEQ_LEN"] + 50)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=CONFIG["NHEAD"],
            dim_feedforward=CONFIG["FF_DIM"],
            dropout=CONFIG["TF_DROPOUT"],
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=CONFIG["TF_LAYERS"],
        )

        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=CONFIG["HIDDEN"],
            num_layers=CONFIG["LSTM_LAYERS"],
            batch_first=True,
        )

        self.head = nn.Sequential(
            nn.Linear(CONFIG["HIDDEN"], 64),
            nn.ReLU(),
            nn.Dropout(CONFIG["DROPOUT"]),
            nn.Linear(64, CONFIG["NUM_CLASSES"]),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.proj(x)
        x = self.pe(x)
        x = self.encoder(x)
        lstm_out, _ = self.lstm(x)
        last = lstm_out[:, -1, :]
        return self.head(last)


# ============================================================
# 训练循环
# ============================================================
def train_one_epoch(
    model: nn.Module, loader: DataLoader,
    criterion: nn.Module, optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> Tuple[float, float]:
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for x_seq, label in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(x_seq)
        loss = criterion(logits, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * label.size(0)
        correct += (logits.argmax(dim=1) == label).sum().item()
        total += label.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def eval_one_epoch(
    model: nn.Module, loader: DataLoader,
    criterion: nn.Module, device: torch.device,
) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    for x_seq, label in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)
        logits = model(x_seq)
        loss = criterion(logits, label)
        total_loss += loss.item() * label.size(0)
        correct += (logits.argmax(dim=1) == label).sum().item()
        total += label.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


def train_model(
    model: nn.Module, loaders: Dict[str, DataLoader],
    criterion: nn.Module, device: torch.device, ticker: str,
) -> nn.Module:
    optimizer = torch.optim.Adam(
        model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5
    )

    ckpt_dir = CONFIG["CKPT_DIR"]
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, f"transformer_lstm_clf_{ticker}.pt")

    best_val_loss = float("inf")
    patience_counter = 0
    best_epoch = 0

    for epoch in range(1, CONFIG["EPOCHS"] + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(
            model, loaders["train"], criterion, optimizer, device)
        val_loss, val_acc = eval_one_epoch(
            model, loaders["val"], criterion, device)
        scheduler.step(val_loss)
        elapsed = time.time() - t0

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_epoch = epoch
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1

        if epoch <= 5 or epoch % 10 == 0 or patience_counter >= CONFIG["PATIENCE"]:
            lr_now = optimizer.param_groups[0]["lr"]
            print(f"    Epoch {epoch:3d} | "
                  f"t_loss={train_loss:.4f} t_acc={train_acc:.4f} | "
                  f"v_loss={val_loss:.4f} v_acc={val_acc:.4f} | "
                  f"lr={lr_now:.1e} | {elapsed:.1f}s | "
                  f"pat={patience_counter}/{CONFIG['PATIENCE']}")

        if patience_counter >= CONFIG["PATIENCE"]:
            print(f"    Early stopping at epoch {epoch}, best={best_epoch}")
            break

    if os.path.isfile(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    return model


# ============================================================
# 推理 & 评估
# ============================================================
@torch.no_grad()
def predict_all(
    model: nn.Module, loader: DataLoader, device: torch.device,
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    preds, trues = [], []
    for x_seq, label in loader:
        x_seq = x_seq.to(device, non_blocking=True)
        logits = model(x_seq)
        preds.append(logits.argmax(dim=1).cpu().numpy())
        trues.append(label.numpy())
    return np.concatenate(preds), np.concatenate(trues)


def _class_distribution(y: np.ndarray) -> Dict[str, str]:
    total = len(y)
    if total == 0:
        return {}
    dist = {}
    for cls_id, cls_name in CONFIG["CLASS_NAMES"].items():
        cnt = int(np.sum(y == cls_id))
        dist[f"{cls_name}_count"] = cnt
        dist[f"{cls_name}_pct"] = cnt / total
    return dist


def _majority_baseline(y: np.ndarray) -> float:
    if len(y) == 0:
        return 0.0
    counts = np.bincount(y, minlength=CONFIG["NUM_CLASSES"])
    return float(counts.max() / len(y))


def _acc_z_score_and_p(acc: float, n: int, baseline: float) -> Tuple[float, float]:
    if n == 0 or baseline <= 0.0 or baseline >= 1.0:
        return 0.0, 1.0
    se = np.sqrt(baseline * (1 - baseline) / n)
    z = (acc - baseline) / se
    p = 1.0 - stats.norm.cdf(z)
    return float(z), float(p)


def eval_classification(y_pred: np.ndarray, y_true: np.ndarray) -> Dict:
    acc = float(accuracy_score(y_true, y_pred))
    prec, rec, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], zero_division=0.0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    dist = _class_distribution(y_true)
    maj = _majority_baseline(y_true)
    z, p = _acc_z_score_and_p(acc, len(y_true), maj)

    result: Dict = {
        "accuracy": acc, "majority_baseline": maj,
        "z_score": z, "p_value": p, "n_test": len(y_true),
    }
    for i, cls_name in CONFIG["CLASS_NAMES"].items():
        result[f"{cls_name}_precision"] = float(prec[i])
        result[f"{cls_name}_recall"] = float(rec[i])
        result[f"{cls_name}_f1"] = float(f1[i])
        result[f"{cls_name}_support"] = int(sup[i])
    result.update(dist)
    result["confusion_matrix"] = cm.tolist()
    return result


# ============================================================
# 单只股票全流程
# ============================================================
def run_single_ticker(ticker: str, device: torch.device) -> Optional[Dict]:
    data = load_ticker_data(ticker)
    if data is None:
        return None

    train, val, test = data
    try:
        arrays = prepare_arrays(train, val, test)
    except (KeyError, ValueError) as e:
        print(f"[ERROR] {ticker} 特征构建失败: {e}")
        return None

    n_features = arrays["n_features"]
    train_labels = arrays["label_train"]
    train_valid = train_labels[train_labels >= 0]
    train_dist = _class_distribution(train_valid)
    print(f"  {ticker}: features={n_features}, "
          f"train={len(arrays['X_train'])}, "
          f"val={len(arrays['X_val'])}, "
          f"test={len(arrays['X_test'])}")
    print(f"    train dist: "
          f"Down={train_dist.get('Down_pct', 0):.1%}  "
          f"Flat={train_dist.get('Flat_pct', 0):.1%}  "
          f"Up={train_dist.get('Up_pct', 0):.1%}")

    loaders = build_dataloaders(arrays)
    print(f"  DataLoader samples: "
          f"train={len(loaders['train'].dataset)}, "
          f"val={len(loaders['val'].dataset)}, "
          f"test={len(loaders['test'].dataset)}")

    class_weights = _compute_class_weights(train_valid, device)
    print(f"    class_weights: {class_weights.cpu().numpy().round(3)}")
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    model = TransformerLSTMClf(input_size=n_features).to(device)
    param_count = sum(p.numel() for p in model.parameters())
    print(f"  Model params: {param_count:,}")

    model = train_model(model, loaders, criterion, device, ticker)

    y_pred, y_true = predict_all(model, loaders["test"], device)
    metrics = eval_classification(y_pred, y_true)
    metrics["ticker"] = ticker
    metrics["model"] = "TransformerLSTM_Clf"

    _save_predictions(ticker, y_true, y_pred)

    print(f"  → acc={metrics['accuracy']:.4f} | "
          f"majority_bl={metrics['majority_baseline']:.4f} | "
          f"z={metrics['z_score']:.2f} | "
          f"Down_f1={metrics['Down_f1']:.3f} | "
          f"Flat_f1={metrics['Flat_f1']:.3f} | "
          f"Up_f1={metrics['Up_f1']:.3f}")

    del model, criterion
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics


# ============================================================
# 保存预测
# ============================================================
def _save_predictions(ticker: str, y_true: np.ndarray, y_pred: np.ndarray) -> None:
    out_dir = CONFIG["OUTPUT_DIR"]
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "transformer_lstm_clf_predictions.csv")
    df = pd.DataFrame({"ticker": ticker, "y_true": y_true, "y_pred": y_pred})
    header = not os.path.isfile(path)
    try:
        df.to_csv(path, mode="a", header=header, index=False)
    except Exception as e:
        print(f"[WARN] 保存预测失败 {path}: {e}")


# ============================================================
# 汇总表
# ============================================================
def _print_per_ticker_table(all_results: List[Dict]) -> None:
    rows = []
    for r in all_results:
        rows.append({
            "model": r["model"], "ticker": r["ticker"], "n_test": r["n_test"],
            "accuracy": f"{r['accuracy']:.4f}",
            "majority_bl": f"{r['majority_baseline']:.4f}",
            "z_score": f"{r['z_score']:.2f}",
            "p_value": f"{r['p_value']:.6f}",
            "Down_f1": f"{r['Down_f1']:.3f}",
            "Flat_f1": f"{r['Flat_f1']:.3f}",
            "Up_f1": f"{r['Up_f1']:.3f}",
        })
    print("\n## Transformer+LSTM Classification — Per-Ticker Results\n")
    print(pd.DataFrame(rows).to_markdown(index=False))


def _print_class_dist_table(all_results: List[Dict]) -> None:
    rows = []
    for r in all_results:
        rows.append({
            "ticker": r["ticker"],
            "Down": f"{r.get('Down_count', 0)} ({r.get('Down_pct', 0):.1%})",
            "Flat": f"{r.get('Flat_count', 0)} ({r.get('Flat_pct', 0):.1%})",
            "Up": f"{r.get('Up_count', 0)} ({r.get('Up_pct', 0):.1%})",
            "majority_bl": f"{r['majority_baseline']:.4f}",
        })
    print(f"\n## Test Set Class Distribution (threshold={CONFIG['THRESHOLD']})\n")
    print(pd.DataFrame(rows).to_markdown(index=False))


def _print_confusion_matrices(all_results: List[Dict]) -> None:
    print("\n## Confusion Matrices\n")
    for r in all_results:
        cm = np.array(r["confusion_matrix"])
        print(f"### {r['ticker']}")
        cm_df = pd.DataFrame(
            cm,
            index=["true_Down", "true_Flat", "true_Up"],
            columns=["pred_Down", "pred_Flat", "pred_Up"],
        )
        print(cm_df.to_markdown())
        print()


def _print_summary(all_results: List[Dict]) -> None:
    df = pd.DataFrame(all_results)
    summary = {
        "mean_acc": f"{df['accuracy'].mean():.4f}",
        "std_acc": f"{df['accuracy'].std():.4f}",
        "mean_majority_bl": f"{df['majority_baseline'].mean():.4f}",
        "mean_Down_f1": f"{df['Down_f1'].mean():.3f}",
        "mean_Flat_f1": f"{df['Flat_f1'].mean():.3f}",
        "mean_Up_f1": f"{df['Up_f1'].mean():.3f}",
        "n_tickers": len(df),
    }
    print("\n## Transformer+LSTM Classification — Summary\n")
    print(pd.DataFrame([summary]).to_markdown(index=False))


# ============================================================
# main
# ============================================================
def main() -> None:
    print("=" * 60)
    print("Transformer+LSTM Classification — 3-Class Direction")
    print(f"D_MODEL={CONFIG['D_MODEL']} | NHEAD={CONFIG['NHEAD']} | "
          f"TF_LAYERS={CONFIG['TF_LAYERS']} | FF_DIM={CONFIG['FF_DIM']}")
    print(f"LSTM_HIDDEN={CONFIG['HIDDEN']} | THRESHOLD={CONFIG['THRESHOLD']} | "
          f"LR={CONFIG['LR']} | EPOCHS≤{CONFIG['EPOCHS']}")
    print("=" * 60)

    os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
    os.makedirs(CONFIG["CKPT_DIR"], exist_ok=True)

    pred_path = os.path.join(CONFIG["OUTPUT_DIR"], "transformer_lstm_clf_predictions.csv")
    if os.path.isfile(pred_path):
        os.remove(pred_path)

    device = get_device()
    all_results: List[Dict] = []

    for ticker in CONFIG["TICKERS"]:
        print(f"\n{'='*40}")
        print(f"[Processing] {ticker}")
        print(f"{'='*40}")
        result = run_single_ticker(ticker, device)
        if result is not None:
            all_results.append(result)

    if len(all_results) == 0:
        print("[ERROR] 所有股票均无有效数据。")
        return

    summary_path = os.path.join(CONFIG["OUTPUT_DIR"], "transformer_lstm_clf_summary.csv")
    try:
        df_save = pd.DataFrame(all_results).copy()
        df_save["confusion_matrix"] = df_save["confusion_matrix"].apply(str)
        df_save.to_csv(summary_path, index=False)
        print(f"\n[SAVED] 汇总 → {summary_path}")
    except Exception as e:
        print(f"[WARN] 保存汇总失败: {e}")

    _print_class_dist_table(all_results)
    _print_per_ticker_table(all_results)
    _print_summary(all_results)
    _print_confusion_matrices(all_results)


if __name__ == "__main__":
    main()

Transformer+LSTM Classification — 3-Class Direction
D_MODEL=64 | NHEAD=4 | TF_LAYERS=2 | FF_DIM=256
LSTM_HIDDEN=128 | THRESHOLD=0.001 | LR=0.0005 | EPOCHS≤100
[DEVICE] CUDA — NVIDIA RTX PRO 6000 Blackwell Server Edition

[Processing] AAPL
  AAPL: features=40, train=8238, val=1176, test=2355
    train dist: Down=32.2%  Flat=32.2%  Up=35.6%
  DataLoader samples: train=8178, val=1116, test=2295
    class_weights: [1.033 1.034 0.933]
  Model params: 210,371
    Epoch   1 | t_loss=1.0724 t_acc=0.4115 | v_loss=1.0618 v_acc=0.4041 | lr=5.0e-04 | 0.5s | pat=0/10
    Epoch   2 | t_loss=1.0405 t_acc=0.4258 | v_loss=1.0404 v_acc=0.4185 | lr=5.0e-04 | 0.5s | pat=0/10
    Epoch   3 | t_loss=1.0261 t_acc=0.4335 | v_loss=1.0364 v_acc=0.4185 | lr=5.0e-04 | 0.5s | pat=0/10
    Epoch   4 | t_loss=1.0124 t_acc=0.4451 | v_loss=1.0242 v_acc=0.4346 | lr=5.0e-04 | 0.4s | pat=0/10
    Epoch   5 | t_loss=1.0037 t_acc=0.4617 | v_loss=1.0236 v_acc=0.4355 | lr=5.0e-04 | 0.5s | pat=0/10
    Epoch  10 | t_loss=0.98

## **timesfm**


In [ ]:
"""
timesfm_regression.py
=====================
TimesFM 2.5 基础模型回归：Zero-shot + Fine-tuned (解冻最后4层)
单变量模型，只使用close价格
从 processed/{ticker}_1hour.csv 读取，70/15/15切分

安装（Colab）：
  !git clone https://github.com/google-research/timesfm.git /content/timesfm_repo
  !cd /content/timesfm_repo && pip install -e ".[torch]"
"""

import os
import copy
import time
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================
CONFIG: Dict = {
    "PROCESSED_DIR": "processed",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "FREQ": "1hour",
    "MODEL_ID": "google/timesfm-2.5-200m-pytorch",
    "CONTEXT_LEN": 1024,
    "HORIZON": 1,
    "TRAIN_RATIO": 0.70,
    "VAL_RATIO": 0.15,
    # Fine-tune 超参
    "FT_LR": 1e-5,
    "FT_EPOCHS": 5,
    "FT_BATCH": 32,
    "FT_UNFREEZE_LAYERS": 4,
    "OUTPUT_DIR": "results",
}


# ============================================================
# 设备检测
# ============================================================
def get_device() -> torch.device:
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        print(f"[DEVICE] CUDA — {torch.cuda.get_device_name(0)}")
        return dev
    print("[DEVICE] CPU")
    return torch.device("cpu")


# ============================================================
# 数据加载
# ============================================================
def _ensure_close_column(df: pd.DataFrame) -> str:
    for c in df.columns:
        if c.lower() == "close":
            return c
    raise KeyError("找不到 close 列")


def load_close_data(ticker: str) -> Optional[np.ndarray]:
    """从 processed/{ticker}_1hour.csv 读取close序列"""
    path = os.path.join(CONFIG["PROCESSED_DIR"], f"{ticker}_{CONFIG['FREQ']}.csv")
    if not os.path.isfile(path):
        print(f"[ERROR] 文件不存在: {path}")
        return None
    try:
        df = pd.read_csv(path)
        close_col = _ensure_close_column(df)
        close = df[close_col].values.astype(np.float64)
        if np.isnan(close).any():
            n_nan = int(np.isnan(close).sum())
            print(f"  [WARN] {ticker}: {n_nan} NaN values in close, forward-filling")
            close = pd.Series(close).ffill().bfill().values
        return close
    except Exception as e:
        print(f"[ERROR] 加载 {ticker} 失败: {e}")
        return None


def split_indices(n: int) -> Tuple[int, int]:
    train_end = int(n * CONFIG["TRAIN_RATIO"])
    val_end = train_end + int(n * CONFIG["VAL_RATIO"])
    return train_end, val_end


# ============================================================
# 滚动窗口构建
# ============================================================
def create_rolling_windows(
    close: np.ndarray,
    start_pos: int,
    end_pos: int,
    context_len: int,
) -> Tuple[List[np.ndarray], np.ndarray, np.ndarray]:
    """
    对于位置 pos in [start_pos, end_pos):
      context = close[pos - context_len + 1 : pos + 1]
      target  = close[pos + 1]
      current = close[pos]
    """
    windows: List[np.ndarray] = []
    targets: List[float] = []
    currents: List[float] = []

    for pos in range(start_pos, end_pos):
        ctx_start = pos - context_len + 1
        if ctx_start < 0:
            continue
        if pos + 1 >= len(close):
            continue
        window = close[ctx_start: pos + 1].copy()
        windows.append(window)
        targets.append(close[pos + 1])
        currents.append(close[pos])

    return windows, np.array(targets), np.array(currents)


# ============================================================
# TimesFM 2.5 加载
# ============================================================
def load_timesfm_model():
    """加载 TimesFM 2.5 模型"""
    import timesfm

    torch.set_float32_matmul_precision("high")

    print(f"  Loading {CONFIG['MODEL_ID']}...")
    model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
        CONFIG["MODEL_ID"],
    )

    model.compile(
        timesfm.ForecastConfig(
            max_context=CONFIG["CONTEXT_LEN"],
            max_horizon=CONFIG["HORIZON"],
            normalize_inputs=True,
            use_continuous_quantile_head=True,
            force_flip_invariance=True,
            infer_is_positive=True,
            fix_quantile_crossing=True,
        )
    )

    print(f"[MODEL] Loaded TimesFM 2.5: {CONFIG['MODEL_ID']}")
    return model


# ============================================================
# Zero-shot 预测
# ============================================================
def zero_shot_predict(
    tfm, windows: List[np.ndarray],
) -> np.ndarray:
    """批量零样本预测，使用 tfm.forecast()"""
    if len(windows) == 0:
        return np.array([])

    t0 = time.time()
    point_forecast, _ = tfm.forecast(
        horizon=CONFIG["HORIZON"],
        inputs=windows,
    )
    elapsed = time.time() - t0

    # point_forecast: (n_windows, horizon)
    if point_forecast.ndim == 2:
        y_pred = point_forecast[:, 0]
    else:
        y_pred = point_forecast.flatten()

    speed = len(windows) / elapsed if elapsed > 0 else 0
    print(f"    Predicted {len(windows)} windows in {elapsed:.1f}s "
          f"({speed:.0f} samples/sec)")

    return np.asarray(y_pred, dtype=np.float64)


# ============================================================
# Fine-tuning
# ============================================================
def _get_internal_torch_model(tfm) -> nn.Module:
    """获取 TimesFM 内部 PyTorch 模型"""
    for attr in ("_model", "model", "torch_model", "_torch_model",
                 "backbone", "_backbone", "net", "_net"):
        if hasattr(tfm, attr):
            candidate = getattr(tfm, attr)
            if isinstance(candidate, nn.Module):
                return candidate

    # 遍历所有属性找nn.Module
    for attr in dir(tfm):
        if attr.startswith("__"):
            continue
        try:
            candidate = getattr(tfm, attr)
            if isinstance(candidate, nn.Module):
                print(f"    [INFO] Found internal model at tfm.{attr}")
                return candidate
        except Exception:
            continue

    raise AttributeError(
        "无法获取TimesFM内部PyTorch模型。"
        f"可用属性: {[a for a in dir(tfm) if not a.startswith('__')]}"
    )


def _freeze_and_unfreeze(model: nn.Module, num_unfreeze: int = 4) -> int:
    """冻结所有参数，解冻最后num_unfreeze个层+输出头"""
    for param in model.parameters():
        param.requires_grad = False

    # 收集所有子模块
    named_children = list(model.named_children())
    all_modules = [(n, m) for n, m in model.named_modules()
                   if sum(1 for _ in m.parameters()) > 0]

    # 尝试识别layer类结构（常见命名: layers, blocks, stacks, encoder.layer）
    layer_list = None
    for attr_name in ("layers", "blocks", "stacks", "encoder_layers",
                      "decoder_layers", "layer"):
        if hasattr(model, attr_name):
            candidate = getattr(model, attr_name)
            if isinstance(candidate, (nn.ModuleList, nn.Sequential)):
                layer_list = list(enumerate(candidate))
                print(f"    Found layer list: model.{attr_name} ({len(layer_list)} layers)")
                break

    if layer_list is None:
        # 递归搜索
        for name, module in model.named_modules():
            if isinstance(module, (nn.ModuleList, nn.Sequential)) and len(list(module)) >= 4:
                layer_list = list(enumerate(module))
                print(f"    Found layer list at: {name} ({len(layer_list)} layers)")
                break

    if layer_list is not None:
        n_unfreeze = min(num_unfreeze, len(layer_list))
        for idx, layer_module in layer_list[-n_unfreeze:]:
            for param in layer_module.parameters():
                param.requires_grad = True
            print(f"    Unfrozen layer {idx}")
    else:
        # fallback：解冻最后20%参数
        print("    [WARN] 未找到layer list，解冻最后20%参数")
        all_params = list(model.named_parameters())
        n_unfreeze_p = max(1, len(all_params) // 5)
        for name, param in all_params[-n_unfreeze_p:]:
            param.requires_grad = True

    # 解冻输出头
    output_keywords = ("output", "head", "horizon", "final", "fc",
                       "proj_out", "prediction", "quantile")
    for name, param in model.named_parameters():
        if any(kw in name.lower() for kw in output_keywords):
            param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"    Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
    return trainable


def _build_finetune_forward(tfm, model: nn.Module, device: torch.device):
    """构建fine-tune用的forward函数"""
    dummy = torch.randn(2, CONFIG["CONTEXT_LEN"]).to(device)
    dummy_pad = torch.zeros(2, CONFIG["CONTEXT_LEN"]).to(device)
    dummy_freq = torch.zeros(2, dtype=torch.long).to(device)

    signatures = [
        ("model(input, paddings, freq, horizon)",
         lambda m, x: m(x, torch.zeros_like(x),
                        torch.zeros(x.size(0), dtype=torch.long, device=x.device),
                        CONFIG["HORIZON"])),
        ("model(input, freq, horizon)",
         lambda m, x: m(x,
                        torch.zeros(x.size(0), dtype=torch.long, device=x.device),
                        CONFIG["HORIZON"])),
        ("model.decode(input, paddings, freq, horizon)",
         lambda m, x: m.decode(x, torch.zeros_like(x),
                               torch.zeros(x.size(0), dtype=torch.long, device=x.device),
                               CONFIG["HORIZON"]) if hasattr(m, "decode") else (_ for _ in ()).throw(AttributeError())),
        ("model(input, paddings)",
         lambda m, x: m(x, torch.zeros_like(x))),
        ("model(input)",
         lambda m, x: m(x)),
    ]

    for desc, fn in signatures:
        try:
            with torch.no_grad():
                out = fn(model, dummy)
            print(f"    Forward方式: {desc}")
            return lambda batch_x, _fn=fn: _fn(model, batch_x)
        except Exception:
            continue

    return None


def fine_tune_model(
    tfm,
    train_windows: List[np.ndarray],
    train_targets: np.ndarray,
    val_windows: List[np.ndarray],
    val_targets: np.ndarray,
    device: torch.device,
) -> bool:
    """微调最后4层，返回True成功/False失败"""
    try:
        model = _get_internal_torch_model(tfm)
    except AttributeError as e:
        print(f"    [SKIP] Fine-tune失败: {e}")
        return False

    model.to(device)
    _freeze_and_unfreeze(model, CONFIG["FT_UNFREEZE_LAYERS"])

    forward_fn = _build_finetune_forward(tfm, model, device)
    if forward_fn is None:
        print("    [SKIP] 无法确定内部forward方法，跳过fine-tune")
        return False

    # 准备数据（padding/truncating到 CONTEXT_LEN）
    ctx = CONFIG["CONTEXT_LEN"]

    def _pad_or_trunc(windows: List[np.ndarray]) -> torch.Tensor:
        result = np.zeros((len(windows), ctx), dtype=np.float32)
        for i, w in enumerate(windows):
            length = min(len(w), ctx)
            result[i, ctx - length:] = w[-length:]
        return torch.tensor(result, dtype=torch.float32)

    train_x = _pad_or_trunc(train_windows)
    train_y = torch.tensor(train_targets, dtype=torch.float32)
    val_x = _pad_or_trunc(val_windows)
    val_y = torch.tensor(val_targets, dtype=torch.float32)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if len(trainable_params) == 0:
        print("    [SKIP] 无可训练参数")
        return False

    optimizer = torch.optim.Adam(trainable_params, lr=CONFIG["FT_LR"])
    criterion = nn.MSELoss()
    batch_size = CONFIG["FT_BATCH"]

    best_val_loss = float("inf")
    best_state = None

    for epoch in range(1, CONFIG["FT_EPOCHS"] + 1):
        # --- 训练 ---
        model.train()
        epoch_loss = 0.0
        n_batches = 0

        indices = np.arange(len(train_x))
        for i in range(0, len(indices), batch_size):
            batch_idx = indices[i:i + batch_size]
            bx = train_x[batch_idx].to(device)
            by = train_y[batch_idx].to(device)

            optimizer.zero_grad()
            try:
                pred = forward_fn(bx)
            except Exception as e:
                print(f"    [ERROR] Forward failed at epoch {epoch}: {e}")
                return False

            if isinstance(pred, tuple):
                pred = pred[0]
            if pred.ndim == 2:
                pred = pred[:, 0]
            pred = pred.squeeze()

            if pred.shape != by.shape:
                pred = pred[:by.shape[0]]

            loss = criterion(pred, by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        avg_train = epoch_loss / max(n_batches, 1)

        # --- 验证 ---
        model.eval()
        val_loss = 0.0
        n_val = 0
        with torch.no_grad():
            for i in range(0, len(val_x), batch_size):
                bx = val_x[i:i + batch_size].to(device)
                by = val_y[i:i + batch_size].to(device)
                try:
                    pred = forward_fn(bx)
                except Exception:
                    continue
                if isinstance(pred, tuple):
                    pred = pred[0]
                if pred.ndim == 2:
                    pred = pred[:, 0]
                pred = pred.squeeze()
                if pred.shape != by.shape:
                    pred = pred[:by.shape[0]]
                val_loss += criterion(pred, by).item()
                n_val += 1

        avg_val = val_loss / max(n_val, 1)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = copy.deepcopy(model.state_dict())

        print(f"    FT Epoch {epoch}/{CONFIG['FT_EPOCHS']} | "
              f"train_loss={avg_train:.6f} | val_loss={avg_val:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    return True


# ============================================================
# 评估
# ============================================================
def _directional_accuracy(y_pred: np.ndarray, y_true: np.ndarray, y_current: np.ndarray) -> float:
    pred_dir = np.sign(y_pred - y_current)
    true_dir = np.sign(y_true - y_current)
    mask = true_dir != 0
    if mask.sum() == 0:
        return 0.5
    return float(np.mean(pred_dir[mask] == true_dir[mask]))


def _prediction_lag(y_pred: np.ndarray, y_current: np.ndarray) -> float:
    if len(y_pred) < 3:
        return np.nan
    return float(np.corrcoef(y_pred, y_current)[0, 1])


def _da_z_score_and_p(da: float, n: int) -> Tuple[float, float]:
    if n == 0:
        return 0.0, 1.0
    se = np.sqrt(0.25 / n)
    z = (da - 0.5) / se
    p = 1.0 - stats.norm.cdf(z)
    return float(z), float(p)


def eval_regression(
    y_pred: np.ndarray, y_true: np.ndarray, y_current: np.ndarray
) -> Dict[str, float]:
    da = _directional_accuracy(y_pred, y_true, y_current)
    mse = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(mse))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    z, p = _da_z_score_and_p(da, len(y_true))
    lag = _prediction_lag(y_pred, y_current)
    return {
        "da": da, "mse": mse, "rmse": rmse, "mae": mae,
        "r2": r2, "z_score": z, "p_value": p, "pred_lag": lag,
    }


# ============================================================
# 保存预测
# ============================================================
def _save_predictions(
    ticker: str, current: np.ndarray, predicted: np.ndarray,
    actual: np.ndarray, path: str,
) -> None:
    df = pd.DataFrame({
        "ticker": ticker, "current": current,
        "predicted": predicted, "actual": actual,
    })
    header = not os.path.isfile(path)
    try:
        df.to_csv(path, mode="a", header=header, index=False)
    except Exception as e:
        print(f"[WARN] 保存失败 {path}: {e}")


# ============================================================
# 单只股票全流程
# ============================================================
def run_single_ticker(
    ticker: str, tfm, device: torch.device,
) -> Optional[List[Dict]]:
    close = load_close_data(ticker)
    if close is None:
        return None

    train_end, val_end = split_indices(len(close))
    ctx = CONFIG["CONTEXT_LEN"]

    print(f"  {ticker}: total={len(close)}, train=0:{train_end}, "
          f"val={train_end}:{val_end}, test={val_end}:{len(close)}")

    # --- test窗口 ---
    test_windows, test_targets, test_currents = create_rolling_windows(
        close, val_end, len(close) - 1, ctx)
    if len(test_windows) == 0:
        # context_len过大时回退到可用最大长度
        available_ctx = val_end
        print(f"  [WARN] context_len={ctx} > available, trying {available_ctx}")
        test_windows, test_targets, test_currents = create_rolling_windows(
            close, val_end, len(close) - 1, available_ctx)
        if len(test_windows) == 0:
            print(f"  [SKIP] {ticker}: 无法构建test窗口")
            return None
    print(f"  Test windows: {len(test_windows)} (context≤{len(test_windows[0])})")

    results: List[Dict] = []

    # === Zero-shot ===
    print(f"  --- Zero-shot ---")
    zs_pred = zero_shot_predict(tfm, test_windows)
    zs_metrics = eval_regression(zs_pred, test_targets, test_currents)
    zs_metrics["ticker"] = ticker
    zs_metrics["model"] = "TimesFM2.5_ZeroShot"
    zs_metrics["n_test"] = len(test_targets)
    results.append(zs_metrics)

    zs_path = os.path.join(CONFIG["OUTPUT_DIR"], "timesfm_zs_reg_predictions.csv")
    _save_predictions(ticker, test_currents, zs_pred, test_targets, zs_path)

    print(f"    DA={zs_metrics['da']:.4f} | RMSE={zs_metrics['rmse']:.4f} | "
          f"R²={zs_metrics['r2']:.4f} | pred_lag={zs_metrics['pred_lag']:.4f}")

    # === Fine-tuned ===
    print(f"  --- Fine-tuning (unfreeze last {CONFIG['FT_UNFREEZE_LAYERS']} layers) ---")

    # 保存原始权重
    original_state = None
    try:
        internal = _get_internal_torch_model(tfm)
        original_state = copy.deepcopy(internal.state_dict())
    except Exception:
        pass

    # train/val 窗口
    train_windows, train_targets_arr, _ = create_rolling_windows(
        close, ctx - 1, train_end - 1, ctx)
    val_windows, val_targets_arr, _ = create_rolling_windows(
        close, train_end, val_end - 1, ctx)

    # 如果context_len过大导致窗口为空，用较短context
    if len(train_windows) == 0:
        shorter_ctx = min(ctx, train_end // 2)
        print(f"    [WARN] 回退context_len={shorter_ctx}")
        train_windows, train_targets_arr, _ = create_rolling_windows(
            close, shorter_ctx - 1, train_end - 1, shorter_ctx)
        val_windows, val_targets_arr, _ = create_rolling_windows(
            close, train_end, val_end - 1, shorter_ctx)

    print(f"    Train windows: {len(train_windows)}, Val windows: {len(val_windows)}")

    ft_success = False
    if len(train_windows) > 0 and len(val_windows) > 0:
        ft_success = fine_tune_model(
            tfm, train_windows, train_targets_arr,
            val_windows, val_targets_arr, device,
        )

    if ft_success:
        ft_pred = zero_shot_predict(tfm, test_windows)
        ft_metrics = eval_regression(ft_pred, test_targets, test_currents)
    else:
        print("    [WARN] Fine-tune失败/跳过，复用zero-shot结果")
        ft_pred = zs_pred.copy()
        ft_metrics = {k: v for k, v in zs_metrics.items()}

    ft_metrics["ticker"] = ticker
    ft_metrics["model"] = "TimesFM2.5_FineTuned"
    ft_metrics["n_test"] = len(test_targets)
    results.append(ft_metrics)

    ft_path = os.path.join(CONFIG["OUTPUT_DIR"], "timesfm_ft_reg_predictions.csv")
    _save_predictions(ticker, test_currents, ft_pred, test_targets, ft_path)

    print(f"    DA={ft_metrics['da']:.4f} | RMSE={ft_metrics['rmse']:.4f} | "
          f"R²={ft_metrics['r2']:.4f} | pred_lag={ft_metrics['pred_lag']:.4f}")

    # 恢复原始权重
    if original_state is not None:
        try:
            internal = _get_internal_torch_model(tfm)
            internal.load_state_dict(original_state)
        except Exception:
            pass

    return results


# ============================================================
# 汇总表
# ============================================================
def _print_per_ticker_table(all_results: List[Dict]) -> None:
    df = pd.DataFrame(all_results)
    cols = ["model", "ticker", "n_test", "da", "mse", "rmse",
            "mae", "r2", "z_score", "p_value", "pred_lag"]
    df = df[[c for c in cols if c in df.columns]]
    fmt_map = {
        "da": "{:.4f}", "mse": "{:.4f}", "rmse": "{:.4f}",
        "mae": "{:.4f}", "r2": "{:.4f}", "z_score": "{:.2f}",
        "p_value": "{:.6f}", "pred_lag": "{:.4f}",
    }
    for col, f in fmt_map.items():
        if col in df.columns:
            df[col] = df[col].apply(lambda x: f.format(x) if pd.notna(x) else "N/A")
    print("\n## TimesFM 2.5 Regression — Per-Ticker Results\n")
    print(df.to_markdown(index=False))


def _print_summary(all_results: List[Dict]) -> None:
    df = pd.DataFrame(all_results)
    rows = []
    for model_name in df["model"].unique():
        sub = df[df["model"] == model_name]
        rows.append({
            "model": model_name,
            "mean_da": f"{sub['da'].mean():.4f}",
            "std_da": f"{sub['da'].std():.4f}",
            "mean_rmse": f"{sub['rmse'].mean():.4f}",
            "mean_r2": f"{sub['r2'].mean():.4f}",
            "mean_pred_lag": f"{sub['pred_lag'].mean():.4f}",
            "n_tickers": len(sub),
        })
    print("\n## TimesFM 2.5 Regression — Model Summary\n")
    print(pd.DataFrame(rows).to_markdown(index=False))


# ============================================================
# main
# ============================================================
def main() -> None:
    print("=" * 60)
    print("TimesFM 2.5 Regression — Zero-Shot + Fine-Tuned")
    print(f"MODEL={CONFIG['MODEL_ID']}")
    print(f"CONTEXT_LEN={CONFIG['CONTEXT_LEN']} | HORIZON={CONFIG['HORIZON']}")
    print("=" * 60)

    os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

    for fname in ("timesfm_zs_reg_predictions.csv", "timesfm_ft_reg_predictions.csv"):
        path = os.path.join(CONFIG["OUTPUT_DIR"], fname)
        if os.path.isfile(path):
            os.remove(path)

    device = get_device()

    print("\n[Loading TimesFM 2.5 model...]")
    t0 = time.time()
    tfm = load_timesfm_model()
    print(f"[Model loaded in {time.time() - t0:.1f}s]\n")

    all_results: List[Dict] = []

    for ticker in CONFIG["TICKERS"]:
        print(f"\n{'='*40}")
        print(f"[Processing] {ticker}")
        print(f"{'='*40}")
        ticker_results = run_single_ticker(ticker, tfm, device)
        if ticker_results is not None:
            all_results.extend(ticker_results)

    if len(all_results) == 0:
        print("[ERROR] 所有股票均无有效数据。")
        return

    summary_path = os.path.join(CONFIG["OUTPUT_DIR"], "timesfm_reg_summary.csv")
    try:
        pd.DataFrame(all_results).to_csv(summary_path, index=False)
        print(f"\n[SAVED] 汇总 → {summary_path}")
    except Exception as e:
        print(f"[WARN] 保存汇总失败: {e}")

    _print_per_ticker_table(all_results)
    _print_summary(all_results)


if __name__ == "__main__":
    main()

TimesFM 2.5 Regression — Zero-Shot + Fine-Tuned
MODEL=google/timesfm-2.5-200m-pytorch
CONTEXT_LEN=1024 | HORIZON=1
[DEVICE] CUDA — NVIDIA RTX PRO 6000 Blackwell Server Edition

[Loading TimesFM 2.5 model...]
  Loading google/timesfm-2.5-200m-pytorch...


config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

Downloaded.


model.safetensors:   0%|          | 0.00/925M [00:00<?, ?B/s]

[MODEL] Loaded TimesFM 2.5: google/timesfm-2.5-200m-pytorch
[Model loaded in 6.6s]


[Processing] AAPL
  AAPL: total=11829, train=0:8280, val=8280:10054, test=10054:11829
  Test windows: 1774 (context≤1024)
  --- Zero-shot ---
    Predicted 1774 windows in 70.5s (25 samples/sec)
    DA=0.5083 | RMSE=0.8125 | R²=0.9946 | pred_lag=1.0000
  --- Fine-tuning (unfreeze last 4 layers) ---
    Train windows: 7256, Val windows: 1773
    Found layer list at: stacked_xf (20 layers)
    Unfrozen layer 16
    Unfrozen layer 17
    Unfrozen layer 18
    Unfrozen layer 19
    Trainable: 73,750,720 / 231,289,280 (31.9%)
    [SKIP] 无法确定内部forward方法，跳过fine-tune
    [WARN] Fine-tune失败/跳过，复用zero-shot结果
    DA=0.5083 | RMSE=0.8125 | R²=0.9946 | pred_lag=1.0000

[Processing] MSFT
  MSFT: total=11826, train=0:8278, val=8278:10051, test=10051:11826
  Test windows: 1774 (context≤1024)
  --- Zero-shot ---
    Predicted 1774 windows in 70.1s (25 samples/sec)
    DA=0.4803 | RMSE=1.3064 | R²=0.9952 | pred_lag=0.99

In [ ]:
"""
generate_tables.py
==================
Reads results/*_predictions.csv, auto-identifies model names, computes:
  Table 1: Regression summary (by DA desc) - reg models only
  Table 2: 3-class classification - clf models + reg post-hoc
  Table 3: Per-ticker DA matrix - reg models only
  Table 4: Prediction Lag matrix - reg models only
Outputs Markdown and LaTeX.

Fixes:
  - Reg table reads only _reg_ files, excludes clf models
  - Clf table reads _clf_ files + reg post-hoc
  - Naive: predicted==current excluded from DA (reports N/A)
"""

import os
import re
import glob
import warnings
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

CONFIG = {
    "RESULTS_DIR": "../results",
    "THRESHOLD": 0.001,
    "OUTPUT_MD": "../results/all_tables.md",
    "OUTPUT_TEX": "../results/all_tables.tex",
}

CLASS_NAMES = {0: "Down", 1: "Flat", 2: "Up"}
CLF_KW = {"_clf_", "logistic", "classification"}
REG_KW = {"_reg_", "regression", "naive", "linear", "timesfm"}


# ---- discovery ----
def discover_files():
    pattern = os.path.join(CONFIG["RESULTS_DIR"], "*_predictions.csv")
    files = sorted(glob.glob(pattern))
    out = []
    for fp in files:
        bn = os.path.basename(fp).lower()
        mn = bn.replace("_predictions.csv", "")
        if any(k in bn for k in CLF_KW):
            ft = "clf"
        elif any(k in bn for k in REG_KW):
            ft = "reg"
        elif "clf" in bn or "class" in bn:
            ft = "clf"
        else:
            ft = "reg"
        out.append((fp, mn, ft))
    return out


def prettify(raw):
    m = {
        "naive_reg": "Naive",
        "linear_reg": "LinearReg",
        "lgbm_reg": "LightGBM",
        "logistic_clf": "Logistic",
        "lgbm_clf": "LightGBM_Clf",
        "vanilla_lstm_reg": "LSTM",
        "vanilla_lstm_clf": "LSTM_Clf",
        "attention_lstm_reg": "Attn-LSTM",
        "attention_lstm_clf": "Attn-LSTM_Clf",
        "transformer_lstm_reg": "TF+LSTM",
        "transformer_lstm_clf": "TF+LSTM_Clf",
        "timesfm_zs_reg": "TimesFM_ZS",
        "timesfm_ft_reg": "TimesFM_FT",
        "timesfm_fin_reg": "TimesFM_Fin",
        "timesfm_zs": "TimesFM_ZS",
        "timesfm_ft": "TimesFM_FT",
        "timesfm_fin": "TimesFM_Fin",
        "vmd_global": "VMD_Global",
        "vmd_rolling": "VMD_Rolling",
        "wavelet_lstm": "Wavelet+LSTM",
    }
    if raw in m:
        return m[raw]
    s = raw.replace("_reg", "").replace("_clf", "")
    return m.get(s, raw)


# ---- load ----
def load_pred(fp):
    df = pd.read_csv(fp)
    df.columns = [c.strip().lower() for c in df.columns]
    renames = {}
    for c in df.columns:
        if "tick" in c:
            renames[c] = "ticker"
        elif "pred" in c and "lag" not in c:
            renames[c] = "predicted"
        elif c in ("actual", "true", "target"):
            renames[c] = "actual"
        elif c in ("current", "prev"):
            renames[c] = "current"
    df = df.rename(columns=renames)
    if "current" not in df.columns:
        df["current"] = df.groupby("ticker")["actual"].shift(1)
        df = df.dropna(subset=["current"])
    return df


# ---- regression metrics (Naive-safe) ----
def reg_metrics(df):
    yp = df["predicted"].values.astype(np.float64)
    yt = df["actual"].values.astype(np.float64)
    yc = df["current"].values.astype(np.float64)

    pd_ = np.sign(yp - yc)
    td_ = np.sign(yt - yc)
    v_true = td_ != 0
    v_pred = pd_ != 0
    valid = v_true & v_pred

    nv = int(valid.sum())
    n_nodir = int((v_true & ~v_pred).sum())
    nt = len(yp)

    da = float(np.mean(pd_[valid] == td_[valid])) if nv > 0 else np.nan
    mse = float(np.mean((yt - yp) ** 2))
    rmse = float(np.sqrt(mse))
    mae = float(np.mean(np.abs(yt - yp)))
    ss_r = np.sum((yt - yp) ** 2)
    ss_t = np.sum((yt - np.mean(yt)) ** 2)
    r2 = float(1 - ss_r / ss_t) if ss_t > 0 else 0.0

    if nv > 0 and not np.isnan(da):
        se = np.sqrt(0.25 / nv)
        z = (da - 0.5) / se
        p = float(1.0 - stats.norm.cdf(z))
    else:
        z, p = 0.0, 1.0

    lag = float(np.corrcoef(yp, yc)[0, 1]) if nt > 2 else np.nan

    return {
        "n_total": nt, "n_valid": nv, "n_nodir": n_nodir,
        "da": da, "mse": mse, "rmse": rmse, "mae": mae, "r2": r2,
        "z": z, "p": p, "lag": lag,
    }


def per_ticker_reg(df):
    return {tk: reg_metrics(g) for tk, g in df.groupby("ticker")}


# ---- classification metrics ----
def _clf_core(pl, tl):
    acc = float(accuracy_score(tl, pl))
    pr, rc, f1, sp = precision_recall_fscore_support(
        tl, pl, labels=[0, 1, 2], zero_division=0.0)
    cm = confusion_matrix(tl, pl, labels=[0, 1, 2])
    cts = np.bincount(tl, minlength=3)
    n = len(tl)
    maj = float(cts.max() / n) if n > 0 else 0.0
    se = np.sqrt(maj * (1 - maj) / n) if n > 0 and 0 < maj < 1 else 1.0
    z = (acc - maj) / se
    p = float(1.0 - stats.norm.cdf(z))
    r = {"n": n, "acc": acc, "maj": maj, "z": z, "p": p}
    for i, nm in CLASS_NAMES.items():
        r[f"{nm}_P"] = float(pr[i])
        r[f"{nm}_R"] = float(rc[i])
        r[f"{nm}_F1"] = float(f1[i])
        r[f"{nm}_n"] = int(cts[i])
    r["cm"] = cm.tolist()
    return r


def clf_from_labels(df):
    yp = df["predicted"].values
    yt = df["actual"].values
    up = set(np.unique(yp).astype(int))
    ut = set(np.unique(yt).astype(int))
    if not up.issubset({0, 1, 2}) or not ut.issubset({0, 1, 2}):
        return None
    return _clf_core(yp.astype(np.int64), yt.astype(np.int64))


def clf_from_prices(df):
    yp = df["predicted"].values.astype(np.float64)
    yt = df["actual"].values.astype(np.float64)
    yc = df["current"].values.astype(np.float64)
    th = CONFIG["THRESHOLD"]

    def _lab(v, ref):
        ret = (v - ref) / np.maximum(np.abs(ref), 1e-8)
        lb = np.ones(len(ret), dtype=np.int64)
        lb[ret > th] = 2
        lb[ret < -th] = 0
        return lb

    return _clf_core(_lab(yp, yc), _lab(yt, yc))


# ---- latex helpers ----
def _esc(s):
    return str(s).replace("_", r"\_").replace("%", r"\%").replace("&", r"\&")


def _sig(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def to_latex(df, cap, lab):
    cf = "l" + "r" * (len(df.columns) - 1)
    L = [r"\begin{table}[htbp]", r"\centering",
         r"\caption{" + _esc(cap) + "}", r"\label{" + lab + "}",
         r"\small", r"\begin{tabular}{" + cf + "}", r"\toprule",
         " & ".join(_esc(c) for c in df.columns) + r" \\", r"\midrule"]
    for _, row in df.iterrows():
        L.append(" & ".join(_esc(v) for v in row.values) + r" \\")
    L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(L)


# ---- Table 1: Regression Summary ----
def build_t1(rd):
    rows = []
    for mn, df in rd.items():
        m = reg_metrics(df)
        da = m["da"]
        if pd.isna(da):
            das = f"N/A ({m['n_nodir']} no-dir)"
        else:
            das = f"{da*100:.1f}{_sig(m['p'])}"
        rows.append({
            "Model": prettify(mn), "N": m["n_total"],
            "N_valid": m["n_valid"], "DA (%)": das,
            "RMSE": f"{m['rmse']:.4f}", "MAE": f"{m['mae']:.4f}",
            "R2": f"{m['r2']:.4f}", "z": f"{m['z']:.2f}",
            "p": f"{m['p']:.4f}",
            "Pred_Lag": f"{m['lag']:.3f}" if pd.notna(m["lag"]) else "N/A",
        })
    df = pd.DataFrame(rows)

    def _x(s):
        try:
            return float(re.sub(r"[^\d.]", "", s.split("*")[0]))
        except Exception:
            return -1.0

    df["_s"] = df["DA (%)"].apply(_x)
    df = df.sort_values("_s", ascending=False).drop(columns="_s").reset_index(drop=True)
    md = "## Table 1: Regression Summary (by DA desc) -- Reg Models Only\n\n"
    md += df.to_markdown(index=False)
    md += "\n\n*p<0.05, **p<0.01, ***p<0.001\n"
    md += "N_valid: samples with nonzero direction in both pred and actual.\n"
    md += "Naive: predicted==current, no direction -> DA=N/A.\n"
    tx = to_latex(df, "Regression Summary sorted by DA", "tab:reg")
    return df, md, tx


# ---- Table 2: Classification Summary ----
def build_t2(cd, rd):
    rows = []
    for mn, df in cd.items():
        m = clf_from_labels(df)
        if m is None:
            continue
        rows.append({
            "Model": prettify(mn), "Src": "native", "N": m["n"],
            "Acc(%)": f"{m['acc']*100:.1f}{_sig(m['p'])}",
            "Maj(%)": f"{m['maj']*100:.1f}",
            "Down_F1": f"{m['Down_F1']:.3f}",
            "Flat_F1": f"{m['Flat_F1']:.3f}",
            "Up_F1": f"{m['Up_F1']:.3f}",
            "z": f"{m['z']:.2f}", "p": f"{m['p']:.4f}",
        })
    for mn, df in rd.items():
        nm = prettify(mn)
        if nm.lower() == "naive":
            continue
        m = clf_from_prices(df)
        rows.append({
            "Model": f"{nm}(ph)", "Src": "post-hoc", "N": m["n"],
            "Acc(%)": f"{m['acc']*100:.1f}{_sig(m['p'])}",
            "Maj(%)": f"{m['maj']*100:.1f}",
            "Down_F1": f"{m['Down_F1']:.3f}",
            "Flat_F1": f"{m['Flat_F1']:.3f}",
            "Up_F1": f"{m['Up_F1']:.3f}",
            "z": f"{m['z']:.2f}", "p": f"{m['p']:.4f}",
        })
    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df, "## Table 2: No clf results.\n", ""

    def _x(s):
        try:
            return float(re.sub(r"[^\d.]", "", s.split("*")[0]))
        except Exception:
            return -1.0

    df["_s"] = df["Acc(%)"].apply(_x)
    df = df.sort_values("_s", ascending=False).drop(columns="_s").reset_index(drop=True)
    th = CONFIG["THRESHOLD"]
    md = f"## Table 2: 3-Class Classification (thresh=+/-{th})\n\n"
    md += df.to_markdown(index=False)
    md += "\n\nnative=direct labels. (ph)=post-hoc from regression.\n"
    tx = to_latex(df, f"3-Class Classification thresh=+/-{th}", "tab:clf")
    return df, md, tx


# ---- Table 3: Per-ticker DA ----
def build_t3(rd):
    tks = sorted(set(t for d in rd.values() for t in d["ticker"].unique()))
    rows = []
    for mn, df in rd.items():
        pt = per_ticker_reg(df)
        row = {"Model": prettify(mn)}
        vs = []
        for tk in tks:
            if tk in pt:
                da = pt[tk]["da"]
                if pd.isna(da):
                    row[tk] = "N/A"
                else:
                    row[tk] = f"{da*100:.1f}{_sig(pt[tk]['p'])}"
                    vs.append(da)
            else:
                row[tk] = "-"
        row["Mean"] = f"{np.mean(vs)*100:.1f}" if vs else "-"
        row["Std"] = f"{np.std(vs)*100:.1f}" if vs else "-"
        rows.append(row)
    df = pd.DataFrame(rows)
    df["_s"] = df["Mean"].apply(lambda x: float(x) if x != "-" else -1.0)
    df = df.sort_values("_s", ascending=False).drop(columns="_s").reset_index(drop=True)
    md = "## Table 3: Per-Ticker DA (%) -- Reg Models Only\n\n"
    md += df.to_markdown(index=False)
    md += "\n\nN/A = no directional prediction (Naive).\n"
    tx = to_latex(df, "Per-Ticker DA", "tab:da")
    return df, md, tx


# ---- Table 4: Prediction Lag ----
def build_t4(rd):
    tks = sorted(set(t for d in rd.values() for t in d["ticker"].unique()))
    rows = []
    for mn, df in rd.items():
        pt = per_ticker_reg(df)
        row = {"Model": prettify(mn)}
        vs = []
        for tk in tks:
            if tk in pt:
                lg = pt[tk]["lag"]
                if pd.notna(lg):
                    row[tk] = f"{lg:.3f}"
                    vs.append(lg)
                else:
                    row[tk] = "N/A"
            else:
                row[tk] = "-"
        row["Mean"] = f"{np.mean(vs):.3f}" if vs else "-"
        rows.append(row)
    df = pd.DataFrame(rows)
    md = "## Table 4: Prediction Lag -- Reg Models Only\n\n"
    md += df.to_markdown(index=False)
    md += "\n\nNear 1.0 = copying current price. Naive=1.000 by definition.\n"
    tx = to_latex(df, "Prediction Lag corr(pred,current)", "tab:lag")
    return df, md, tx


# ---- main ----
def main():
    print("=" * 60)
    print("generate_tables.py -- Unified Results")
    print("=" * 60)

    triples = discover_files()
    if not triples:
        print(f"[ERROR] No *_predictions.csv in {CONFIG['RESULTS_DIR']}/")
        return

    print(f"\nFound {len(triples)} files:")
    for fp, mn, ft in triples:
        print(f"  [{ft}] {os.path.basename(fp):45s} -> {prettify(mn)}")

    reg_data, clf_data = {}, {}
    for fp, mn, ft in triples:
        try:
            df = load_pred(fp)
            print(f"  OK {mn}: {len(df)} rows, {df['ticker'].nunique()} tickers")
            (clf_data if ft == "clf" else reg_data)[mn] = df
        except Exception as e:
            print(f"  FAIL {mn}: {e}")

    if not reg_data and not clf_data:
        print("[ERROR] No valid data.")
        return

    print(f"\nReg: {len(reg_data)} | Clf: {len(clf_data)}")
    print("=" * 60)

    amd = ["# Unified Results Tables\n",
           f"Reg: {len(reg_data)} | Clf: {len(clf_data)}\n"]
    atx = ["% Auto-generated", r"% \usepackage{booktabs}", ""]

    if reg_data:
        _, m1, t1 = build_t1(reg_data)
        amd.append(m1); atx.append(t1); print("\n" + m1)

    _, m2, t2 = build_t2(clf_data, reg_data)
    amd.append(m2); atx.append(t2); print("\n" + m2)

    if reg_data:
        _, m3, t3 = build_t3(reg_data)
        amd.append(m3); atx.append(t3); print("\n" + m3)
        _, m4, t4 = build_t4(reg_data)
        amd.append(m4); atx.append(t4); print("\n" + m4)

    with open(CONFIG["OUTPUT_MD"], "w", encoding="utf-8") as f:
        f.write("\n\n".join(amd))
    print(f"\n[SAVED] {CONFIG['OUTPUT_MD']}")

    with open(CONFIG["OUTPUT_TEX"], "w", encoding="utf-8") as f:
        f.write("\n\n".join(atx))
    print(f"[SAVED] {CONFIG['OUTPUT_TEX']}")

    print("\n" + "=" * 60 + "\nQuick Stats\n" + "=" * 60)
    for mn, df in reg_data.items():
        m = reg_metrics(df)
        nm = prettify(mn)
        da = m["da"]
        if pd.isna(da):
            print(f"  {nm:25s} DA=  N/A  z={m['z']:+.2f}  -- no direction")
        else:
            s = "BEATS" if da > 0.5 and m["p"] < 0.05 else "~random"
            lg = f"{m['lag']:.3f}" if pd.notna(m["lag"]) else "N/A"
            print(f"  {nm:25s} DA={da*100:5.1f}%  z={m['z']:+.2f}  lag={lg}  {s}")
    for mn, df in clf_data.items():
        m = clf_from_labels(df)
        if m is None:
            continue
        nm = prettify(mn)
        s = "BEATS" if m["acc"] > m["maj"] and m["p"] < 0.05 else "~majority"
        print(f"  {nm:25s} Acc={m['acc']*100:.1f}%  maj={m['maj']*100:.1f}%  {s}")


if __name__ == "__main__":
    main()

In [ ]:
"""
generate_plots.py — Thesis Figures (v2)
Reads results/*_predictions.csv, generates:
  Fig 1: DA bar chart (reg models, 50% baseline)
  Fig 2: Prediction Lag bar chart
  Fig 3: Per-ticker DA heatmap
  Fig 4: 3-panel (raw price / zoomed+markers / return comparison)
  Fig 4b: Illusion grid (all models one ticker)
  Fig 5: Confusion matrices
  Fig 6: Classification accuracy vs majority
  Fig 7: DA by model type (grouped)

Usage:  cd毕设/data && python generate_plots.py
"""

import os, re, glob, warnings
from typing import Dict, List, Tuple, Optional
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

warnings.filterwarnings("ignore")

CONFIG = {
    "RESULTS_DIR": "../results",
    "PLOT_DIR": "../results/figures",
    "THRESHOLD": 0.001,
    "DPI": 200,
    "FIG_WIDTH": 10,
}

CLF_KW = {"_clf_", "logistic", "classification"}
REG_KW = {"_reg_", "regression", "naive", "linear", "timesfm"}

PRETTY = {
    "naive_reg": "Naive", "linear_reg": "LinearReg", "lgbm_reg": "LightGBM",
    "logistic_clf": "Logistic", "lgbm_clf": "LightGBM_Clf",
    "vanilla_lstm_reg": "LSTM", "vanilla_lstm_clf": "LSTM_Clf",
    "attention_lstm_reg": "Attn-LSTM", "attention_lstm_clf": "Attn-LSTM_Clf",
    "transformer_lstm_reg": "TF+LSTM", "transformer_lstm_clf": "TF+LSTM_Clf",
    "timesfm_zs_reg": "TimesFM_ZS", "timesfm_ft_reg": "TimesFM_FT",
    "timesfm_zs": "TimesFM_ZS", "timesfm_ft": "TimesFM_FT",
}


def prettify(raw):
    if raw in PRETTY: return PRETTY[raw]
    s = raw.replace("_reg", "").replace("_clf", "")
    return PRETTY.get(s, raw)


def discover_files():
    pattern = os.path.join(CONFIG["RESULTS_DIR"], "*_predictions.csv")
    out = []
    for fp in sorted(glob.glob(pattern)):
        bn = os.path.basename(fp).lower()
        mn = bn.replace("_predictions.csv", "")
        if any(k in bn for k in CLF_KW): ft = "clf"
        elif any(k in bn for k in REG_KW): ft = "reg"
        elif "clf" in bn: ft = "clf"
        else: ft = "reg"
        out.append((fp, mn, ft))
    return out


def load_pred(fp):
    df = pd.read_csv(fp)
    df.columns = [c.strip().lower() for c in df.columns]
    renames = {}
    for c in df.columns:
        if "tick" in c: renames[c] = "ticker"
        elif "pred" in c and "lag" not in c: renames[c] = "predicted"
        elif c in ("actual","true","target"): renames[c] = "actual"
        elif c in ("current","prev"): renames[c] = "current"
    df = df.rename(columns=renames)
    if "current" not in df.columns:
        df["current"] = df.groupby("ticker")["actual"].shift(1)
        df = df.dropna(subset=["current"])
    return df


def reg_metrics(df):
    yp = df["predicted"].values.astype(np.float64)
    yt = df["actual"].values.astype(np.float64)
    yc = df["current"].values.astype(np.float64)
    pd_ = np.sign(yp - yc); td_ = np.sign(yt - yc)
    valid = (td_ != 0) & (pd_ != 0); nv = int(valid.sum())
    da = float(np.mean(pd_[valid] == td_[valid])) if nv > 0 else np.nan
    if nv > 0 and not np.isnan(da):
        z = (da - 0.5) / np.sqrt(0.25 / nv)
        p = float(1.0 - stats.norm.cdf(z))
    else: z, p = 0.0, 1.0
    lag = float(np.corrcoef(yp, yc)[0, 1]) if len(yp) > 2 else np.nan
    return {"da": da, "z": z, "p": p, "lag": lag, "n": nv}


def per_ticker_reg(df):
    return {tk: reg_metrics(g) for tk, g in df.groupby("ticker")}


def clf_from_prices(df):
    yp = df["predicted"].values.astype(np.float64)
    yt = df["actual"].values.astype(np.float64)
    yc = df["current"].values.astype(np.float64)
    th = CONFIG["THRESHOLD"]
    def _lab(v, ref):
        ret = (v - ref) / np.maximum(np.abs(ref), 1e-8)
        lb = np.ones(len(ret), dtype=np.int64)
        lb[ret > th] = 2; lb[ret < -th] = 0
        return lb
    return _lab(yp, yc), _lab(yt, yc)


def setup_style():
    plt.rcParams.update({
        "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 11,
        "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9,
        "figure.dpi": CONFIG["DPI"], "savefig.dpi": CONFIG["DPI"],
        "savefig.bbox": "tight",
        "axes.spines.top": False, "axes.spines.right": False,
    })


def _sig(p):
    if pd.isna(p): return ""
    if p < 0.001: return "***"
    if p < 0.01: return "**"
    if p < 0.05: return "*"
    return ""


# ============ Fig 1: DA Bar ============
def plot_da_bar(reg_data, save_dir):
    names, das, colors, sigs = [], [], [], []
    for mn, df in reg_data.items():
        m = reg_metrics(df); nm = prettify(mn); da = m["da"]
        if pd.isna(da):
            names.append(nm); das.append(0); colors.append("#cccccc"); sigs.append("N/A")
        else:
            names.append(nm); das.append(da * 100)
            colors.append("#2196F3" if da>0.5 and m["p"]<0.05
                          else "#FF9800" if da>0.5 else "#E0E0E0")
            sigs.append(_sig(m["p"]))
    idx = np.argsort(das)[::-1]
    names=[names[i] for i in idx]; das=[das[i] for i in idx]
    colors=[colors[i] for i in idx]; sigs=[sigs[i] for i in idx]
    fig, ax = plt.subplots(figsize=(CONFIG["FIG_WIDTH"], max(4,len(names)*0.5)))
    bars = ax.barh(range(len(names)), das, color=colors, edgecolor="white", height=0.6)
    ax.axvline(x=50, color="red", ls="--", lw=1.5, label="Random (50%)", zorder=0)
    for i,(bar,sig) in enumerate(zip(bars,sigs)):
        w = bar.get_width()
        ax.text(max(w,1)+0.5, i, f"{w:.1f}%{sig}" if w>0 else sig,
                va="center", fontsize=9, color="black" if w>0 else "gray")
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    ax.set_xlabel("DA (%)"); ax.set_title("Fig 1: DA All Regression Models")
    ax.legend(loc="lower right"); ax.set_xlim(0, max(das)+8 if das else 60); ax.invert_yaxis()
    fig.savefig(os.path.join(save_dir, "fig1_da_bar.png")); plt.close(fig)
    print(f"  [OK] fig1_da_bar.png")


# ============ Fig 2: Pred Lag ============
def plot_lag_bar(reg_data, save_dir):
    names, lags = [], []
    for mn, df in reg_data.items():
        m = reg_metrics(df); names.append(prettify(mn))
        lags.append(m["lag"] if pd.notna(m["lag"]) else 1.0)
    idx = np.argsort(lags)[::-1]
    names=[names[i] for i in idx]; lags=[lags[i] for i in idx]
    colors = ["#E53935" if l>0.99 else "#FF9800" if l>0.95 else "#4CAF50" for l in lags]
    fig, ax = plt.subplots(figsize=(CONFIG["FIG_WIDTH"], max(4,len(names)*0.5)))
    bars = ax.barh(range(len(names)), lags, color=colors, edgecolor="white", height=0.6)
    ax.axvline(x=1.0, color="red", ls="--", lw=1, alpha=0.7, label="Perfect copy")
    for i,bar in enumerate(bars):
        ax.text(bar.get_width()+0.005, i, f"{bar.get_width():.3f}", va="center", fontsize=9)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    ax.set_xlabel("Pred Lag"); ax.set_title("Fig 2: Prediction Lag")
    ax.legend(loc="lower right",fontsize=8); ax.set_xlim(0,1.15); ax.invert_yaxis()
    fig.savefig(os.path.join(save_dir, "fig2_pred_lag.png")); plt.close(fig)
    print(f"  [OK] fig2_pred_lag.png")


# ============ Fig 3: DA Heatmap ============
def plot_da_heatmap(reg_data, save_dir):
    tickers = sorted(set(t for d in reg_data.values() for t in d["ticker"].unique()))
    mnames, mat = [], []
    for mn, df in reg_data.items():
        pt = per_ticker_reg(df); row = []
        for tk in tickers:
            row.append(pt[tk]["da"]*100 if tk in pt and not pd.isna(pt[tk]["da"]) else np.nan)
        mnames.append(prettify(mn)); mat.append(row)
    arr = np.array(mat)
    means = np.nanmean(arr, axis=1); idx = np.argsort(means)[::-1]
    arr = arr[idx]; mnames = [mnames[i] for i in idx]
    fig, ax = plt.subplots(figsize=(max(8,len(tickers)*1.2), max(4,len(mnames)*0.55)))
    vn = min(np.nanmin(arr),48) if not np.all(np.isnan(arr)) else 45
    vx = max(np.nanmax(arr),52) if not np.all(np.isnan(arr)) else 55
    im = ax.imshow(arr, cmap="RdYlGn", norm=TwoSlopeNorm(vcenter=50,vmin=vn,vmax=vx), aspect="auto")
    ax.set_xticks(range(len(tickers))); ax.set_xticklabels(tickers, rotation=45, ha="right")
    ax.set_yticks(range(len(mnames))); ax.set_yticklabels(mnames)
    for i in range(len(mnames)):
        for j in range(len(tickers)):
            v = arr[i,j]
            if np.isnan(v): ax.text(j,i,"N/A",ha="center",va="center",fontsize=8,color="gray")
            else: ax.text(j,i,f"{v:.1f}",ha="center",va="center",fontsize=8,
                          color="white" if abs(v-50)>3 else "black")
    fig.colorbar(im, ax=ax, shrink=0.8).set_label("DA (%)")
    ax.set_title("Fig 3: Per-Ticker DA (%)")
    fig.savefig(os.path.join(save_dir, "fig3_da_heatmap.png")); plt.close(fig)
    print(f"  [OK] fig3_da_heatmap.png")


# ============ Fig 4: THREE-PANEL ============
def plot_pred_vs_actual(reg_data, save_dir):
    # 3-panel for EVERY regression model (including Naive)
    for mn, df in reg_data.items():
        nm = prettify(mn); m = reg_metrics(df)
        is_naive = (nm.lower() == "naive")
        da_str = f"{m['da']*100:.1f}%" if pd.notna(m["da"]) else "N/A"
        tk = df.groupby("ticker").size().idxmax()
        sub = df[df["ticker"]==tk].tail(200).reset_index(drop=True)
        act = sub["actual"].values; pred = sub["predicted"].values; curr = sub["current"].values
        x = np.arange(len(sub))
        fig = plt.figure(figsize=(CONFIG["FIG_WIDTH"], 9))
        gs = gridspec.GridSpec(3, 1, height_ratios=[2,2,2], hspace=0.35)

        # (A) Full price
        ax1 = fig.add_subplot(gs[0])
        ax1.plot(x, act, color="#1976D2", lw=1.2, label="Actual", alpha=0.9)
        plbl = "Predicted (= current)" if is_naive else "Predicted"
        ax1.plot(x, pred, color="#E53935", lw=1.0, label=plbl, alpha=0.75, ls="--")
        ax1.set_title(f"(A) {nm} on {tk} full view  (DA={da_str}, Lag={m['lag']:.3f})", fontsize=11)
        ax1.legend(fontsize=8, loc="upper left"); ax1.set_ylabel("Price ($)")
        ax1.tick_params(labelsize=8); ax1.set_xlim(0, len(x))
        ann = ("Predicted = previous price\n=> zero directional info" if is_naive
               else "Curves overlap but DA ~ 50%")
        afc = "#FFEBEE" if is_naive else "#FFF3E0"
        aec = "#E53935" if is_naive else "#FF9800"
        ax1.annotate(ann,
                     xy=(0.98,0.95), xycoords="axes fraction", ha="right", va="top", fontsize=8,
                     bbox=dict(boxstyle="round,pad=0.3", fc=afc, ec=aec, alpha=0.9))

        # (B) Zoomed 30 steps
        ax2 = fig.add_subplot(gs[1])
        w=30; s=len(sub)//2; e=s+w
        xs=x[s:e]; aw=act[s:e]; pw=pred[s:e]; cw=curr[s:e]
        ax2.plot(xs, aw, color="#1976D2", lw=2, label="Actual", marker="o", ms=3, alpha=0.9)
        ax2.plot(xs, pw, color="#E53935", lw=1.5, label="Predicted", marker="s", ms=3, alpha=0.8, ls="--")
        ax2.plot(xs, cw, color="#9E9E9E", lw=1, label="Current(t-1)", alpha=0.5, ls=":")
        for i in range(len(xs)):
            pdd = np.sign(pw[i]-cw[i]); tdd = np.sign(aw[i]-cw[i])
            if tdd==0: continue
            if pdd==0: clr,mk="#9E9E9E","x"
            elif pdd==tdd: clr,mk="#4CAF50","^"
            else: clr,mk="#F44336","v"
            ax2.scatter(xs[i], aw[i], color=clr, marker=mk, s=60, zorder=5,
                        edgecolors="white", linewidth=0.5)
        ax2.scatter([],[],color="#4CAF50",marker="^",s=50,label="Correct")
        ax2.scatter([],[],color="#F44336",marker="v",s=50,label="Wrong")
        ax2.set_title(f"(B) Zoomed steps {s}-{e}, direction markers", fontsize=11)
        ax2.legend(fontsize=7, loc="upper left", ncol=2); ax2.set_ylabel("Price ($)")
        ax2.tick_params(labelsize=8)

        # (C) Returns
        ax3 = fig.add_subplot(gs[2])
        aret = (act-curr)/np.maximum(np.abs(curr),1e-8)*100
        pret = (pred-curr)/np.maximum(np.abs(curr),1e-8)*100
        ax3.plot(x, aret, color="#1976D2", lw=0.8, label="Actual return", alpha=0.8)
        ax3.plot(x, pret, color="#E53935", lw=0.8, label="Predicted return", alpha=0.6)
        ax3.axhline(y=0, color="black", lw=0.5, alpha=0.3)
        ax3.fill_between(x, aret, 0, alpha=0.1, color="#1976D2")
        pdir=np.sign(pret); tdir=np.sign(aret)
        dis = (pdir!=tdir)&(tdir!=0)&(pdir!=0)
        for i in range(len(x)):
            if dis[i]: ax3.axvspan(x[i]-0.4, x[i]+0.4, color="#FFCDD2", alpha=0.3, zorder=0)
        vm = (tdir!=0)&(pdir!=0)
        pw_ = dis.sum()/vm.sum()*100 if vm.sum()>0 else 0
        cnote = ("Predicted return ~ 0 (no direction)" if is_naive
                 else f"Red = wrong direction ({pw_:.0f}%)")
        ax3.annotate(cnote,
                     xy=(0.98,0.95), xycoords="axes fraction", ha="right", va="top", fontsize=8,
                     bbox=dict(boxstyle="round,pad=0.3", fc="#FFEBEE", ec="#E53935", alpha=0.9))
        ax3.set_title("(C) Return comparison, reveals true quality", fontsize=11)
        ax3.set_xlabel("Time step"); ax3.set_ylabel("Return (%)")
        ax3.legend(fontsize=8, loc="upper left"); ax3.tick_params(labelsize=8); ax3.set_xlim(0,len(x))

        safe = nm.lower().replace("+","_").replace("-","_")
        path = os.path.join(save_dir, f"fig4_{safe}_3panel.png")
        fig.savefig(path); plt.close(fig)
        print(f"  [OK] {os.path.basename(path)}")

    # Naive: single panel showing pred==current
    for mn, df in reg_data.items():
        if prettify(mn).lower() != "naive": continue
        tk = df.groupby("ticker").size().idxmax()
        sub = df[df["ticker"]==tk].tail(100).reset_index(drop=True)
        fig, ax = plt.subplots(figsize=(CONFIG["FIG_WIDTH"], 3.5))
        x = np.arange(len(sub))
        ax.plot(x, sub["actual"].values, color="#1976D2", lw=1.2, label="Actual", alpha=0.9)
        ax.plot(x, sub["predicted"].values, color="#E53935", lw=1.0, label="Predicted (=current)", alpha=0.75, ls="--")
        ax.set_title(f"Naive on {tk}: predicted = current price (DA=N/A, Lag=1.000)", fontsize=11)
        ax.legend(fontsize=9); ax.set_xlabel("Time step"); ax.set_ylabel("Price")
        ax.annotate("Predicted IS previous price, zero directional info",
                    xy=(0.98,0.95), xycoords="axes fraction", ha="right", va="top", fontsize=9,
                    bbox=dict(boxstyle="round,pad=0.3", fc="#FFEBEE", ec="#E53935", alpha=0.9))
        fig.savefig(os.path.join(save_dir, "fig4_naive.png")); plt.close(fig)
        print(f"  [OK] fig4_naive.png")
        break


# ============ Fig 4b: Illusion Grid ============
def plot_illusion_grid(reg_data, save_dir):
    models = {mn: df for mn, df in reg_data.items() if prettify(mn).lower() != "naive"}
    if not models: return
    tks = sorted(list(models.values())[0]["ticker"].unique())
    tk = "AAPL" if "AAPL" in tks else tks[0]
    n=len(models); nc=min(3,n); nr=(n+nc-1)//nc
    fig, axes = plt.subplots(nr, nc, figsize=(CONFIG["FIG_WIDTH"], nr*2.5), squeeze=False)
    for idx,(mn,df) in enumerate(models.items()):
        r,c = idx//nc, idx%nc; ax = axes[r][c]; nm = prettify(mn); m = reg_metrics(df)
        sub = df[df["ticker"]==tk].tail(100).reset_index(drop=True)
        if len(sub)==0: ax.set_visible(False); continue
        ax.plot(range(len(sub)), sub["actual"].values, color="#1976D2", lw=1, alpha=0.9)
        ax.plot(range(len(sub)), sub["predicted"].values, color="#E53935", lw=0.8, alpha=0.7, ls="--")
        da_s = f"{m['da']*100:.1f}%" if pd.notna(m["da"]) else "N/A"
        lg_s = f"{m['lag']:.3f}" if pd.notna(m["lag"]) else "N/A"
        ax.set_title(f"{nm}\nDA={da_s} lag={lg_s}", fontsize=9); ax.tick_params(labelsize=6)
        if idx==0:
            ax.plot([],[],color="#1976D2",label="Actual")
            ax.plot([],[],color="#E53935",ls="--",label="Predicted"); ax.legend(fontsize=7)
    for idx in range(n, nr*nc): axes[idx//nc][idx%nc].set_visible(False)
    fig.suptitle(f"Fig 4b: All Models on {tk}, ALL Look Identical (DA~50%)", fontsize=12, y=1.03)
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, "fig4b_illusion_grid.png")); plt.close(fig)
    print(f"  [OK] fig4b_illusion_grid.png")


# ============ Fig 5: Confusion ============
def plot_confusion(clf_data, reg_data, save_dir):
    from sklearn.metrics import confusion_matrix as cm_fn
    items = []
    for mn, df in clf_data.items():
        yp=df["predicted"].values.astype(np.int64); yt=df["actual"].values.astype(np.int64)
        if set(np.unique(yp)).issubset({0,1,2}): items.append((prettify(mn), yp, yt))
    for mn, df in reg_data.items():
        nm=prettify(mn)
        if nm.lower()=="naive": continue
        pl,tl = clf_from_prices(df); items.append((f"{nm}(ph)", pl, tl))
    if not items: print("  [SKIP] Fig 5"); return
    n=len(items); nc=min(4,n); nr=(n+nc-1)//nc
    fig, axes = plt.subplots(nr, nc, figsize=(nc*3, nr*2.8), squeeze=False)
    for idx,(name,pl,tl) in enumerate(items):
        r,c = idx//nc, idx%nc; ax = axes[r][c]
        cm = cm_fn(tl, pl, labels=[0,1,2]); cmn = cm.astype(float)
        rs = cm.sum(axis=1, keepdims=True); rs[rs==0]=1; cmn /= rs
        ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1, aspect="auto")
        for i in range(3):
            for j in range(3):
                clr = "white" if cmn[i,j]>0.5 else "black"
                ax.text(j,i,f"{cm[i,j]}\n({cmn[i,j]:.0%})",ha="center",va="center",fontsize=7,color=clr)
        ax.set_xticks([0,1,2]); ax.set_xticklabels(["Down","Flat","Up"],fontsize=7)
        ax.set_yticks([0,1,2]); ax.set_yticklabels(["Down","Flat","Up"],fontsize=7)
        acc = float(np.sum(pl==tl))/len(tl)*100
        ax.set_title(f"{name}\nAcc={acc:.1f}%", fontsize=9)
    for idx in range(n, nr*nc): axes[idx//nc][idx%nc].set_visible(False)
    fig.suptitle("Fig 5: Confusion Matrices (row-normalized)", fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, "fig5_confusion.png")); plt.close(fig)
    print(f"  [OK] fig5_confusion.png")


# ============ Fig 6: Clf Accuracy ============
def plot_clf_accuracy(clf_data, reg_data, save_dir):
    from sklearn.metrics import accuracy_score
    items = []
    for mn, df in clf_data.items():
        yp=df["predicted"].values.astype(np.int64); yt=df["actual"].values.astype(np.int64)
        if not set(np.unique(yp)).issubset({0,1,2}): continue
        acc=accuracy_score(yt,yp)*100; cts=np.bincount(yt,minlength=3)
        items.append((prettify(mn), acc, cts.max()/len(yt)*100))
    for mn, df in reg_data.items():
        nm=prettify(mn)
        if nm.lower()=="naive": continue
        pl,tl=clf_from_prices(df); acc=accuracy_score(tl,pl)*100
        cts=np.bincount(tl,minlength=3); items.append((f"{nm}(ph)",acc,cts.max()/len(tl)*100))
    if not items: print("  [SKIP] Fig 6"); return
    items.sort(key=lambda x: x[1], reverse=True)
    names=[x[0] for x in items]; accs=[x[1] for x in items]; majs=[x[2] for x in items]
    fig, ax = plt.subplots(figsize=(CONFIG["FIG_WIDTH"], max(4,len(names)*0.5)))
    ax.barh(range(len(names)), accs, height=0.5, color="#2196F3", label="Model Acc", alpha=0.85)
    for i,m in enumerate(majs): ax.plot(m, i, marker="D", color="#E53935", ms=7, zorder=5)
    ax.plot([],[],marker="D",color="#E53935",ls="None",label="Majority BL")
    for i in range(len(names)):
        d=accs[i]-majs[i]; s="+" if d>0 else ""
        ax.text(accs[i]+0.5, i, f"{accs[i]:.1f}% ({s}{d:.1f})", va="center", fontsize=8)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    ax.set_xlabel("Accuracy (%)"); ax.set_title("Fig 6: 3-Class Acc vs Majority")
    ax.legend(loc="lower right"); ax.invert_yaxis()
    fig.savefig(os.path.join(save_dir, "fig6_clf_accuracy.png")); plt.close(fig)
    print(f"  [OK] fig6_clf_accuracy.png")


# ============ Fig 7: DA by Type ============
def plot_da_by_type(reg_data, save_dir):
    groups = {"Statistical":["Naive","LinearReg","LightGBM"],
              "LSTM":["LSTM","Attn-LSTM","TF+LSTM"],
              "Foundation":["TimesFM_ZS","TimesFM_FT"]}
    cm = {"Statistical":"#78909C","LSTM":"#42A5F5","Foundation":"#AB47BC"}
    fig, ax = plt.subplots(figsize=(CONFIG["FIG_WIDTH"], 5))
    xo=0; xt,xl,gb = [],[],[]
    for gn,ml in groups.items():
        gs=xo
        for mp in ml:
            for mn, df in reg_data.items():
                if prettify(mn)==mp:
                    m=reg_metrics(df); da=m["da"]; dv=da*100 if pd.notna(da) else 0
                    ax.bar(xo, dv, width=0.7, color=cm[gn], edgecolor="white", alpha=0.85)
                    if dv>0: ax.text(xo, dv+0.5, f"{dv:.1f}", ha="center", fontsize=8)
                    elif pd.isna(da): ax.text(xo, 1, "N/A", ha="center", fontsize=8, color="gray")
                    break
            xt.append(xo); xl.append(mp); xo+=1
        gb.append((gs,xo-1,gn)); xo+=0.5
    ax.axhline(y=50, color="red", ls="--", lw=1.5, label="Random (50%)", zorder=0)
    for gs,ge,gn in gb:
        ax.text((gs+ge)/2, -4, gn, ha="center", fontsize=10, fontweight="bold", color=cm[gn])
    ax.set_xticks(xt); ax.set_xticklabels(xl, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("DA (%)"); ax.set_title("Fig 7: DA by Category, The 50% Ceiling")
    ax.legend(loc="upper right"); ax.set_ylim(0, 60)
    fig.savefig(os.path.join(save_dir, "fig7_da_by_type.png")); plt.close(fig)
    print(f"  [OK] fig7_da_by_type.png")


# ============ Main ============
def main():
    print("="*60+"\ngenerate_plots.py v3 -- ALL models get 3-panel\n"+"="*60)
    setup_style(); os.makedirs(CONFIG["PLOT_DIR"], exist_ok=True)
    triples = discover_files()
    if not triples: print(f"[ERR] No files in {CONFIG['RESULTS_DIR']}/"); return
    reg_data, clf_data = {}, {}
    for fp, mn, ft in triples:
        try:
            df = load_pred(fp)
            print(f"  [{ft}] {prettify(mn):20s} {len(df)} rows")
            (clf_data if ft=="clf" else reg_data)[mn] = df
        except Exception as e: print(f"  FAIL {mn}: {e}")
    if not reg_data and not clf_data: print("[ERR] No data."); return
    print(f"\nReg: {len(reg_data)} | Clf: {len(clf_data)}\n")
    if reg_data:
        plot_da_bar(reg_data, CONFIG["PLOT_DIR"])
        plot_lag_bar(reg_data, CONFIG["PLOT_DIR"])
        plot_da_heatmap(reg_data, CONFIG["PLOT_DIR"])
        plot_pred_vs_actual(reg_data, CONFIG["PLOT_DIR"])
        plot_illusion_grid(reg_data, CONFIG["PLOT_DIR"])
        plot_da_by_type(reg_data, CONFIG["PLOT_DIR"])
    if clf_data or reg_data:
        plot_confusion(clf_data, reg_data, CONFIG["PLOT_DIR"])
        plot_clf_accuracy(clf_data, reg_data, CONFIG["PLOT_DIR"])
    print(f"\nAll saved to {CONFIG['PLOT_DIR']}/")

if __name__ == "__main__":
    main()

# 2